# 7BZ Dev Cloud Game

Controller-first Tesla T4 cloud gaming and development console on Kaggle.

### Architecture & Storefront Suite
- **Unprivileged Execution**: Storefronts and games execute under standard user `gamer` for credential protection and clean user space.
- **Steam (Valve Native)**: Full multiarch client running with software CEF flags. License agreement presented directly to the user in the GUI.
- **Epic Games**: Heroic GUI and Legendary CLI are available for testing. Launch each installed title to verify compatibility.
- **Wine Runner**: Heroic settings select a complete Wine runner, disable UMU/DXVK/VKD3D for this container, and back up existing JSON before repair.
- **GOG Galaxy (Minigalaxy & Heroic)**: DRM-free client for GOG games. Supports native Linux builds and Wine.
- **Lutris Gaming Hub**: Multi-storefront aggregator installed from Ubuntu multiverse.
- **EA App (experimental)**: Wine installer opens; login and game execution need interactive validation.
- **7BZ Console Dashboard**: Card library and storefront hub with mouse, keyboard, and full gamepad navigation.
- **Streaming HUD**: English glass overlay with WebSocket telemetry, adaptive bitrate, browser controls, and emergency game exit.
- **Desktop Window Management**: Openbox window manager and tint2 taskbar panel for smooth multi-window multitasking.
- **Selkies 2.1.0 Streaming**: Tesla T4 NVENC hardware H.264 video encoding at 60 FPS, Opus stereo audio, and 4 virtual Xbox gamepad sockets over Kaggle signed Jupyter Server proxy.

### Hardware Reality & Constraints
- **NVENC Hardware Acceleration**: Encoding is hardware-accelerated on Tesla T4 via NVENC.
- **Desktop 3D Rendering**: The verified Wine OpenGL path is direct NVIDIA Xorg `:1` streamed on port `8080`. GPU 0 handles direct hardware 3D rendering and GPU 1 encodes with NVENC.

### Instructions
1. Set **Accelerator: GPU T4 x2** and **Internet: On** in the Kaggle sidebar.
2. Run all cells in an active interactive session.
3. Copy the **VSCode Compatible URL** from **Run → Kaggle Jupyter Server**, paste it into the connection cell, and connect!
4. The connection cell opens the direct NVIDIA desktop on port `8080`. Openbox, Legendary, and games share that display directly.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

BENCH = Path('/tmp/kaggle-game-dev-bench')
BUILDS = Path('/tmp/game-builds')
LOGS = BENCH / 'logs'
EA_STORE = Path('/tmp/ea-store')
REPO_URL = 'https://github.com/Mohammedsbastawy/kaggle-gaming-bench.git'

BUILDS.mkdir(parents=True, exist_ok=True)
EA_STORE.mkdir(parents=True, exist_ok=True)

# 1. Automatic recovery for apt/dpkg locks (prevents RuntimeError exit 100 on interactive sessions)
print('▶ Clearing stale apt/dpkg locks and recovering package manager...', flush=True)
for lock in ['/var/lib/dpkg/lock-frontend', '/var/lib/dpkg/lock', '/var/lib/apt/lists/lock', '/var/cache/apt/archives/lock']:
    subprocess.run(['fuser', '-k', lock], check=False)
    try:
        os.remove(lock)
    except OSError:
        pass
subprocess.run(['dpkg', '--configure', '-a'], check=False, env=dict(os.environ, DEBIAN_FRONTEND='noninteractive'))

# 2. Clone or update the ready Git repository
setup_sh = BENCH / 'setup.sh'
if not setup_sh.exists():
    shutil.rmtree(str(BENCH), ignore_errors=True)
    print(f'▶ Cloning ready gaming bench suite from {REPO_URL}...', flush=True)
    res = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(BENCH)], timeout=60)
    if res.returncode != 0 or not setup_sh.exists():
        print('Notice: Git clone unavailable, using embedded setup suite.', flush=True)
        BENCH.mkdir(parents=True, exist_ok=True)
        setup_sh.write_text('#!/bin/bash\nset -e\n\nBENCH="/tmp/kaggle-game-dev-bench"\nLOGS="/home/gamer/logs"\nBUILDS="/tmp/game-builds"\nmkdir -p "$BENCH/logs" "$BUILDS" /tmp/ea-store\n\nexport DEBIAN_FRONTEND=noninteractive\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\n\necho "=========================================================="\necho "    7BZ DEV CLOUD GAME BOOTSTRAPPER                       "\necho "=========================================================="\n\n# 1. Kill stale apt/dpkg locks if running in dirty interactive session\necho "▶ Checking and clearing stale apt locks..."\nfuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\nfuser -k /var/lib/dpkg/lock 2>/dev/null || true\nfuser -k /var/lib/apt/lists/lock 2>/dev/null || true\nfuser -cache/apt/archives/lock 2>/dev/null || true\nrm -f /var/lib/dpkg/lock* /var/lib/apt/lists/lock* /var/cache/apt/archives/lock* 2>/dev/null || true\ndpkg --configure -a 2>/dev/null || true\n\n# Helper function for resilient apt install with timeout and retries\napt_install() {\n    local label="$1"\n    shift\n    echo "▶ Installing $label..."\n    local attempts=0\n    while [ $attempts -lt 3 ]; do\n        attempts=$((attempts + 1))\n        if apt-get install -y --no-install-recommends \\\n            -o DPkg::Lock::Timeout=120 \\\n            -o APT::Get::Assume-Yes=true \\\n            "$@"; then\n            return 0\n        fi\n        echo "Notice: apt install for $label failed (attempt $attempts/3). Retrying in 4s..."\n        sleep 4\n        fuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\n        rm -f /var/lib/dpkg/lock* 2>/dev/null || true\n        dpkg --configure -a 2>/dev/null || true\n    done\n    echo "Warning: Could not install some packages for $label, continuing..."\n    return 0\n}\n\n# 2. Multiarch & repositories setup\ndpkg --add-architecture i386 || true\nif [ -f /etc/apt/sources.list.d/ubuntu.sources ]; then\n    sed -i \'s/Components: main universe/Components: main universe multiverse/g\' /etc/apt/sources.list.d/ubuntu.sources || true\nfi\n\n# Resilient apt update\napt-get update -o DPkg::Lock::Timeout=120 -qq || true\n\n# 3. Core desktop, X11, audio, and compression utilities\napt_install "core desktop" \\\n    xvfb openbox tint2 xterm pulseaudio pulseaudio-utils xauth dbus-x11 \\\n    libgl1-mesa-dri libgbm1 libpixman-1-0 libxkbcommon0 mesa-utils x11-utils vulkan-tools \\\n    xdotool xinput wmctrl wget curl cabextract p7zip-full zstd winetricks python3-pygame python3-aiohttp\n\n# Build tools (isolated so multilib issues don\'t block core)\napt_install "build tools" git make gcc\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 gcc-multilib 2>/dev/null || true\n\n# 4. Wine and complete 32-bit/64-bit graphics runtime\napt_install "wine runtime" wine wine64 winbind\napt_install "32-bit graphics runtime" \\\n    wine32:i386 libgl1:i386 libglx0:i386 libglx-mesa0:i386 libgl1-mesa-dri:i386 \\\n    mesa-vulkan-drivers:i386 libx11-6:i386 libxext6:i386 libxfixes3:i386 \\\n    libxcomposite1:i386 libxrandr2:i386 libxinerama1:i386 libxi6:i386 libasound2:i386 \\\n    libpulse0:i386 libgnutls30:i386 libvulkan1:i386\n\n# 4b. NVIDIA 3D Hardware Userspace Driver & Vulkan Provisioner\nif [ -c /dev/nvidiactl ] && [ -c /dev/nvidia0 ]; then\n    echo "▶ Initializing NVIDIA Tesla T4 userspace driver and Vulkan ICD..."\n    if [ -f "$BENCH/probe_nvidia_userspace.sh" ]; then\n        bash "$BENCH/probe_nvidia_userspace.sh" || true\n    fi\n    work="$BENCH/nvidia-userspace-probe"\n    if [ -f "$work/env.sh" ]; then\n        # shellcheck disable=SC1090\n        . "$work/env.sh"\n    fi\nfi\n\n# 4c. VirtualGL 3.1.5 installation\nif [ -c /dev/nvidiactl ] && [ -c /dev/nvidia0 ]; then\n    if ! [ -x /opt/VirtualGL/bin/vglrun ]; then\n        VGL64=/tmp/virtualgl_3.1.5_amd64.deb\n        VGL32=/tmp/virtualgl32_3.1.5_amd64.deb\n        wget -q https://github.com/VirtualGL/virtualgl/releases/download/3.1.5/virtualgl_3.1.5_amd64.deb -O "$VGL64" || true\n        wget -q https://github.com/VirtualGL/virtualgl/releases/download/3.1.5/virtualgl32_3.1.5_amd64.deb -O "$VGL32" || true\n        if echo \'df3f7788ce41b182a47c0d298e5cd6d2d63579522cb41825970b7726e825485e  /tmp/virtualgl_3.1.5_amd64.deb\' | sha256sum -c - && \\\n           echo \'7dc63ecb47d7ab9cb486bb21a9a968dee853380da760cc7f36f616d09ec62b9e  /tmp/virtualgl32_3.1.5_amd64.deb\' | sha256sum -c -; then\n            apt-get install -y --no-install-recommends "$VGL64" "$VGL32" || true\n        else\n            echo \'VirtualGL download or checksum failed; GPU GL bridge unavailable.\'\n        fi\n    fi\nelse\n    echo \'NVIDIA GPU devices unavailable; GPU 3D cannot be enabled inside this container.\'\nfi\nif [ -c /dev/nvidia0 ]; then\n    # Ensure nvidia-modeset character device node exists\n    if [ ! -c /dev/nvidia-modeset ]; then\n        mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || nvidia-modprobe -m 2>/dev/null || true\n    fi\n    echo "▶ Installing NVIDIA Xorg server support..."\n    apt_install "NVIDIA Xorg GLX server support" xserver-xorg-core\n    # Ensure host NVIDIA userspace libraries take precedence\n    if [ -d /usr/local/nvidia/lib64 ]; then\n        echo "/usr/local/nvidia/lib64" > /etc/ld.so.conf.d/00-nvidia-host.conf\n        ldconfig 2>/dev/null || true\n    fi\nfi\n\n# 4d. DXVK 2.6.2 (DirectX 9/10/11 -> Vulkan) installation\nDXVK_TAR=/tmp/dxvk-2.6.2.tar.gz\nif [ ! -d /opt/gamelab-dxvk/dxvk-2.6.2 ]; then\n    echo "▶ Downloading and extracting DXVK 2.6.2 Vulkan translation layer..."\n    wget -q https://github.com/doitsujin/dxvk/releases/download/v2.6.2/dxvk-2.6.2.tar.gz -O "$DXVK_TAR" || true\n    if echo \'17761876556afd55736cb895d184f5a1c55d43350f1b1e3b129f8d28706d7992  /tmp/dxvk-2.6.2.tar.gz\' | sha256sum -c -; then\n        mkdir -p /opt/gamelab-dxvk\n        tar -xzf "$DXVK_TAR" -C /opt/gamelab-dxvk || true\n    else\n        echo \'Warning: DXVK download or checksum failed; DXVK unavailable.\'\n    fi\nfi\n\n# 4e. VKD3D-Proton 2.14.1 (DirectX 12 -> Vulkan) installation for modern AAA games\nVKD3D_TAR=/tmp/vkd3d-proton-2.14.1.tar.zst\nif [ ! -d /opt/gamelab-vkd3d ]; then\n    echo "▶ Downloading VKD3D-Proton (DirectX 12) translation layer for modern AAA games..."\n    wget -q https://github.com/HansKristian-Work/vkd3d-proton/releases/download/v2.14.1/vkd3d-proton-2.14.1.tar.zst -O "$VKD3D_TAR" || true\n    if [ -f "$VKD3D_TAR" ] && [ $(stat -c%s "$VKD3D_TAR" 2>/dev/null || echo 0) -gt 1000000 ]; then\n        mkdir -p /opt/gamelab-vkd3d\n        tar --zstd -xf "$VKD3D_TAR" -C /opt/gamelab-vkd3d 2>/dev/null || \\\n        tar -xf "$VKD3D_TAR" -C /opt/gamelab-vkd3d 2>/dev/null || true\n    fi\nfi\n\n# 5. Extra libraries for Lutris and Wine Gecko\napt_install "extra libraries" python3-tk python3.12 python3-dbus python3-gi python3-yaml python3-lxml python3-requests python3-pil\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 wine-gecko 2>/dev/null || true\n\n# 6. Python streaming, interface and gaming packages\npython3 -m pip install --no-cache-dir \\\n    \'selkies[jupyter]==2.1.0\' \'pygame==2.6.1\' \'psutil\' \'legendary-gl\'\n\n# 7. Steam installation (Valve official deb)\nSTEAM_DEB="/tmp/steam_latest.deb"\nif [ ! -f "$STEAM_DEB" ] || [ $(stat -c%s "$STEAM_DEB" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://repo.steampowered.com/steam/archive/stable/steam_latest.deb -O "$STEAM_DEB" || true\nfi\nif [ -f "$STEAM_DEB" ] && [ $(stat -c%s "$STEAM_DEB") -gt 1000000 ]; then\n    apt_install "steam deb" "$STEAM_DEB"\nelse\n    apt_install "steam apt" steam\nfi\n\n# 8. Heroic Games Launcher\nHEROIC_DEB="/tmp/heroic.deb"\nif [ ! -f "$HEROIC_DEB" ] || [ $(stat -c%s "$HEROIC_DEB" 2>/dev/null || echo 0) -lt 10000000 ]; then\n    wget -q https://github.com/Heroic-Games-Launcher/HeroicGamesLauncher/releases/download/v2.22.4/Heroic-2.22.4-linux-amd64.deb -O "$HEROIC_DEB" || true\nfi\nif [ -f "$HEROIC_DEB" ] && [ $(stat -c%s "$HEROIC_DEB") -gt 10000000 ]; then\n    dpkg -i "$HEROIC_DEB" || apt-get install -f -y -o DPkg::Lock::Timeout=120 || true\nfi\n\n# 9. Lutris & Minigalaxy\napt_install "lutris" lutris\napt_install "minigalaxy" minigalaxy\n\n# 10. EA App installer download\nEA_EXE="/tmp/ea-store/EAappInstaller.exe"\nif [ ! -f "$EA_EXE" ] || [ $(stat -c%s "$EA_EXE" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://origin-a.akamaihd.net/EA-Desktop-Client-Download/installer-releases/EAappInstaller.exe -O "$EA_EXE" || true\nfi\n\n# 11. Configure unprivileged gamer user\nid -u gamer >/dev/null 2>&1 || useradd -m -s /bin/bash -G audio,video gamer\nfor sub in .config .local .steam game-builds logs ea-prefix; do\n    mkdir -p "/home/gamer/$sub"\ndone\n\n# 12. Wine-GE runner installation for Heroic & Lutris\nWINE_GE_DIR="/home/gamer/.config/heroic/tools/wine"\nmkdir -p "$WINE_GE_DIR"\nWINE_GE_TAR="/tmp/wine-lutris-GE-Proton8-26-x86_64.tar.xz"\nif [ ! -d "$WINE_GE_DIR/Wine-GE-Proton8-26" ] && [ ! -d "$WINE_GE_DIR/lutris-GE-Proton8-26-x86_64" ]; then\n    echo "▶ Downloading Wine-GE-Proton8-26 compatibility runner..."\n    wget -q -c https://github.com/GloriousEggroll/wine-ge-custom/releases/download/GE-Proton8-26/wine-lutris-GE-Proton8-26-x86_64.tar.xz -O "$WINE_GE_TAR" || true\n    if [ -f "$WINE_GE_TAR" ] && [ $(stat -c%s "$WINE_GE_TAR") -gt 50000000 ]; then\n        echo "▶ Extracting Wine-GE-Proton8-26..."\n        tar -xf "$WINE_GE_TAR" -C "$WINE_GE_DIR" || true\n        for extracted in "$WINE_GE_DIR"/*; do\n            if [ -d "$extracted" ] && [ -f "$extracted/bin/wine" ]; then\n                ln -sf "$extracted" "$WINE_GE_DIR/Wine-GE-Proton8-26" || true\n                mkdir -p "/home/gamer/.local/share/lutris/runners/wine"\n                ln -sf "$extracted" "/home/gamer/.local/share/lutris/runners/wine/Wine-GE-Proton8-26" || true\n                if [ ! -f /usr/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/bin/wine || true\n                    ln -sf "$(dirname "$extracted/bin/wine")/wineserver" /usr/bin/wineserver || true\n                    ln -sf "$(dirname "$extracted/bin/wine")/wineboot" /usr/bin/wineboot || true\n                fi\n                if [ ! -f /usr/local/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/local/bin/wine || true\n                fi\n                break\n            fi\n        done\n    fi\nfi\n\n# 13. Pre-configure Heroic default settings\npython3 "$BENCH/fix_heroic_config.py" || true\n\n# 14. Pre-initialize clean 64-bit Wine prefix with Wine-GE\nif [ -x "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" ]; then\n    echo "▶ Initializing clean default Wine prefix (/home/gamer/.wine)..."\n    export WINEPREFIX="/home/gamer/.wine"\n    export WINEARCH=win64\n    export WINEDLLOVERRIDES="mscoree,mshtml=d"\n    runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n        "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" --init 2>/dev/null || true\n\n    # Pre-install DXVK and VKD3D DirectX-to-Vulkan libraries into Wine prefix\n    echo "▶ Installing DXVK and VKD3D DirectX-to-Vulkan libraries into Wine prefix..."\n    if [ -d /opt/gamelab-dxvk/dxvk-2.6.2 ]; then\n        for arch_pair in "x64:system32" "x32:syswow64"; do\n            IFS=\':\' read -r darch dsub <<< "$arch_pair"\n            for dll in d3d9.dll d3d10core.dll d3d11.dll dxgi.dll; do\n                src="/opt/gamelab-dxvk/dxvk-2.6.2/$darch/$dll"\n                dst="/home/gamer/.wine/drive_c/windows/$dsub/$dll"\n                [ -f "$src" ] && cp -f "$src" "$dst" 2>/dev/null || true\n            done\n        done\n        echo "✓ DXVK installed into Wine prefix."\n    fi\n    for vkd3d_dir in /opt/gamelab-vkd3d/vkd3d-proton-*; do\n        if [ -d "$vkd3d_dir" ]; then\n            for arch_pair in "x64:system32" "x86:syswow64"; do\n                IFS=\':\' read -r darch dsub <<< "$arch_pair"\n                for dll in d3d12.dll d3d12core.dll; do\n                    src="$vkd3d_dir/$darch/$dll"\n                    dst="/home/gamer/.wine/drive_c/windows/$dsub/$dll"\n                    [ -f "$src" ] && cp -f "$src" "$dst" 2>/dev/null || true\n                done\n            done\n            echo "✓ VKD3D-Proton (DirectX 12) installed into Wine prefix."\n            break\n        fi\n    done\n\n    # Pre-install Microsoft Visual C++ 2015-2022 runtimes (x64 and x86) into Wine prefix\n    echo "▶ Pre-installing Microsoft Visual C++ runtimes into Wine prefix..."\n    REDIST_DIR="/tmp/vc_redist"\n    mkdir -p "$REDIST_DIR"\n    wget -q -c "https://aka.ms/vs/17/release/vc_redist.x64.exe" -O "$REDIST_DIR/vc_redist.x64.exe" || true\n    wget -q -c "https://aka.ms/vs/17/release/vc_redist.x86.exe" -O "$REDIST_DIR/vc_redist.x86.exe" || true\n\n    if [ -f "$REDIST_DIR/vc_redist.x64.exe" ]; then\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" "$REDIST_DIR/vc_redist.x64.exe" /install /quiet /norestart 2>/dev/null || true\n    fi\n    if [ -f "$REDIST_DIR/vc_redist.x86.exe" ]; then\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" "$REDIST_DIR/vc_redist.x86.exe" /install /quiet /norestart 2>/dev/null || true\n    fi\n    echo "▶ Installing the Unreal Engine Visual C++ prerequisite..."\n    runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n        winetricks -q vcrun2019 2>/dev/null || true\n\n    echo "▶ Applying verified DirectInput, AMD FSR, and zero-acceleration mouse settings..."\n    for reg_entry in \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DirectInput|MouseWarpOverride|force\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\X11 Driver|GrabPointer|Y\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\X11 Driver|UseTakeFocus|N\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseSpeed|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseThreshold1|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseThreshold2|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseSensitivity|10\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\Direct3D|WINE_FULLSCREEN_FSR|1\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\Direct3D|WINE_FULLSCREEN_FSR_STRENGTH|2\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|d3d11|native,builtin\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|d3d12|native,builtin\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|dxgi|native,builtin\'; do\n        IFS=\'|\' read -r rkey rname rval <<< "$reg_entry"\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" reg add "$rkey" \\\n            /v "$rname" /t REG_SZ /d "$rval" /f 2>/dev/null || true\n    done\n    echo "✓ Wine prefix fully initialized with DXVK, VKD3D-Proton, and FSR upscaling."\nfi\n\n# 15. Inject keyboard lock into Selkies HTML (prevents Esc from exiting browser fullscreen)\npython3 -c "\nimport sys; sys.path.insert(0, \'$BENCH\')\ntry:\n    from auto_optimize import inject_keyboard_lock\n    result = inject_keyboard_lock()\n    if result: print(\'Keyboard lock injected into:\', result)\n    else: print(\'No Selkies HTML found to patch for keyboard lock.\')\nexcept Exception as e: print(\'Keyboard lock injection skipped:\', e)\n" || true\n\n# 16. Compile Selkies gamepad interposers if not present\nif [ ! -f "$BENCH/lib/x86_64-linux-gnu/selkies_input_interposer.so" ]; then\n    SOURCE="$BENCH/selkies-source"\n    if [ ! -d "$SOURCE" ]; then\n        git clone --depth 1 --branch 2.1.0 https://github.com/selkies-project/selkies.git "$SOURCE" || true\n    fi\n    if [ -d "$SOURCE/addons/input-interposer" ]; then\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install || true\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install32 2>/dev/null || true\n        make -C "$SOURCE/addons/fake-udev" || true\n        make -C "$SOURCE/addons/fake-udev" all32 2>/dev/null || true\n        mkdir -p "$BENCH/lib/x86_64-linux-gnu" "$BENCH/lib/i386-linux-gnu"\n        cp -f "$SOURCE/addons/fake-udev/libudev.so.1" "$BENCH/lib/x86_64-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n        cp -f "$SOURCE/addons/fake-udev/libudev_x86.so.1" "$BENCH/lib/i386-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n    fi\n    ln -sf "$BENCH/lib/x86_64-linux-gnu" "$BENCH/lib64" 2>/dev/null || true\n    ln -sf "$BENCH/lib/i386-linux-gnu" "$BENCH/lib32" 2>/dev/null || true\n    cp -f "$BENCH/lib/x86_64-linux-gnu/"*.so /usr/lib/x86_64-linux-gnu/ 2>/dev/null || true\n    cp -f "$BENCH/lib/i386-linux-gnu/"*.so /usr/lib/i386-linux-gnu/ 2>/dev/null || true\nfi\n\n# 17. Prepare /dev/input gamepad device nodes and permissions\nmkdir -p /dev/input 2>/dev/null || true\nfor i in 0 1 2 3; do touch /dev/input/js$i 2>/dev/null || true; done\nfor i in 1000 1001 1002 1003; do touch /dev/input/event$i 2>/dev/null || true; done\nchmod -R 666 /dev/input 2>/dev/null || true\n\n# 18. Set permissions and ownership\nchmod -R 755 "$BENCH"/*.sh 2>/dev/null || true\nchmod -R 777 "$BENCH" 2>/dev/null || true\nchown -R gamer:gamer /home/gamer 2>/dev/null || true\nchmod -R 777 /home/gamer 2>/dev/null || true\n\necho "=========================================================="\necho "    BENCH ENVIRONMENT SETUP COMPLETED SUCCESSFULLY!       "\necho "=========================================================="\nwineserver -k 2>/dev/null || true\nexit 0\n', encoding='utf-8')
else:
    print('▶ Updating bench repository from git...', flush=True)
    subprocess.run(['git', '-C', str(BENCH), 'reset', '--hard', 'HEAD'], check=False, timeout=30)
    subprocess.run(['git', '-C', str(BENCH), 'clean', '-fd'], check=False, timeout=30)
    subprocess.run(['git', '-C', str(BENCH), 'pull', '--force'], check=False, timeout=30)

# Use the setup bundled with this notebook, including its GPU driver probes.
# The public repository can lag behind a newly uploaded notebook version.
setup_sh.write_text('#!/bin/bash\nset -e\n\nBENCH="/tmp/kaggle-game-dev-bench"\nLOGS="/home/gamer/logs"\nBUILDS="/tmp/game-builds"\nmkdir -p "$BENCH/logs" "$BUILDS" /tmp/ea-store\n\nexport DEBIAN_FRONTEND=noninteractive\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\n\necho "=========================================================="\necho "    7BZ DEV CLOUD GAME BOOTSTRAPPER                       "\necho "=========================================================="\n\n# 1. Kill stale apt/dpkg locks if running in dirty interactive session\necho "▶ Checking and clearing stale apt locks..."\nfuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\nfuser -k /var/lib/dpkg/lock 2>/dev/null || true\nfuser -k /var/lib/apt/lists/lock 2>/dev/null || true\nfuser -cache/apt/archives/lock 2>/dev/null || true\nrm -f /var/lib/dpkg/lock* /var/lib/apt/lists/lock* /var/cache/apt/archives/lock* 2>/dev/null || true\ndpkg --configure -a 2>/dev/null || true\n\n# Helper function for resilient apt install with timeout and retries\napt_install() {\n    local label="$1"\n    shift\n    echo "▶ Installing $label..."\n    local attempts=0\n    while [ $attempts -lt 3 ]; do\n        attempts=$((attempts + 1))\n        if apt-get install -y --no-install-recommends \\\n            -o DPkg::Lock::Timeout=120 \\\n            -o APT::Get::Assume-Yes=true \\\n            "$@"; then\n            return 0\n        fi\n        echo "Notice: apt install for $label failed (attempt $attempts/3). Retrying in 4s..."\n        sleep 4\n        fuser -k /var/lib/dpkg/lock-frontend 2>/dev/null || true\n        rm -f /var/lib/dpkg/lock* 2>/dev/null || true\n        dpkg --configure -a 2>/dev/null || true\n    done\n    echo "Warning: Could not install some packages for $label, continuing..."\n    return 0\n}\n\n# 2. Multiarch & repositories setup\ndpkg --add-architecture i386 || true\nif [ -f /etc/apt/sources.list.d/ubuntu.sources ]; then\n    sed -i \'s/Components: main universe/Components: main universe multiverse/g\' /etc/apt/sources.list.d/ubuntu.sources || true\nfi\n\n# Resilient apt update\napt-get update -o DPkg::Lock::Timeout=120 -qq || true\n\n# 3. Core desktop, X11, audio, and compression utilities\napt_install "core desktop" \\\n    xvfb openbox tint2 xterm pulseaudio pulseaudio-utils xauth dbus-x11 \\\n    libgl1-mesa-dri libgbm1 libpixman-1-0 libxkbcommon0 mesa-utils x11-utils vulkan-tools \\\n    xdotool xinput wmctrl wget curl cabextract p7zip-full zstd winetricks python3-pygame python3-aiohttp\n\n# Build tools (isolated so multilib issues don\'t block core)\napt_install "build tools" git make gcc\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 gcc-multilib 2>/dev/null || true\n\n# 4. Wine and complete 32-bit/64-bit graphics runtime\napt_install "wine runtime" wine wine64 winbind\napt_install "32-bit graphics runtime" \\\n    wine32:i386 libgl1:i386 libglx0:i386 libglx-mesa0:i386 libgl1-mesa-dri:i386 \\\n    mesa-vulkan-drivers:i386 libx11-6:i386 libxext6:i386 libxfixes3:i386 \\\n    libxcomposite1:i386 libxrandr2:i386 libxinerama1:i386 libxi6:i386 libasound2:i386 \\\n    libpulse0:i386 libgnutls30:i386 libvulkan1:i386\n\n# 4b. NVIDIA 3D Hardware Userspace Driver & Vulkan Provisioner\nif [ -c /dev/nvidiactl ] && [ -c /dev/nvidia0 ]; then\n    echo "▶ Initializing NVIDIA Tesla T4 userspace driver and Vulkan ICD..."\n    if [ -f "$BENCH/probe_nvidia_userspace.sh" ]; then\n        bash "$BENCH/probe_nvidia_userspace.sh" || true\n    fi\n    work="$BENCH/nvidia-userspace-probe"\n    if [ -f "$work/env.sh" ]; then\n        # shellcheck disable=SC1090\n        . "$work/env.sh"\n    fi\nfi\n\n# 4c. VirtualGL 3.1.5 installation\nif [ -c /dev/nvidiactl ] && [ -c /dev/nvidia0 ]; then\n    if ! [ -x /opt/VirtualGL/bin/vglrun ]; then\n        VGL64=/tmp/virtualgl_3.1.5_amd64.deb\n        VGL32=/tmp/virtualgl32_3.1.5_amd64.deb\n        wget -q https://github.com/VirtualGL/virtualgl/releases/download/3.1.5/virtualgl_3.1.5_amd64.deb -O "$VGL64" || true\n        wget -q https://github.com/VirtualGL/virtualgl/releases/download/3.1.5/virtualgl32_3.1.5_amd64.deb -O "$VGL32" || true\n        if echo \'df3f7788ce41b182a47c0d298e5cd6d2d63579522cb41825970b7726e825485e  /tmp/virtualgl_3.1.5_amd64.deb\' | sha256sum -c - && \\\n           echo \'7dc63ecb47d7ab9cb486bb21a9a968dee853380da760cc7f36f616d09ec62b9e  /tmp/virtualgl32_3.1.5_amd64.deb\' | sha256sum -c -; then\n            apt-get install -y --no-install-recommends "$VGL64" "$VGL32" || true\n        else\n            echo \'VirtualGL download or checksum failed; GPU GL bridge unavailable.\'\n        fi\n    fi\nelse\n    echo \'NVIDIA GPU devices unavailable; GPU 3D cannot be enabled inside this container.\'\nfi\nif [ -c /dev/nvidia0 ]; then\n    # Ensure nvidia-modeset character device node exists\n    if [ ! -c /dev/nvidia-modeset ]; then\n        mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || nvidia-modprobe -m 2>/dev/null || true\n    fi\n    echo "▶ Installing NVIDIA Xorg server support..."\n    apt_install "NVIDIA Xorg GLX server support" xserver-xorg-core\n    # Ensure host NVIDIA userspace libraries take precedence\n    if [ -d /usr/local/nvidia/lib64 ]; then\n        echo "/usr/local/nvidia/lib64" > /etc/ld.so.conf.d/00-nvidia-host.conf\n        ldconfig 2>/dev/null || true\n    fi\nfi\n\n# 4d. DXVK 2.6.2 (DirectX 9/10/11 -> Vulkan) installation\nDXVK_TAR=/tmp/dxvk-2.6.2.tar.gz\nif [ ! -d /opt/gamelab-dxvk/dxvk-2.6.2 ]; then\n    echo "▶ Downloading and extracting DXVK 2.6.2 Vulkan translation layer..."\n    wget -q https://github.com/doitsujin/dxvk/releases/download/v2.6.2/dxvk-2.6.2.tar.gz -O "$DXVK_TAR" || true\n    if echo \'17761876556afd55736cb895d184f5a1c55d43350f1b1e3b129f8d28706d7992  /tmp/dxvk-2.6.2.tar.gz\' | sha256sum -c -; then\n        mkdir -p /opt/gamelab-dxvk\n        tar -xzf "$DXVK_TAR" -C /opt/gamelab-dxvk || true\n    else\n        echo \'Warning: DXVK download or checksum failed; DXVK unavailable.\'\n    fi\nfi\n\n# 4e. VKD3D-Proton 2.14.1 (DirectX 12 -> Vulkan) installation for modern AAA games\nVKD3D_TAR=/tmp/vkd3d-proton-2.14.1.tar.zst\nif [ ! -d /opt/gamelab-vkd3d ]; then\n    echo "▶ Downloading VKD3D-Proton (DirectX 12) translation layer for modern AAA games..."\n    wget -q https://github.com/HansKristian-Work/vkd3d-proton/releases/download/v2.14.1/vkd3d-proton-2.14.1.tar.zst -O "$VKD3D_TAR" || true\n    if [ -f "$VKD3D_TAR" ] && [ $(stat -c%s "$VKD3D_TAR" 2>/dev/null || echo 0) -gt 1000000 ]; then\n        mkdir -p /opt/gamelab-vkd3d\n        tar --zstd -xf "$VKD3D_TAR" -C /opt/gamelab-vkd3d 2>/dev/null || \\\n        tar -xf "$VKD3D_TAR" -C /opt/gamelab-vkd3d 2>/dev/null || true\n    fi\nfi\n\n# 5. Extra libraries for Lutris and Wine Gecko\napt_install "extra libraries" python3-tk python3.12 python3-dbus python3-gi python3-yaml python3-lxml python3-requests python3-pil\napt-get install -y --no-install-recommends -o DPkg::Lock::Timeout=120 wine-gecko 2>/dev/null || true\n\n# 6. Python streaming, interface and gaming packages\npython3 -m pip install --no-cache-dir \\\n    \'selkies[jupyter]==2.1.0\' \'pygame==2.6.1\' \'psutil\' \'legendary-gl\'\n\n# 7. Steam installation (Valve official deb)\nSTEAM_DEB="/tmp/steam_latest.deb"\nif [ ! -f "$STEAM_DEB" ] || [ $(stat -c%s "$STEAM_DEB" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://repo.steampowered.com/steam/archive/stable/steam_latest.deb -O "$STEAM_DEB" || true\nfi\nif [ -f "$STEAM_DEB" ] && [ $(stat -c%s "$STEAM_DEB") -gt 1000000 ]; then\n    apt_install "steam deb" "$STEAM_DEB"\nelse\n    apt_install "steam apt" steam\nfi\n\n# 8. Heroic Games Launcher\nHEROIC_DEB="/tmp/heroic.deb"\nif [ ! -f "$HEROIC_DEB" ] || [ $(stat -c%s "$HEROIC_DEB" 2>/dev/null || echo 0) -lt 10000000 ]; then\n    wget -q https://github.com/Heroic-Games-Launcher/HeroicGamesLauncher/releases/download/v2.22.4/Heroic-2.22.4-linux-amd64.deb -O "$HEROIC_DEB" || true\nfi\nif [ -f "$HEROIC_DEB" ] && [ $(stat -c%s "$HEROIC_DEB") -gt 10000000 ]; then\n    dpkg -i "$HEROIC_DEB" || apt-get install -f -y -o DPkg::Lock::Timeout=120 || true\nfi\n\n# 9. Lutris & Minigalaxy\napt_install "lutris" lutris\napt_install "minigalaxy" minigalaxy\n\n# 10. EA App installer download\nEA_EXE="/tmp/ea-store/EAappInstaller.exe"\nif [ ! -f "$EA_EXE" ] || [ $(stat -c%s "$EA_EXE" 2>/dev/null || echo 0) -lt 1000000 ]; then\n    wget -q https://origin-a.akamaihd.net/EA-Desktop-Client-Download/installer-releases/EAappInstaller.exe -O "$EA_EXE" || true\nfi\n\n# 11. Configure unprivileged gamer user\nid -u gamer >/dev/null 2>&1 || useradd -m -s /bin/bash -G audio,video gamer\nfor sub in .config .local .steam game-builds logs ea-prefix; do\n    mkdir -p "/home/gamer/$sub"\ndone\n\n# 12. Wine-GE runner installation for Heroic & Lutris\nWINE_GE_DIR="/home/gamer/.config/heroic/tools/wine"\nmkdir -p "$WINE_GE_DIR"\nWINE_GE_TAR="/tmp/wine-lutris-GE-Proton8-26-x86_64.tar.xz"\nif [ ! -d "$WINE_GE_DIR/Wine-GE-Proton8-26" ] && [ ! -d "$WINE_GE_DIR/lutris-GE-Proton8-26-x86_64" ]; then\n    echo "▶ Downloading Wine-GE-Proton8-26 compatibility runner..."\n    wget -q -c https://github.com/GloriousEggroll/wine-ge-custom/releases/download/GE-Proton8-26/wine-lutris-GE-Proton8-26-x86_64.tar.xz -O "$WINE_GE_TAR" || true\n    if [ -f "$WINE_GE_TAR" ] && [ $(stat -c%s "$WINE_GE_TAR") -gt 50000000 ]; then\n        echo "▶ Extracting Wine-GE-Proton8-26..."\n        tar -xf "$WINE_GE_TAR" -C "$WINE_GE_DIR" || true\n        for extracted in "$WINE_GE_DIR"/*; do\n            if [ -d "$extracted" ] && [ -f "$extracted/bin/wine" ]; then\n                ln -sf "$extracted" "$WINE_GE_DIR/Wine-GE-Proton8-26" || true\n                mkdir -p "/home/gamer/.local/share/lutris/runners/wine"\n                ln -sf "$extracted" "/home/gamer/.local/share/lutris/runners/wine/Wine-GE-Proton8-26" || true\n                if [ ! -f /usr/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/bin/wine || true\n                    ln -sf "$(dirname "$extracted/bin/wine")/wineserver" /usr/bin/wineserver || true\n                    ln -sf "$(dirname "$extracted/bin/wine")/wineboot" /usr/bin/wineboot || true\n                fi\n                if [ ! -f /usr/local/bin/wine ]; then\n                    ln -sf "$extracted/bin/wine" /usr/local/bin/wine || true\n                fi\n                break\n            fi\n        done\n    fi\nfi\n\n# 13. Pre-configure Heroic default settings\npython3 "$BENCH/fix_heroic_config.py" || true\n\n# 14. Pre-initialize clean 64-bit Wine prefix with Wine-GE\nif [ -x "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" ]; then\n    echo "▶ Initializing clean default Wine prefix (/home/gamer/.wine)..."\n    export WINEPREFIX="/home/gamer/.wine"\n    export WINEARCH=win64\n    export WINEDLLOVERRIDES="mscoree,mshtml=d"\n    runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n        "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wineboot" --init 2>/dev/null || true\n\n    # Pre-install DXVK and VKD3D DirectX-to-Vulkan libraries into Wine prefix\n    echo "▶ Installing DXVK and VKD3D DirectX-to-Vulkan libraries into Wine prefix..."\n    if [ -d /opt/gamelab-dxvk/dxvk-2.6.2 ]; then\n        for arch_pair in "x64:system32" "x32:syswow64"; do\n            IFS=\':\' read -r darch dsub <<< "$arch_pair"\n            for dll in d3d9.dll d3d10core.dll d3d11.dll dxgi.dll; do\n                src="/opt/gamelab-dxvk/dxvk-2.6.2/$darch/$dll"\n                dst="/home/gamer/.wine/drive_c/windows/$dsub/$dll"\n                [ -f "$src" ] && cp -f "$src" "$dst" 2>/dev/null || true\n            done\n        done\n        echo "✓ DXVK installed into Wine prefix."\n    fi\n    for vkd3d_dir in /opt/gamelab-vkd3d/vkd3d-proton-*; do\n        if [ -d "$vkd3d_dir" ]; then\n            for arch_pair in "x64:system32" "x86:syswow64"; do\n                IFS=\':\' read -r darch dsub <<< "$arch_pair"\n                for dll in d3d12.dll d3d12core.dll; do\n                    src="$vkd3d_dir/$darch/$dll"\n                    dst="/home/gamer/.wine/drive_c/windows/$dsub/$dll"\n                    [ -f "$src" ] && cp -f "$src" "$dst" 2>/dev/null || true\n                done\n            done\n            echo "✓ VKD3D-Proton (DirectX 12) installed into Wine prefix."\n            break\n        fi\n    done\n\n    # Pre-install Microsoft Visual C++ 2015-2022 runtimes (x64 and x86) into Wine prefix\n    echo "▶ Pre-installing Microsoft Visual C++ runtimes into Wine prefix..."\n    REDIST_DIR="/tmp/vc_redist"\n    mkdir -p "$REDIST_DIR"\n    wget -q -c "https://aka.ms/vs/17/release/vc_redist.x64.exe" -O "$REDIST_DIR/vc_redist.x64.exe" || true\n    wget -q -c "https://aka.ms/vs/17/release/vc_redist.x86.exe" -O "$REDIST_DIR/vc_redist.x86.exe" || true\n\n    if [ -f "$REDIST_DIR/vc_redist.x64.exe" ]; then\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" "$REDIST_DIR/vc_redist.x64.exe" /install /quiet /norestart 2>/dev/null || true\n    fi\n    if [ -f "$REDIST_DIR/vc_redist.x86.exe" ]; then\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" "$REDIST_DIR/vc_redist.x86.exe" /install /quiet /norestart 2>/dev/null || true\n    fi\n    echo "▶ Installing the Unreal Engine Visual C++ prerequisite..."\n    runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 WINEDLLOVERRIDES="mscoree,mshtml=d" \\\n        winetricks -q vcrun2019 2>/dev/null || true\n\n    echo "▶ Applying verified DirectInput, AMD FSR, and zero-acceleration mouse settings..."\n    for reg_entry in \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DirectInput|MouseWarpOverride|force\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\X11 Driver|GrabPointer|Y\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\X11 Driver|UseTakeFocus|N\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseSpeed|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseThreshold1|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseThreshold2|0\' \\\n        \'HKEY_CURRENT_USER\\Control Panel\\Mouse|MouseSensitivity|10\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\Direct3D|WINE_FULLSCREEN_FSR|1\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\Direct3D|WINE_FULLSCREEN_FSR_STRENGTH|2\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|d3d11|native,builtin\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|d3d12|native,builtin\' \\\n        \'HKEY_CURRENT_USER\\Software\\Wine\\DllOverrides|dxgi|native,builtin\'; do\n        IFS=\'|\' read -r rkey rname rval <<< "$reg_entry"\n        runuser -u gamer -- env WINEPREFIX="/home/gamer/.wine" WINEARCH=win64 \\\n            "$WINE_GE_DIR/Wine-GE-Proton8-26/bin/wine" reg add "$rkey" \\\n            /v "$rname" /t REG_SZ /d "$rval" /f 2>/dev/null || true\n    done\n    echo "✓ Wine prefix fully initialized with DXVK, VKD3D-Proton, and FSR upscaling."\nfi\n\n# 15. Inject keyboard lock into Selkies HTML (prevents Esc from exiting browser fullscreen)\npython3 -c "\nimport sys; sys.path.insert(0, \'$BENCH\')\ntry:\n    from auto_optimize import inject_keyboard_lock\n    result = inject_keyboard_lock()\n    if result: print(\'Keyboard lock injected into:\', result)\n    else: print(\'No Selkies HTML found to patch for keyboard lock.\')\nexcept Exception as e: print(\'Keyboard lock injection skipped:\', e)\n" || true\n\n# 16. Compile Selkies gamepad interposers if not present\nif [ ! -f "$BENCH/lib/x86_64-linux-gnu/selkies_input_interposer.so" ]; then\n    SOURCE="$BENCH/selkies-source"\n    if [ ! -d "$SOURCE" ]; then\n        git clone --depth 1 --branch 2.1.0 https://github.com/selkies-project/selkies.git "$SOURCE" || true\n    fi\n    if [ -d "$SOURCE/addons/input-interposer" ]; then\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install || true\n        make -C "$SOURCE/addons/input-interposer" PREFIX="$BENCH" install32 2>/dev/null || true\n        make -C "$SOURCE/addons/fake-udev" || true\n        make -C "$SOURCE/addons/fake-udev" all32 2>/dev/null || true\n        mkdir -p "$BENCH/lib/x86_64-linux-gnu" "$BENCH/lib/i386-linux-gnu"\n        cp -f "$SOURCE/addons/fake-udev/libudev.so.1" "$BENCH/lib/x86_64-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n        cp -f "$SOURCE/addons/fake-udev/libudev_x86.so.1" "$BENCH/lib/i386-linux-gnu/selkies_fake_udev.so" 2>/dev/null || true\n    fi\n    ln -sf "$BENCH/lib/x86_64-linux-gnu" "$BENCH/lib64" 2>/dev/null || true\n    ln -sf "$BENCH/lib/i386-linux-gnu" "$BENCH/lib32" 2>/dev/null || true\n    cp -f "$BENCH/lib/x86_64-linux-gnu/"*.so /usr/lib/x86_64-linux-gnu/ 2>/dev/null || true\n    cp -f "$BENCH/lib/i386-linux-gnu/"*.so /usr/lib/i386-linux-gnu/ 2>/dev/null || true\nfi\n\n# 17. Prepare /dev/input gamepad device nodes and permissions\nmkdir -p /dev/input 2>/dev/null || true\nfor i in 0 1 2 3; do touch /dev/input/js$i 2>/dev/null || true; done\nfor i in 1000 1001 1002 1003; do touch /dev/input/event$i 2>/dev/null || true; done\nchmod -R 666 /dev/input 2>/dev/null || true\n\n# 18. Set permissions and ownership\nchmod -R 755 "$BENCH"/*.sh 2>/dev/null || true\nchmod -R 777 "$BENCH" 2>/dev/null || true\nchown -R gamer:gamer /home/gamer 2>/dev/null || true\nchmod -R 777 /home/gamer 2>/dev/null || true\n\necho "=========================================================="\necho "    BENCH ENVIRONMENT SETUP COMPLETED SUCCESSFULLY!       "\necho "=========================================================="\nwineserver -k 2>/dev/null || true\nexit 0\n', encoding='utf-8')
probe_sh = BENCH / 'probe_nvidia_userspace.sh'
probe_sh.write_text('#!/usr/bin/env bash\n# High-Performance NVIDIA Userspace 3D Driver & Vulkan ICD Provisioner for Kaggle\n# Offloads 3D rendering to NVIDIA GPU hardware (OpenGL & Vulkan) without modifying the kernel.\nset -uo pipefail\n\nbench=/tmp/kaggle-game-dev-bench\nwork="$bench/nvidia-userspace-probe"\nmkdir -p "$work" "$work/lib64" "$work/lib32"\n\necho "=========================================================="\necho "    NVIDIA 3D HARDWARE ACCELERATION PROVISIONER           "\necho "=========================================================="\n\nif ! command -v nvidia-smi >/dev/null 2>&1; then\n    echo "Notice: nvidia-smi not found. Running in CPU-only environment."\n    exit 0\nfi\n\nversion=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | head -n 1 | tr -d \'[:space:]\')\nif [[ ! "$version" =~ ^[0-9]+(\\.[0-9]+)+$ ]]; then\n    echo "Notice: Cannot determine loaded NVIDIA driver version ($version)."\n    exit 0\nfi\n\nmajor=$(echo "$version" | cut -d. -f1)\necho "▶ Host NVIDIA Driver Version detected: $version (Major Branch: $major)"\n\ninstalled_ok=false\n\n# Tier 1: Fast APT package installation (~10 seconds)\necho "▶ Tier 1: Checking official NVIDIA 3D graphics packages for driver $major..."\nif [ "$(id -u)" -eq 0 ]; then\n    apt-get update -o DPkg::Lock::Timeout=60 -qq 2>/dev/null || true\n    for pkg in "libnvidia-gl-$major" "libnvidia-gl-${major}-server"; do\n        if apt-cache show "$pkg" >/dev/null 2>&1; then\n            echo "▶ Found matching repository package: $pkg. Installing userspace libraries..."\n            apt-get install -y --no-install-recommends \\\n                -o DPkg::Lock::Timeout=120 -o APT::Get::Assume-Yes=true \\\n                "$pkg" "${pkg}:i386" vulkan-tools libgl1-mesa-dri mesa-utils 2>/dev/null || \\\n            apt-get install -y --no-install-recommends \\\n                -o DPkg::Lock::Timeout=120 -o APT::Get::Assume-Yes=true \\\n                "$pkg" vulkan-tools libgl1-mesa-dri mesa-utils 2>/dev/null || true\n            break\n        fi\n    done\nfi\n\n# Verify if 64-bit GLX/EGL library is now present in standard paths\nif ldconfig -p | grep -q "libGLX_nvidia.so.0" || [ -f "/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0" ]; then\n    echo "✓ NVIDIA 3D graphics libraries verified via system package manager."\n    installed_ok=true\nfi\n\n# Tier 2: Direct matching NVIDIA archive extraction if APT package was unavailable\nif [ "$installed_ok" = false ]; then\n    echo "▶ Tier 2: Fetching matching NVIDIA userspace binary archive for $version..."\n    runfile="$work/NVIDIA-Linux-x86_64-$version.run"\n    if [ ! -f "$runfile" ] || [ $(stat -c%s "$runfile" 2>/dev/null || echo 0) -lt 10000000 ]; then\n        urls=(\n            "https://us.download.nvidia.com/tesla/$version/NVIDIA-Linux-x86_64-$version.run"\n            "https://us.download.nvidia.com/XFree86/Linux-x86_64/$version/NVIDIA-Linux-x86_64-$version.run"\n            "https://download.nvidia.com/XFree86/Linux-x86_64/$version/NVIDIA-Linux-x86_64-$version.run"\n        )\n        for u in "${urls[@]}"; do\n            echo "Attempting download from: $u"\n            if curl -fL --connect-timeout 10 --max-time 120 "$u" -o "$runfile" 2>/dev/null; then\n                if [ -f "$runfile" ] && [ $(stat -c%s "$runfile") -gt 10000000 ]; then\n                    echo "✓ Downloaded matching NVIDIA installer: $(stat -c%s "$runfile") bytes"\n                    break\n                fi\n            fi\n            rm -f "$runfile"\n        done\n    fi\n\n    driver="$work/NVIDIA-Linux-x86_64-$version"\n    if [ -f "$runfile" ] && [ ! -d "$driver" ]; then\n        echo "▶ Extracting userspace graphics components from archive..."\n        (cd "$work" && sh "$runfile" --extract-only) 2>/dev/null || true\n    fi\n\n    if [ -d "$driver" ]; then\n        link_libraries() {\n            local src=$1 dst=$2\n            mkdir -p "$dst"\n            find "$src" -maxdepth 1 -type f \\( -name \'libnvidia-*.so*\' -o -name \'libGLX_nvidia.so*\' -o -name \'libEGL_nvidia.so*\' \\) -exec cp -sf {} "$dst/" \\; 2>/dev/null || true\n        }\n        link_libraries "$driver" "$work/lib64"\n        [ -d "$driver/32" ] && link_libraries "$driver/32" "$work/lib32"\n\n        for libdir in "$work/lib64" "$work/lib32"; do\n            for name in libGLX_nvidia libEGL_nvidia; do\n                vlib=$(find "$libdir" -name "${name}.so.*" 2>/dev/null | head -n1)\n                [ -n "$vlib" ] && ln -sfn "$(basename "$vlib")" "$libdir/${name}.so.0" 2>/dev/null || true\n            done\n        done\n        installed_ok=true\n    fi\nfi\n\n# Step 3: Register GLVND and Vulkan ICD manifests\nmkdir -p /usr/share/glvnd/egl_vendor.d /etc/vulkan/icd.d /usr/share/vulkan/icd.d "$work"\n\ncat > "$work/10_nvidia.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}\nEOF\n\ncat > "$work/nvidia_icd.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libGLX_nvidia.so.0","api_version":"1.3.0"}}\nEOF\n\ncat > "$work/nvidia_egl_icd.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0","api_version":"1.3.0"}}\nEOF\n\ncp -f "$work/10_nvidia.json" /usr/share/glvnd/egl_vendor.d/10_nvidia.json 2>/dev/null || true\ncp -f "$work/nvidia_icd.json" /etc/vulkan/icd.d/nvidia_icd.json 2>/dev/null || true\ncp -f "$work/nvidia_icd.json" /usr/share/vulkan/icd.d/nvidia_icd.json 2>/dev/null || true\n\n# Step 4: Register libraries with dynamic linker\nif [ -d "$work/lib64" ] && [ "$(ls -A "$work/lib64" 2>/dev/null)" ]; then\n    echo "$work/lib64" > /etc/ld.so.conf.d/nvidia-gpu.conf\n    [ -d "$work/lib32" ] && [ "$(ls -A "$work/lib32" 2>/dev/null)" ] && echo "$work/lib32" >> /etc/ld.so.conf.d/nvidia-gpu.conf\nfi\nldconfig 2>/dev/null || true\n\n# Step 5: Write environment exports\n{\n    printf \'export __EGL_VENDOR_LIBRARY_FILENAMES="/usr/share/glvnd/egl_vendor.d/10_nvidia.json"\\n\'\n    printf \'export VK_DRIVER_FILES="/etc/vulkan/icd.d/nvidia_icd.json"\\n\'\n    printf \'export VK_ICD_FILENAMES="/etc/vulkan/icd.d/nvidia_icd.json"\\n\'\n    if [ -d "$work/lib64" ] && [ "$(ls -A "$work/lib64" 2>/dev/null)" ]; then\n        printf \'export LD_LIBRARY_PATH="%s:%s${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"\\n\' "$work/lib64" "$work/lib32"\n        printf \'export GAMELAB_NVIDIA_LIB32="%s"\\n\' "$work/lib32"\n    fi\n} > "$work/env.sh"\n\necho "✓ NVIDIA userspace graphics bridge successfully configured."\necho "  Vulkan ICD: /etc/vulkan/icd.d/nvidia_icd.json"\necho "  EGL Vendor: /usr/share/glvnd/egl_vendor.d/10_nvidia.json"\n', encoding='utf-8')
probe_sh.chmod(0o755)
(BENCH / 'gpu_runtime.py').write_text('"""Choose a measured graphics path for Legendary/Wine games.\n\nNVENC and nvidia-smi are deliberately not treated as proof of 3D rendering.\nRun ``python3 gpu_runtime.py diagnose`` inside the live desktop session.\n"""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\n\n\ndef run(command, env=None, timeout=12):\n    try:\n        result = subprocess.run(command, env=env, text=True, errors="replace",\n                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                                timeout=timeout, check=False)\n        return result.returncode, result.stdout\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        return 1, str(exc)\n\n\ndef clean_env():\n    env = os.environ.copy()\n    for key in ("LIBGL_ALWAYS_SOFTWARE", "GALLIUM_DRIVER", "MESA_LOADER_DRIVER_OVERRIDE",\n                "MESA_GL_VERSION_OVERRIDE", "LIBGL_DRIVERS_PATH",\n                "__GLX_VENDOR_LIBRARY_NAME"):\n        env.pop(key, None)\n    # The Selkies input interposer must not affect graphics probes.\n    env.pop("LD_PRELOAD", None)\n    return env\n\n\ndef nvidia_renderer(output):\n    vendor = re.search(r"OpenGL vendor string:\\s*(.+)", output, re.I)\n    renderer = re.search(r"OpenGL renderer string:\\s*(.+)", output, re.I)\n    # Tesla renderers often say "Tesla T4/PCIe/SSE2", without "NVIDIA".\n    return bool(vendor and "NVIDIA" in vendor.group(1).upper() and renderer\n                and not re.search(r"llvmpipe|softpipe|software", renderer.group(1), re.I))\n\n\ndef has_32bit_library(name):\n    injected = os.environ.get("GAMELAB_NVIDIA_LIB32", "")\n    if injected:\n        candidate = Path(injected) / name\n        try:\n            with candidate.open("rb") as library:\n                if library.read(5) == b"\\x7fELF\\x01":\n                    return True\n        except OSError:\n            pass\n    rc, out = run(["ldconfig", "-p"], timeout=4)\n    return rc == 0 and any(f"{name} (libc6)" in line\n                           for line in out.splitlines())\n\n\ndef dxvk_available():\n    root = Path("/opt/gamelab-dxvk/dxvk-2.6.2")\n    return all((root / arch / (name + ".dll")).is_file()\n               for arch in ("x32", "x64")\n               for name in ("d3d9", "d3d10core", "d3d11", "dxgi"))\n\n\ndef egl_device_probe(env):\n    """Test the NVIDIA EGL device directly, independent of Xorg and Xvfb."""\n    eglinfo = Path("/opt/VirtualGL/bin/eglinfo")\n    if not eglinfo.is_file():\n        return False, "VirtualGL eglinfo is not installed", ""\n    rc, listing = run([str(eglinfo), "-e"], env, timeout=8)\n    candidates = list(dict.fromkeys(re.findall(r"\\begl\\d+\\b", listing)))[:8] if rc == 0 else []\n    if not candidates:\n        candidates = ["egl0"]\n    last_result = listing[-240:]\n    for target in candidates:\n        rc, out = run([str(eglinfo), target], env, timeout=12)\n        renderer = next((line.strip() for line in out.splitlines()\n                         if line.strip().startswith("OpenGL renderer string:")), "")\n        vendor = next((line.strip() for line in out.splitlines()\n                       if line.strip().startswith("OpenGL vendor string:")), "")\n        if (rc == 0 and "NVIDIA" in vendor.upper() and renderer\n                and not re.search(r"llvmpipe|softpipe|software", renderer, re.I)):\n            return True, renderer, target\n        last_result = renderer or out[-240:]\n    return False, last_result, ""\n\n\ndef cpu_quota_count():\n    """Return the cgroup CPU quota as a whole-core budget, if one exists."""\n    try:\n        quota, period = Path("/sys/fs/cgroup/cpu.max").read_text().split()[:2]\n        if quota != "max":\n            return max(1, math.floor(int(quota) / int(period)))\n    except (OSError, ValueError, ZeroDivisionError):\n        pass\n    try:\n        quota = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text())\n        period = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text())\n        if quota > 0:\n            return max(1, math.floor(quota / period))\n    except (OSError, ValueError, ZeroDivisionError):\n        pass\n    return None\n\n\ndef software_threads():\n    override = os.environ.get("LP_NUM_THREADS", "")\n    if override.isdigit() and 1 <= int(override) <= 32:\n        return int(override)\n    try:\n        cores = len(os.sched_getaffinity(0))\n    except AttributeError:\n        cores = os.cpu_count() or 1\n    quota = cpu_quota_count()\n    if quota is not None:\n        cores = min(cores, quota)\n    reserve = 2 if cores >= 4 else 1\n    return max(1, min(8, cores - reserve))\n\n\ndef lower_software_resolution(env):\n    """Best-effort temporary Xvfb mode change; game settings may differ."""\n    requested = os.environ.get("GAMELAB_SOFTWARE_RESOLUTION", "854x480")\n    if requested == "off" or not shutil.which("xrandr") or not env.get("DISPLAY"):\n        return None\n    if requested not in ("640x360", "854x480", "1024x576", "1280x720"):\n        print(f"Unsupported GAMELAB_SOFTWARE_RESOLUTION={requested!r}; keeping desktop mode.",\n              file=sys.stderr)\n        return None\n    rc, out = run(["xrandr", "--current"], env, timeout=5)\n    match = re.search(r"\\bcurrent (\\d+) x (\\d+)\\b", out)\n    if rc != 0 or not match:\n        return None\n    previous = f"{match.group(1)}x{match.group(2)}"\n    if previous == requested:\n        return None\n    rc, out = run(["xrandr", "-s", requested], env, timeout=5)\n    if rc != 0:\n        print(f"Could not set {requested} desktop mode: {out[-200:]}", file=sys.stderr)\n        return None\n    print(f"Software desktop mode: {requested} (restore {previous} on exit)", flush=True)\n    return previous\n\n\ndef install_dxvk(prefix):\n    """Install both bitnesses without following Wine DLL symlinks."""\n    root = Path("/opt/gamelab-dxvk/dxvk-2.6.2")\n    for arch, subdir in (("x64", "system32"), ("x32", "syswow64")):\n        destination = Path(prefix) / "drive_c/windows" / subdir\n        if not destination.is_dir():\n            raise RuntimeError(f"Wine prefix is incomplete: {destination}")\n        for name in ("d3d9", "d3d10core", "d3d11", "dxgi"):\n            target = destination / (name + ".dll")\n            backup = destination / (name + ".dll.gamelab-original")\n            temporary = destination / (name + ".dll.gamelab-new")\n            shutil.copyfile(root / arch / target.name, temporary)\n            if (target.exists() or target.is_symlink()) and not backup.exists():\n                os.replace(target, backup)\n            os.replace(temporary, target)\n\n\ndef install_vkd3d(prefix):\n    """Install VKD3D-Proton (DirectX 12) if available."""\n    candidates = list(Path("/opt/gamelab-vkd3d").glob("vkd3d-proton-*"))\n    if not candidates:\n        return\n    root = candidates[0]\n    for arch, subdir in (("x64", "system32"), ("x86", "syswow64")):\n        src_dir = root / arch\n        if not src_dir.is_dir():\n            continue\n        dst_dir = Path(prefix) / "drive_c/windows" / subdir\n        if not dst_dir.is_dir():\n            continue\n        for name in ("d3d12.dll", "d3d12core.dll"):\n            src_file = src_dir / name\n            if src_file.is_file():\n                shutil.copyfile(src_file, dst_dir / name)\n\n\ndef diagnose():\n    env = clean_env()\n    facts = {\n        "display": env.get("DISPLAY", ""),\n        "nvidia_control": Path("/dev/nvidiactl").exists(),\n        "nvidia_gpu": Path("/dev/nvidia0").exists(),\n        "nvidia_modeset": Path("/dev/nvidia-modeset").exists(),\n        "nvidia_icd": any(Path(p).exists() for p in (\n            "/etc/vulkan/icd.d/nvidia_icd.json",\n            "/usr/share/vulkan/icd.d/nvidia_icd.json")) or any(\n                Path(p).is_file() and "nvidia" in Path(p).name.lower()\n                for p in (env.get("VK_DRIVER_FILES") or env.get("VK_ICD_FILENAMES") or "").split(":") if p),\n        "direct_gl": False, "virtualgl": False, "virtualgl_display": "",\n        "vulkan_device": False,\n        "vulkan_window": False,\n        "nvidia_32bit_gl": has_32bit_library("libGLX_nvidia.so.0"),\n        "vulkan_32bit_loader": has_32bit_library("libvulkan.so.1"),\n        "dxvk_files": dxvk_available(),\n    }\n    facts["egl_device"], facts["egl_renderer"], facts["egl_device_display"] = egl_device_probe(env)\n    if shutil.which("glxinfo") and env.get("DISPLAY"):\n        rc, out = run(["glxinfo", "-B"], env)\n        facts["direct_gl"] = rc == 0 and nvidia_renderer(out)\n        facts["gl_renderer"] = next((line.strip() for line in out.splitlines()\n                                     if "OpenGL renderer string:" in line), out[-240:])\n        vgl = shutil.which("vglrun") or "/opt/VirtualGL/bin/vglrun"\n        if Path(vgl).is_file() and facts["nvidia_control"] and facts["nvidia_gpu"]:\n            candidates = [facts["egl_device_display"]] if facts["egl_device"] else ["egl0"]\n            if Path("/tmp/kaggle-game-dev-bench/gpu-xorg-ready").exists():\n                candidates.append(":1")\n            for target in candidates:\n                rc, out = run([vgl, "-d", target, "glxinfo", "-B"], env)\n                if rc == 0 and nvidia_renderer(out):\n                    facts["virtualgl"] = True\n                    facts["virtualgl_display"] = target\n                    facts["vgl_renderer"] = next((line.strip() for line in out.splitlines()\n                                                   if "OpenGL renderer string:" in line), out[-240:])\n                    break\n    if facts["nvidia_control"] and facts["nvidia_gpu"] and shutil.which("vulkaninfo"):\n        rc, out = run(["vulkaninfo", "--summary"], env)\n        facts["vulkan_device"] = rc == 0 and bool(re.search(\n            r"deviceName\\s*=.*(?:NVIDIA|Tesla T4)", out, re.I))\n        facts["vulkan_summary"] = out[-500:]\n    if facts["vulkan_device"] and env.get("DISPLAY"):\n        rc, out = run([sys.executable, "-m", "selkies.gpu_probe", "--vulkan-window"], env)\n        facts["vulkan_window"] = rc == 0 and out.strip() == "true"\n    return facts\n\n\ndef legendary_install_path(app):\n    """Return an installed Epic title\'s directory from Legendary metadata."""\n    home = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\n    candidates = (\n        home / ".config/heroic/legendaryConfig/legendary/installed.json",\n        home / ".config/legendary/installed.json",\n    )\n    for installed in candidates:\n        try:\n            record = json.loads(installed.read_text(encoding="utf-8")).get(app, {})\n            path = Path(record.get("install_path") or record.get("installPath") or "")\n            if path.is_dir():\n                return path\n        except (OSError, ValueError, AttributeError):\n            continue\n    return None\n\n\ndef unreal_shipping_binary(app):\n    """Find a UE Shipping executable while excluding prerequisite launchers."""\n    install = legendary_install_path(app)\n    if install is None:\n        return None\n    matches = []\n    for path in install.rglob("*-Win64-Shipping.exe"):\n        lowered = str(path).lower()\n        if "prereq" not in lowered and "redist" not in lowered and "crashreport" not in lowered:\n            matches.append(path)\n    matches.sort(key=lambda path: ("binaries\\\\win64" not in str(path).lower().replace("/", "\\\\"), len(str(path))))\n    return matches[0] if matches else None\n\n\ndef find_game_binary(app):\n    """Find the game\'s actual executable binary (supporting UE, id Tech, Unity, custom engines)."""\n    ue_shipping = unreal_shipping_binary(app)\n    if ue_shipping and ue_shipping.is_file():\n        return ue_shipping\n\n    home = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\n    install_candidates = []\n\n    # Check installed.json metadata (exact key match, then fuzzy/case-insensitive match)\n    for installed_file in (\n        home / ".config/heroic/legendaryConfig/legendary/installed.json",\n        home / ".config/legendary/installed.json",\n    ):\n        try:\n            data = json.loads(installed_file.read_text(encoding="utf-8"))\n            if app in data:\n                record = data[app]\n                ip = Path(record.get("install_path") or record.get("installPath") or "")\n                ex = record.get("executable") or ""\n                if ip.is_dir():\n                    if ex and (ip / ex).is_file():\n                        return ip / ex\n                    for p in ip.rglob("*.exe"):\n                        low = p.name.lower()\n                        if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                            continue\n                        if ex and p.name.lower() == Path(ex).name.lower():\n                            return p\n                        if "x64" in low or "wolf" in low:\n                            return p\n                    install_candidates.append(ip)\n            for key, record in data.items():\n                if not isinstance(record, dict):\n                    continue\n                k_low = key.lower()\n                a_low = app.lower()\n                t_low = str(record.get("title", "")).lower()\n                if a_low in k_low or k_low in a_low or a_low in t_low:\n                    ip = Path(record.get("install_path") or record.get("installPath") or "")\n                    ex = record.get("executable") or ""\n                    if ip.is_dir():\n                        if ex and (ip / ex).is_file():\n                            return ip / ex\n                        for p in ip.rglob("*.exe"):\n                            low = p.name.lower()\n                            if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                                continue\n                            if ex and p.name.lower() == Path(ex).name.lower():\n                                return p\n                            if "x64" in low or "wolf" in low:\n                                return p\n                        install_candidates.append(ip)\n        except Exception:\n            pass\n\n    install_candidates.extend([\n        home / "game-builds",\n        Path("/kaggle/working"),\n    ])\n\n    # Only include Program Files directories that actually match the app or game name\n    for pf_root in (home / ".wine/drive_c/Program Files", home / ".wine/drive_c/Program Files (x86)"):\n        if pf_root.is_dir():\n            for sub in pf_root.iterdir():\n                if sub.is_dir() and not sub.name.lower().startswith("internet"):\n                    if app.lower() in sub.name.lower() or "wolf" in sub.name.lower():\n                        install_candidates.append(sub)\n\n    SYSTEM_EXES = {"iexplore.exe", "explorer.exe", "notepad.exe", "regedit.exe", "msiexec.exe", "wineboot.exe", "rundll32.exe", "cmd.exe", "taskmgr.exe"}\n    found_exes = []\n    for root in install_candidates:\n        if not root.is_dir():\n            continue\n        try:\n            for p in root.rglob("*.exe"):\n                low = p.name.lower()\n                if low in SYSTEM_EXES:\n                    continue\n                if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                    continue\n                found_exes.append(p)\n        except Exception:\n            pass\n\n    if found_exes:\n        a_low = app.lower()\n        found_exes.sort(key=lambda p: (\n            0 if a_low in p.name.lower() else 1,\n            0 if ("x64" in p.name.lower() or "64" in p.name.lower()) else 1,\n            len(p.parts)\n        ))\n        return found_exes[0]\n    return None\n\n\ndef get_input_mode() -> str:\n    """Read the active input mode (mouse_keyboard or gamepad)."""\n    mode_file = Path("/tmp/kaggle-game-dev-bench/input-mode.json")\n    try:\n        if mode_file.is_file():\n            data = json.loads(mode_file.read_text(encoding="utf-8"))\n            return data.get("mode", "mouse_keyboard")\n    except Exception:\n        pass\n    return os.environ.get("SEVEN_BZ_INPUT_MODE", "mouse_keyboard")\n\n\ndef configure_mouse(wine_bin, env):\n    """Apply verified Wine DirectInput, X11 pointer capture, and raw sensitivity."""\n    if not wine_bin:\n        return\n    values = (\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\DirectInput", "MouseWarpOverride", "force"),\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\X11 Driver", "GrabPointer", "Y"),\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\X11 Driver", "UseTakeFocus", "N"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseSpeed", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseThreshold1", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseThreshold2", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseSensitivity", "10"),\n    )\n    for key, name, value in values:\n        try:\n            subprocess.run([wine_bin, "reg", "add", key, "/v", name, "/t", "REG_SZ",\n                            "/d", value, "/f"], env=env, check=False,\n                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=8)\n        except (OSError, subprocess.TimeoutExpired):\n            pass\n\n\ndef select_graphics_backend(facts, preference="auto", is_64bit=True):\n    """Select only paths whose required presentation probe succeeded."""\n    if preference not in ("auto", "dxvk", "wined3d"):\n        raise ValueError("GAMELAB_GRAPHICS_BACKEND must be auto, dxvk, or wined3d")\n    dxvk_ready = (facts["vulkan_window"] and (facts["vulkan_32bit_loader"] or is_64bit)\n                  and facts["dxvk_files"])\n    gl_ready = facts["nvidia_32bit_gl"] or is_64bit\n    if preference == "dxvk":\n        if not dxvk_ready:\n            raise RuntimeError("DXVK requires a working Vulkan X11 swapchain, 32-bit loader, and both DLL sets")\n        return "dxvk"\n    if preference == "auto" and dxvk_ready:\n        return "dxvk"\n    if facts["direct_gl"] and gl_ready:\n        return "direct_gl"\n    if facts["virtualgl"] and gl_ready:\n        return "virtualgl"\n    if preference == "wined3d":\n        if facts["virtualgl"] and gl_ready:\n            return "virtualgl"\n        raise RuntimeError("WineD3D requires verified NVIDIA GL and 32-bit GL libraries")\n    return "software"\n\n\ndef launch(app, wine):\n    facts = diagnose()\n    env = clean_env()\n    direct_xorg = (env.get("DISPLAY", "").split(".", 1)[0] == ":1" and\n                   facts["direct_gl"] and\n                   os.environ.get("GAMELAB_DIRECT_XORG") == "1")\n    env["WINEPREFIX"] = os.environ.get("WINEPREFIX", "/home/gamer/.wine")\n    env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n    env["SDL_AUDIODRIVER"] = "pulse"\n    env["HOME"] = os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer")\n    env["__GL_SYNC_TO_VBLANK"] = "0"\n\n    # Dedicate GPU 0 exclusively to 3D rendering and physics simulation\n    env["CUDA_VISIBLE_DEVICES"] = "0"\n    env["DRI_PRIME"] = "0"\n    if not direct_xorg:\n        env["VGL_DISPLAY"] = "egl0"\n    env["DXVK_FILTER_DEVICE_NAME"] = "Tesla T4"\n    env["WINE_FULLSCREEN_FSR"] = "1"\n    env["WINE_FULLSCREEN_FSR_STRENGTH"] = "2"\n    env["VKD3D_CONFIG"] = "dxr11,dxr"\n    env["VKD3D_FEATURE_LEVEL"] = "12_1"\n\n    # Guarantee full OpenGL 4.5 Compatibility Profile & WGL extensions for Wine/idTech\n    if not direct_xorg:\n        env["MESA_GL_VERSION_OVERRIDE"] = "4.5COMPAT"\n        env["MESA_GLSL_VERSION_OVERRIDE"] = "450"\n        env["__NV_PRIME_RENDER_OFFLOAD"] = "1"\n        env["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"\n    if not env.get("DISPLAY"):\n        env["DISPLAY"] = ":0"\n\n    # Propagate hardware profile to Legendary configuration\n    try:\n        leg_cfg_path = Path(env["HOME"]) / ".config/legendary/config.ini"\n        leg_cfg_path.parent.mkdir(parents=True, exist_ok=True)\n        current_cfg = leg_cfg_path.read_text(encoding="utf-8") if leg_cfg_path.is_file() else ""\n        if "[default.env]" not in current_cfg:\n            with leg_cfg_path.open("a", encoding="utf-8") as f:\n                f.write("\\n[default.env]\\nVGL_DISPLAY = egl0\\nMESA_GL_VERSION_OVERRIDE = 4.5COMPAT\\nMESA_GLSL_VERSION_OVERRIDE = 450\\n__NV_PRIME_RENDER_OFFLOAD = 1\\n__GLX_VENDOR_LIBRARY_NAME = nvidia\\n")\n    except Exception:\n        pass\n\n    if not env.get("PULSE_SERVER"):\n        for candidate in ("/run/user/1000/pulse/native", "/tmp/pulse-PKdhtXMmr18n/native"):\n            if Path(candidate).exists():\n                env["PULSE_SERVER"] = f"unix:{candidate}"\n                break\n        if not env.get("PULSE_SERVER"):\n            try:\n                found = list(Path("/tmp").glob("pulse-*/native")) + list(Path("/run/user").glob("*/pulse/native"))\n                if found:\n                    env["PULSE_SERVER"] = f"unix:{found[0]}"\n            except Exception:\n                pass\n\n    wine_bin = wine or shutil.which("wine")\n    configure_mouse(wine_bin, env)\n\n    # Expose the virtual controller only in gamepad mode. In mouse mode Wine\n    # otherwise sees an Xbox controller even though the HUD says M+K.\n    input_mode = get_input_mode()\n    env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n    if input_mode == "mouse_keyboard":\n        env.pop("SELKIES_JS_SOCKET_PATH", None)\n        env.pop("LD_PRELOAD", None)\n        env["SDL_JOYSTICK_DEVICE"] = ""\n        env["WINEDLLOVERRIDES"] += ";winebus.sys=d"\n    else:\n        env["SELKIES_JS_SOCKET_PATH"] = ("/tmp/kaggle-game-dev-bench/direct-stream-sockets"\n                                         if direct_xorg else "/tmp")\n        env["SDL_JOYSTICK_DEVICE"] = "/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003"\n        interposer = "/usr/$LIB/selkies_input_interposer.so:/usr/$LIB/selkies_fake_udev.so"\n        env["LD_PRELOAD"] = (interposer + ":" + env.get("LD_PRELOAD", "")).rstrip(":")\n\n    target_bin = find_game_binary(app)\n    is_64bit = True\n    if target_bin and "32" in target_bin.name.lower() and "64" not in target_bin.name.lower():\n        is_64bit = False\n    launch_cwd = str(target_bin.parent) if target_bin else None\n\n    # Switch to 64-bit Wine binary for 64-bit binaries to prevent 32-bit wrapper dropping 64-bit VGL faker\n    if is_64bit and wine_bin:\n        wine64_candidate = Path(wine_bin).with_name("wine64")\n        if wine64_candidate.is_file():\n            wine_bin = str(wine64_candidate)\n\n    vgl = shutil.which("vglrun") or "/opt/VirtualGL/bin/vglrun"\n    vgl_ready = (not direct_xorg and Path(vgl).is_file() and\n                 (facts.get("virtualgl") or facts.get("egl_device")))\n    vgl_display = facts.get("virtualgl_display") or facts.get("egl_device_display") or "egl0"\n\n    # vglrun enables its dlopen interposer by default.  -dl is not a valid\n    # option in current VirtualGL and is interpreted as the executable.\n    wrap_script = Path("/tmp/kaggle-game-dev-bench/vgl-wrap.sh")\n    try:\n        wrap_script.write_text(f"#!/bin/bash\\nexec {vgl} -d {vgl_display} \\"$@\\"\\n", encoding="utf-8")\n        wrap_script.chmod(0o755)\n    except Exception:\n        pass\n\n    if vgl_ready:\n        env["VGL_DISPLAY"] = vgl_display\n        env.update(VGL_READBACK="sync", VGL_SPOIL="0", VGL_SYNC="1", VGL_COMPRESS="proxy")\n        vgl_fps = os.environ.get("GAMELAB_VGL_FPS", "60")\n        try:\n            if 0 <= float(vgl_fps) <= 240:\n                env["VGL_FPS"] = vgl_fps\n        except ValueError:\n            pass\n\n    if target_bin and wine_bin:\n        print(f"Target game binary detected: {target_bin}", flush=True)\n        try:\n            configure_mouse(str(wine_bin), env)\n        except Exception:\n            pass\n        command = [wine_bin, str(target_bin)]\n        if "wolf" in target_bin.name.lower() or "idtech" in target_bin.name.lower():\n            command.extend([\n                "+set", "com_skipIntroVideo", "1",\n                "+set", "com_skipBootSequence", "1",\n                "+set", "r_fullscreen", "0",\n                "+set", "r_mode", "-1",\n                "+set", "r_windowedWidth", "1280",\n                "+set", "r_windowedHeight", "720",\n                "+set", "vt_cuda", "0",\n                "+set", "vt_maxPPF", "16"\n            ])\n    else:\n        command = ["legendary", "launch", app, "--wine", wine]\n        if vgl_ready:\n            command.extend(["--wrapper", str(wrap_script)])\n\n    # Prepend vglrun so the process tree inherits its OpenGL interposers.\n    if vgl_ready:\n        command = [vgl, "-d", vgl_display, *command]\n\n    try:\n        selected = select_graphics_backend(\n            facts, os.environ.get("GAMELAB_GRAPHICS_BACKEND", "auto"), is_64bit=is_64bit)\n    except (ValueError, RuntimeError) as exc:\n        print(f"Graphics backend unavailable: {exc}", file=sys.stderr)\n        return 78\n    if selected == "dxvk":\n        try:\n            install_dxvk(env["WINEPREFIX"])\n        except (OSError, RuntimeError) as exc:\n            print(f"DXVK setup failed: {exc}", file=sys.stderr)\n            return 78\n        try:\n            install_vkd3d(env["WINEPREFIX"])\n        except Exception as exc:\n            print(f"VKD3D setup notice: {exc}", file=sys.stderr)\n        env["WINEDLLOVERRIDES"] = "d3d12,d3d12core,d3d11,dxgi,d3d9,d3d10core=n,b;mscoree,mshtml=d"\n        env["DXVK_HUD"] = "devinfo,fps"\n        backend = "NVIDIA Vulkan through DXVK & VKD3D + VirtualGL (GPU 0)"\n    elif selected == "direct_gl":\n        backend = "NVIDIA OpenGL"\n    elif selected == "virtualgl":\n        display = facts.get("virtualgl_display") or "egl0"\n        backend = f"NVIDIA OpenGL through VirtualGL ({display})"\n    elif os.environ.get("GAMELAB_ALLOW_SOFTWARE") == "1":\n        backend = "Mesa software (explicit opt-in)"\n        env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n        env["LIBGL_ALWAYS_SOFTWARE"] = "1"\n        env["GALLIUM_DRIVER"] = "llvmpipe"\n        env["LP_NUM_THREADS"] = str(software_threads())\n        if shutil.which("nice"):\n            command = ["nice", "-n", "5", *command]\n    else:\n        backend = "Mesa software (fallback)"\n        env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n        env["LIBGL_ALWAYS_SOFTWARE"] = "1"\n        env["GALLIUM_DRIVER"] = "llvmpipe"\n        env["LP_NUM_THREADS"] = str(software_threads())\n        if shutil.which("nice"):\n            command = ["nice", "-n", "5", *command]\n    print(f"Rendering backend: {backend}", flush=True)\n    if backend.startswith("Mesa software"):\n        print(f"llvmpipe worker threads: {env[\'LP_NUM_THREADS\']}", flush=True)\n        previous_resolution = lower_software_resolution(env)\n        try:\n            return subprocess.call(command, env=env, cwd=launch_cwd)\n        finally:\n            if previous_resolution:\n                run(["xrandr", "-s", previous_resolution], env, timeout=5)\n    return subprocess.call(command, env=env, cwd=launch_cwd)\n\n\nif __name__ == "__main__":\n    if len(sys.argv) == 2 and sys.argv[1] == "diagnose":\n        print(json.dumps(diagnose(), indent=2))\n    elif len(sys.argv) == 4 and sys.argv[1] == "launch":\n        sys.exit(launch(sys.argv[2], sys.argv[3]))\n    else:\n        print("Usage: gpu_runtime.py diagnose | launch APP WINE_BIN", file=sys.stderr)\n        sys.exit(2)\n', encoding='utf-8')
(BENCH / 'launch_idtech5.py').write_text('"""Launch a 64-bit id Tech OpenGL game on the Selkies X desktop.\n\nThe GLX extension must exist on the captured X server.  VirtualGL sends\nOpenGL work to the selected NVIDIA GLX server or EGL device and copies\ncompleted frames into the Wine window on DISPLAY. Selkies captures :0.\n"""\n\nimport argparse\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\nfrom gpu_runtime import configure_mouse, get_input_mode\n\n\nVGL = Path("/opt/VirtualGL/bin/vglrun")\n\n\ndef probe(args, env, timeout=15):\n    result = subprocess.run(args, env=env, text=True, errors="replace",\n                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                            timeout=timeout, check=False)\n    return result.returncode, result.stdout\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("game", type=Path,\n                        help="Path to a 64-bit Windows OpenGL game executable")\n    parser.add_argument("--wine", type=Path, required=True,\n                        help="Path to Wine-GE\'s bin/wine or bin/wine64")\n    parser.add_argument("--vgl-display", default="auto",\n                        help="VirtualGL target: auto, :1, or egl0 (default: auto)")\n    parser.add_argument("--direct-xorg", action="store_true",\n                        help="Run directly on the NVIDIA Xorg display :1 (use its Selkies stream)")\n    parser.add_argument("--egl-device", default=None,\n                        help="Deprecated alias for --vgl-display")\n    parser.add_argument("--prefix", type=Path,\n                        default=Path("/home/gamer/.wine"))\n    parser.add_argument("game_args", nargs=argparse.REMAINDER)\n    args = parser.parse_args()\n\n    game = args.game.resolve()\n    wine = args.wine.resolve()\n    if not game.is_file() or not wine.is_file() or (not args.direct_xorg and not VGL.is_file()):\n        parser.error("game, Wine binary, and selected graphics launcher must exist")\n\n    env = os.environ.copy()\n    env["DISPLAY"] = ":1" if args.direct_xorg else (env.get("DISPLAY") or ":0")\n    env["WINEPREFIX"] = str(args.prefix)\n    env["VGL_COMPRESS"] = "proxy"\n    env["SDL_AUDIODRIVER"] = "pulse"\n    overrides = env.get("WINEDLLOVERRIDES", "")\n    env["WINEDLLOVERRIDES"] = (overrides + ";" if overrides else "") + "opengl32=b"\n    # These variables cannot add GLX to Xvfb or select the VirtualGL GPU.\n    for key in ("LIBGL_ALWAYS_SOFTWARE", "GALLIUM_DRIVER",\n                "MESA_LOADER_DRIVER_OVERRIDE", "MESA_GL_VERSION_OVERRIDE",\n                "MESA_GLSL_VERSION_OVERRIDE", "__NV_PRIME_RENDER_OFFLOAD",\n                "__GLX_VENDOR_LIBRARY_NAME"):\n        env.pop(key, None)\n\n    if args.direct_xorg:\n        # A direct GLX context and the captured window must live on the same\n        # NVIDIA X server. VirtualGL\'s cross-server drawable mapping is not\n        # reliable for this game\'s multi-context WGL path.\n        env.pop("VGL_DISPLAY", None)\n        for key in ("VGL_SYNC", "VGL_READBACK", "VGL_LOGO", "VGL_PROFILE"):\n            env.pop(key, None)\n        input_mode = get_input_mode()\n        env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        if input_mode == "mouse_keyboard":\n            # Wine must not enumerate Selkies\' virtual Xbox devices in FPS\n            # mouse mode. X11/XTEST still carries pointer and keyboard input.\n            env.pop("LD_PRELOAD", None)\n            env.pop("SELKIES_JS_SOCKET_PATH", None)\n            env["SDL_JOYSTICK_DEVICE"] = ""\n            env["WINEDLLOVERRIDES"] += ";winebus.sys=d"\n        else:\n            socket_dir = Path("/tmp/kaggle-game-dev-bench/direct-stream-sockets")\n            candidates = [\n                "/usr/$LIB/selkies_input_interposer.so:/usr/$LIB/selkies_fake_udev.so",\n                "/tmp/kaggle-game-dev-bench/lib64/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/lib64/selkies_fake_udev.so",\n                "/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so"\n            ]\n            interposer = next((c for c in candidates if Path(c.split(":")[0].replace("$LIB", "lib64")).is_file() or Path(c.split(":")[0].replace("$LIB", "lib/x86_64-linux-gnu")).is_file()), candidates[0])\n            env["SELKIES_JS_SOCKET_PATH"] = str(socket_dir) if socket_dir.is_dir() else "/tmp"\n            env["SDL_JOYSTICK_DEVICE"] = "/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003"\n            env["LD_PRELOAD"] = (interposer + ":" + env.get("LD_PRELOAD", "")).rstrip(":")\n        try:\n            rc, info = probe(["glxinfo", "-B"], env)\n            if rc or not re.search(r"OpenGL vendor string:\\s*NVIDIA", info):\n                raise RuntimeError("NVIDIA GLX is unavailable on :1: " + info[-500:])\n        except (OSError, subprocess.TimeoutExpired, RuntimeError) as exc:\n            print(f"Graphics prerequisite failed: {exc}", file=sys.stderr)\n            return 78\n        wine_bin = wine.with_name("wine64") if wine.with_name("wine64").is_file() else wine\n        configure_mouse(str(wine_bin), env)\n        command = [str(wine_bin), str(game)]\n        command.extend(args.game_args or\n                       ["+set", "com_skipIntroVideo", "1",\n                        "+set", "com_skipBootSequence", "1",\n                        "+set", "r_fullscreen", "0", "+set", "r_mode", "-1",\n                        "+set", "r_windowedWidth", "1280",\n                        "+set", "r_windowedHeight", "720",\n                        "+set", "in_mouse", "1"])\n        print("OpenGL target: direct NVIDIA Xorg :1", flush=True)\n        print("Input mode:", input_mode, flush=True)\n        print("Stream this display using start_direct_stream.sh", flush=True)\n        return subprocess.call(command, cwd=game.parent, env=env)\n\n    try:\n        rc, extensions = probe(["xdpyinfo", "-queryExtensions"], env)\n        if rc or not re.search(r"(?m)^\\s*GLX\\s+\\(opcode:", extensions):\n            raise RuntimeError(\n                f"{env[\'DISPLAY\']} does not advertise GLX. Wine\'s winex11.drv "\n                "will disable WGL. Restart the captured Xvfb with GLX enabled "\n                "or use a GLX-capable captured X server."\n            )\n\n        selected = args.egl_device or args.vgl_display\n        if selected == "auto":\n            candidates = ([":1"] if Path("/tmp/kaggle-game-dev-bench/gpu-xorg-ready").exists()\n                          else []) + ["egl0"]\n        else:\n            candidates = [selected]\n        failures = []\n        for candidate in candidates:\n            rc, info = probe([str(VGL), "-d", candidate, "glxinfo", "-B"], env)\n            if (not rc and re.search(r"OpenGL vendor string:\\s*NVIDIA", info)\n                    and not re.search(r"OpenGL renderer string:.*(?:llvmpipe|softpipe)", info, re.I)):\n                selected = candidate\n                break\n            failures.append(f"{candidate}: {info[-500:]}")\n        else:\n            raise RuntimeError("VirtualGL NVIDIA probe failed:\\n" + "\\n".join(failures))\n    except (OSError, subprocess.TimeoutExpired, RuntimeError) as exc:\n        print(f"Graphics prerequisite failed: {exc}", file=sys.stderr)\n        return 78\n\n    # vglrun already preloads libdlfaker and libvglfaker by default.\n    wine_bin = wine.with_name("wine64") if wine.with_name("wine64").is_file() else wine\n    env["VGL_DISPLAY"] = selected\n    command = [str(VGL), "-d", selected, str(wine_bin), str(game)]\n    if not args.game_args:\n        command.extend(["-window", "+com_skipIntroVideo", "1", "+r_fullscreen", "0"])\n    else:\n        command.extend(args.game_args)\n    print("OpenGL target:", selected, flush=True)\n    print("Wine window:", env["DISPLAY"], flush=True)\n    return subprocess.call(command, cwd=game.parent, env=env)\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', encoding='utf-8')
(BENCH / 'start_direct_stream.sh').write_text('#!/bin/bash\n# Stream the NVIDIA Xorg display directly. Run as root after the notebook\'s\n# ordinary desktop session has created PulseAudio and gpu_xorg.sh has run.\nset -euo pipefail\nBENCH=/tmp/kaggle-game-dev-bench\nPORT="${DIRECT_STREAM_PORT:-8080}"\nWIDTH="${DIRECT_STREAM_WIDTH:-1280}"\nHEIGHT="${DIRECT_STREAM_HEIGHT:-720}"\nORIGIN="${SELKIES_ALLOWED_ORIGINS:-https://kkb-production.jupyter-proxy.kaggle.net}"\nif [ "$(id -u)" -ne 0 ]; then\n    echo \'This Xorg installation requires root for Selkies MIT-SHM capture.\' >&2\n    exit 1\nfi\nbash "$BENCH/gpu_xorg.sh"\ntest -e "$BENCH/gpu-xorg-ready"\nDISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" \\\n    glxinfo -B | grep -q \'OpenGL vendor string: NVIDIA\'\nif command -v xrandr >/dev/null; then\n    DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" \\\n        xrandr -s "${WIDTH}x${HEIGHT}" 2>/dev/null || true\nfi\n\n# Use one stable, gamer-owned PulseAudio runtime for Wine and Selkies. A\n# socket pathname alone is not proof of a running or authenticated server.\nAUDIO_RUNTIME="$BENCH/pulse-runtime"\nAUDIO_COOKIE=/home/gamer/.config/pulse/cookie\nPULSE_SERVER="unix:$AUDIO_RUNTIME/pulse/native"\ninstall -d -o gamer -g gamer -m 700 /home/gamer/.config/pulse "$AUDIO_RUNTIME" "$AUDIO_RUNTIME/pulse"\nif [ -f "$AUDIO_COOKIE" ]; then\n    chown gamer:gamer "$AUDIO_COOKIE"\n    chmod 600 "$AUDIO_COOKIE"\nfi\npulse_user() {\n    runuser -u gamer -- env HOME=/home/gamer XDG_RUNTIME_DIR="$AUDIO_RUNTIME" \\\n        PULSE_SERVER="$PULSE_SERVER" PULSE_COOKIE="$AUDIO_COOKIE" "$@"\n}\nif ! pulse_user pactl info >/dev/null 2>&1; then\n    runuser -u gamer -- env HOME=/home/gamer XDG_RUNTIME_DIR="$AUDIO_RUNTIME" \\\n        PULSE_COOKIE="$AUDIO_COOKIE" \\\n        pulseaudio --daemonize=yes --exit-idle-time=-1 --disallow-exit=yes \\\n        >> "$BENCH/logs/pulseaudio.log" 2>&1 || true\nfi\naudio_ready=false\nfor attempt in $(seq 1 20); do\n    if [ -S "$AUDIO_RUNTIME/pulse/native" ] && pulse_user pactl info >/dev/null 2>&1; then\n        audio_ready=true\n        break\n    fi\n    sleep 0.5\ndone\nif [ "$audio_ready" = true ]; then\n    if ! pulse_user pactl list short sinks | awk \'$2 == "output" { found=1 } END { exit !found }\'; then\n        pulse_user pactl load-module module-null-sink sink_name=output \\\n            sink_properties=device.description=GameAudio >/dev/null || true\n    fi\n    pulse_user pactl set-default-sink output || true\n    pulse_user pactl set-sink-mute output 0 || true\n    pulse_user pactl set-sink-volume output 100% || true\n    if ! PULSE_SERVER="$PULSE_SERVER" PULSE_COOKIE="$AUDIO_COOKIE" pactl info >/dev/null 2>&1; then\n        echo \'Selkies cannot authenticate to the gamer PulseAudio server.\' >&2\n    fi\nelse\n    echo "PulseAudio did not start; see $BENCH/logs/pulseaudio.log" >&2\nfi\nprintf \'%s\' "$PULSE_SERVER" > "$BENCH/pulse-server-path"\nchmod 644 "$BENCH/pulse-server-path"\nexport DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" PULSE_SERVER PULSE_COOKIE="$AUDIO_COOKIE"\nexport PULSE_SINK=output\nENCODER_DEVICE=0\nif [ -e /dev/nvidia1 ]; then ENCODER_DEVICE=1; fi\nexport CUDA_VISIBLE_DEVICES="$ENCODER_DEVICE"\nexport SELKIES_ALLOWED_ORIGINS="$ORIGIN"\nexport SELKIES_JS_SOCKET_PATH="$BENCH/direct-stream-sockets"\nmkdir -p "$SELKIES_JS_SOCKET_PATH" "$BENCH/logs"\ntest -f "$BENCH/web-root/index.html"\n# Keep the system Openbox settings, with a rule for the game\'s Wine window.\n# EWMH fullscreen removes both the title bar and tint2\'s reserved screen area.\npython3 - "$BENCH/direct-openbox-rc.xml" <<\'PY\'\nfrom pathlib import Path\nimport sys\n\nsource = Path(\'/etc/xdg/openbox/rc.xml\').read_text(encoding=\'utf-8\')\nrule = \'\'\'<applications>\n  <application class="wolfneworder_x64.exe">\n    <decor>no</decor>\n    <fullscreen>yes</fullscreen>\n    <focus>yes</focus>\n  </application>\n  <application class="*dev_launcher*">\n    <decor>no</decor>\n    <position force="yes"><x>0</x><y>0</y></position>\n    <focus>yes</focus>\n  </application>\n  <application title="*7BZ Cloud Gaming*">\n    <decor>no</decor>\n    <position force="yes"><x>0</x><y>0</y></position>\n    <focus>yes</focus>\n  </application>\'\'\'\nif source.count(\'<applications>\') != 1:\n    raise SystemExit(\'Cannot locate Openbox application rules\')\nPath(sys.argv[1]).write_text(source.replace(\'<applications>\', rule, 1), encoding=\'utf-8\')\nPY\nstart_desktop_program() {\n    local label="$1"\n    shift\n    local pidfile="$BENCH/direct-${label}.pid"\n    if [ -f "$pidfile" ] && kill -0 "$(cat "$pidfile")" 2>/dev/null; then\n        return\n    fi\n    runuser -u gamer -- env DISPLAY=:1 HOME=/home/gamer \\\n        XDG_RUNTIME_DIR="$AUDIO_RUNTIME" PULSE_SERVER="$PULSE_SERVER" \\\n        PULSE_COOKIE="$AUDIO_COOKIE" PULSE_SINK=output SDL_AUDIODRIVER=pulse \\\n        CUDA_VISIBLE_DEVICES=0 GAMELAB_DIRECT_XORG=1 \\\n        SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \\\n        PYTHONPATH=/root/.local/lib/python3.13/site-packages:/usr/local/lib/python3.13/dist-packages \\\n        "$@" > "$BENCH/logs/direct-${label}.log" 2>&1 < /dev/null &\n    echo $! > "$pidfile"\n}\nstart_desktop() {\n    chmod 666 "$SELKIES_JS_SOCKET_PATH"/*.sock 2>/dev/null || true\n    # The HUD\'s input-mode switch and diagnostics live on this control port.\n    # The fallback desktop starts it separately, but direct Xorg must too.\n    start_desktop_program control python3 "$BENCH/session_control.py"\n    start_desktop_program openbox openbox --config-file "$BENCH/direct-openbox-rc.xml"\n    if command -v tint2 >/dev/null; then\n        start_desktop_program tint2 tint2 -c "$BENCH/tint2rc"\n    fi\n    start_desktop_program dashboard python3 "$BENCH/dev_launcher.py"\n    start_desktop_program legendary python3 "$BENCH/legendary_gui.py"\n}\nif curl -fsS "http://127.0.0.1:${PORT}/api/health" >/dev/null 2>&1; then\n    start_desktop\n    echo "Direct desktop already streaming on port $PORT"\n    exit 0\nfi\n# This process runs as root because Xorg rejected the gamer\'s MIT-SHM attach.\n# Disable command execution and transfers on this privileged stream.\nnohup env SELKIES_ENCODER_DEVICE="$ENCODER_DEVICE" SELKIES_ENCODER=h264enc \\\n    SELKIES_AUDIO_ENABLED=true SELKIES_AUDIO_ON_START=true \\\n    SELKIES_AUDIO_DEVICE_NAME=output.monitor \\\n    python3 -m selkies --port="$PORT" --enable-basic-auth=false \\\n    --framerate=60-60 --video-bitrate=3000,800-6000 \\\n    --rate-control-mode=cbr --congestion-control=true \\\n    --backpressure-queue-size=4 \\\n    --command-enabled=false --file-transfers=none --printing-enabled=false \\\n    --web-root="$BENCH/web-root" --ui-title=\'7BZ Dev Cloud Game\' \\\n    --ui-show-sidebar=true --ui-show-logo=false \\\n    --ui-sidebar-show-video-settings=false \\\n    --ui-sidebar-show-screen-settings=false \\\n    --manual-resolution=true --manual-width="$WIDTH" --manual-height="$HEIGHT" \\\n    > "$BENCH/logs/direct-stream.log" 2>&1 < /dev/null &\necho $! > "$BENCH/direct-stream.pid"\nfor attempt in $(seq 1 20); do\n    if curl -fsS "http://127.0.0.1:${PORT}/api/health" >/dev/null 2>&1; then\n        start_desktop\n        echo "Direct NVIDIA stream ready on port $PORT"\n        exit 0\n    fi\n    sleep 0.5\ndone\necho "Direct stream did not start; see $BENCH/logs/direct-stream.log" >&2\nexit 1\n', encoding='utf-8')

# 3. Execute setup.sh with live output streaming
setup_sh.chmod(0o755)
print('▶ Running automated setup.sh...', flush=True)
proc = subprocess.Popen(['/bin/bash', str(setup_sh)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
while True:
    line = proc.stdout.readline()
    if line:
        print(line, end='', flush=True)
    elif proc.poll() is not None:
        rest = proc.stdout.read()
        if rest:
            print(rest, end='', flush=True)
        break
    else:
        time.sleep(0.05)
print(f'setup.sh completed with exit code: {proc.returncode}', flush=True)

# 4. Configure unprivileged gamer user
subprocess.run(['useradd', '-m', '-s', '/bin/bash', '-G', 'audio,video', 'gamer'], check=False)
gamer_home = Path('/home/gamer')
for sub in ['.config', '.local', '.steam', 'game-builds', 'logs', 'ea-prefix']:
    (gamer_home / sub).mkdir(parents=True, exist_ok=True)

# Grant open permissions and ensure gamer ownership recursively
subprocess.run(['chown', '-R', 'gamer:gamer', str(gamer_home)], check=False)
subprocess.run(['chmod', '-R', '777', str(gamer_home)], check=False)
subprocess.run(['chmod', '-R', '777', str(BENCH)], check=False)

# 5. Storefront verification audit
def check_paths(candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
        p = shutil.which(c)
        if p:
            return p
    return None

wine_ge_dir = gamer_home / '.config' / 'heroic' / 'tools' / 'wine'
storefront_status = {
    'steam': check_paths(['/usr/games/steam', '/usr/bin/steam']),
    'heroic': check_paths(['/usr/bin/heroic', '/opt/Heroic/heroic']),
    'legendary': check_paths(['/usr/local/bin/legendary', '/usr/bin/legendary', 'legendary']),
    'minigalaxy': check_paths(['/usr/games/minigalaxy', '/usr/bin/minigalaxy']),
    'lutris': check_paths(['/usr/games/lutris', '/usr/bin/lutris']),
    'wine': check_paths(['/usr/bin/wine', 'wine']),
    'wine_ge': check_paths([str(wine_ge_dir / 'Wine-GE-Proton8-26/bin/wine'), *[str(p) for p in wine_ge_dir.glob('*/bin/wine')]]),
    'ea_installer': str(EA_STORE / 'EAappInstaller.exe') if (EA_STORE / 'EAappInstaller.exe').exists() else None,
    'interposer_64': str(BENCH / 'lib/x86_64-linux-gnu/selkies_input_interposer.so') if (BENCH / 'lib/x86_64-linux-gnu/selkies_input_interposer.so').exists() else None,
    'interposer_32': str(BENCH / 'lib/i386-linux-gnu/selkies_input_interposer.so') if (BENCH / 'lib/i386-linux-gnu/selkies_input_interposer.so').exists() else None
}

print('================ STOREFRONT INSTALLATION AUDIT ================')
for k, v in storefront_status.items():
    print(f'  {k:15}: {v if v else "NOT FOUND / PENDING"}')
print('================================================================')


In [ ]:
from pathlib import Path
import os, subprocess

BENCH = Path('/tmp/kaggle-game-dev-bench')
BENCH.mkdir(parents=True, exist_ok=True)
(BENCH / 'logs').mkdir(parents=True, exist_ok=True)
(BENCH / 'web_client').mkdir(parents=True, exist_ok=True)

# Write the console dashboard, catalog, emergency service, and streaming HUD
(BENCH / 'dev_launcher.py').write_text('"""Next-Gen Console Dashboard for 7BZ Cloud Gaming.\n\nMatches the official 7BZ Cloud Gaming futuristic cybernetic identity:\n- Real server telemetry: CPU, Dual GPUs (G0 Render & G1 NVENC), RAM, and 60 FPS WebRTC.\n- True unified Gaming OS: Epic Games Storefront & Local Games built-in (no separate Tkinter popups).\n- Real game covers & metadata from user\'s Epic account (Gamerhbaza - 117 owned, Wolfenstein installed).\n- Real dynamic task & download manager with live progress, speed, ETA, and verification tracking.\n- Direct Xorg :1 launch with Wine-GE, PulseAudio native sound, and Selkies gamepad interposer.\n"""\nimport glob\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport threading\nimport time\nfrom typing import Any, Dict, List, Optional, Tuple\n\ntry:\n    import pygame\nexcept ImportError:\n    pygame = None\n\nimport embedded_assets\nimport game_catalog\n\nos.environ["PATH"] = f"/usr/games:/usr/local/bin:/usr/bin:{os.environ.get(\'PATH\', \'\')}"\nBENCH = Path("/tmp/kaggle-game-dev-bench")\nLOGS = Path("/home/gamer/logs")\nLOGS.mkdir(parents=True, exist_ok=True)\n\nWIDTH, HEIGHT = 1280, 720\n\n# Official 7BZ Cybernetic Theme Colors\nBG_COLOR = (11, 14, 22)\nCHASSIS_LINE = (28, 38, 56)\nPANEL_BG = (17, 23, 35)\nPANEL_BORDER = (30, 42, 62)\nHEADER_MUTED = (140, 155, 180)\nTEXT_WHITE = (245, 248, 255)\nTEXT_MUTED = (130, 145, 170)\nCYAN_NEON = (0, 240, 255)\nMAGENTA_NEON = (217, 70, 239)\nLIME_GREEN = (34, 197, 94)\nBLUE_ACCENT = (59, 130, 246)\nPLAY_CYAN = (0, 229, 255)\nDANGER_RED = (239, 68, 68)\n\nTABS = ["EPIC GAMES", "LOCAL GAMES"]\n\nif pygame:\n    pygame.init()\n    pygame.display.set_caption("7BZ Cloud Gaming")\n    display = pygame.display.set_mode((WIDTH, HEIGHT), pygame.NOFRAME)\n    screen = pygame.Surface((WIDTH, HEIGHT)).convert()\n    clock = pygame.time.Clock()\n\n    font_logo = pygame.font.SysFont("Liberation Sans", 24, bold=True)\n    font_hero_title = pygame.font.SysFont("Liberation Sans", 17, bold=True)\n    font_title = pygame.font.SysFont("Liberation Sans", 12, bold=True)\n    font_body = pygame.font.SysFont("Liberation Sans", 10, bold=True)\n    font_small = pygame.font.SysFont("Liberation Sans", 9)\n    font_tiny = pygame.font.SysFont("Liberation Sans", 8, bold=True)\n    font_btn = pygame.font.SysFont("Liberation Sans", 11, bold=True)\n    font_hud = pygame.font.SysFont("Liberation Mono", 10, bold=True)\n\n\ndef pulse_server() -> str:\n    configured = os.environ.get("PULSE_SERVER", "")\n    if configured:\n        return configured\n    candidates = [Path("/run/user/1000/pulse/native")]\n    candidates.extend(Path("/tmp").glob("pulse-*/native"))\n    for candidate in candidates:\n        if candidate.exists():\n            return f"unix:{candidate}"\n    return ""\n\n\ndef game_env() -> Dict[str, str]:\n    env = os.environ.copy()\n    env.update(DISPLAY=env.get("DISPLAY", ":1"), HOME="/home/gamer", SDL_AUDIODRIVER="pulse")\n    server = pulse_server()\n    if server:\n        env["PULSE_SERVER"] = server\n    env["SELKIES_JS_SOCKET_PATH"] = ("/tmp/kaggle-game-dev-bench/direct-stream-sockets"\n                                     if env.get("GAMELAB_DIRECT_XORG") == "1" else "/tmp")\n    env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n    env["SDL_JOYSTICK_DEVICE"] = "/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003"\n    interposer = "/usr/$LIB/selkies_input_interposer.so:/usr/$LIB/selkies_fake_udev.so"\n    env["LD_PRELOAD"] = (interposer + ":" + env.get("LD_PRELOAD", "")).rstrip(":")\n    return env\n\n\ndef wine_runner() -> str:\n    candidates = [Path("/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine"),\n                  *Path("/home/gamer/.config/heroic/tools/wine").glob("*/bin/wine"), Path("/usr/bin/wine")]\n    return str(next((path for path in candidates if path.is_file()), Path("/usr/bin/wine")))\n\n\n# Real Server Telemetry Cache\ntelemetry = {\n    "cpu": 24, "cpu_temp": 58,\n    "g0_util": 5, "g0_temp": 46, "g0_vram": 66,\n    "g1_util": 0, "g1_temp": 72, "g1_vram": 105,\n    "ram_used": 6.2, "ram_total": 30.0,\n    "disk_free": 1042.5, "disk_total": 8156.9, "disk_pct": 87.2,\n    "last_update": 0.0\n}\n\n\ndef update_telemetry():\n    now = time.monotonic()\n    if now - telemetry["last_update"] < 2.0:\n        return\n    telemetry["last_update"] = now\n    try:\n        import psutil\n        telemetry["cpu"] = int(psutil.cpu_percent())\n        mem = psutil.virtual_memory()\n        telemetry["ram_used"] = round(mem.used / (1024**3), 1)\n        telemetry["ram_total"] = round(mem.total / (1024**3), 1)\n    except Exception:\n        pass\n\n    try:\n        total, used, free = shutil.disk_usage("/home/gamer")\n        telemetry["disk_free"] = round(free / (1024**3), 1)\n        telemetry["disk_total"] = round(total / (1024**3), 1)\n        telemetry["disk_pct"] = round((used / total) * 100, 1)\n    except Exception:\n        pass\n\n    try:\n        res = subprocess.run([\n            "nvidia-smi",\n            "--query-gpu=index,utilization.gpu,temperature.gpu,memory.used",\n            "--format=csv,noheader,nounits"\n        ], capture_output=True, text=True, timeout=1.5)\n        if res.returncode == 0:\n            lines = res.stdout.strip().splitlines()\n            if len(lines) >= 1:\n                parts0 = [p.strip() for p in lines[0].split(",")]\n                if len(parts0) >= 4:\n                    telemetry["g0_util"] = int(parts0[1])\n                    telemetry["g0_temp"] = int(parts0[2])\n                    telemetry["g0_vram"] = int(parts0[3])\n            if len(lines) >= 2:\n                parts1 = [p.strip() for p in lines[1].split(",")]\n                if len(parts1) >= 4:\n                    telemetry["g1_util"] = int(parts1[1])\n                    telemetry["g1_temp"] = int(parts1[2])\n                    telemetry["g1_vram"] = int(parts1[3])\n    except Exception:\n        pass\n\n\n# Active Task State (Downloads / Verifications / Installs)\nactive_task = {\n    "proc": None,\n    "type": "idle",       # "idle", "download", "verify", "install", "uninstall"\n    "title": "",\n    "app_name": "",\n    "percent": 0.0,\n    "speed": "",\n    "eta": "",\n    "log_line": "Ready",\n    "status_text": "All tasks idle",\n    "start_time": 0.0,\n}\n\n\ndef run_legendary_task(cmd_type: str, app_name: str, game_title: str):\n    """Run a legendary operation in the background and track progress."""\n    global active_task\n    if active_task["proc"] is not None and active_task["proc"].poll() is None:\n        return "Another task is already running."\n\n    active_task["type"] = cmd_type\n    active_task["title"] = game_title\n    active_task["app_name"] = app_name\n    active_task["percent"] = 0.0\n    active_task["speed"] = "--"\n    active_task["eta"] = "--"\n    active_task["log_line"] = f"Starting {cmd_type} for {game_title}..."\n    active_task["status_text"] = f"{cmd_type.capitalize()}ing {game_title}"\n    active_task["start_time"] = time.monotonic()\n\n    def task_worker():\n        try:\n            if cmd_type == "verify":\n                cmd = ["runuser", "-u", "gamer", "--", "legendary", "verify", app_name]\n            elif cmd_type == "install":\n                cmd = ["runuser", "-u", "gamer", "--", "legendary", "install", app_name, "--yes"]\n            elif cmd_type == "uninstall":\n                cmd = ["runuser", "-u", "gamer", "--", "legendary", "uninstall", app_name, "--yes"]\n            else:\n                return\n\n            proc = subprocess.Popen(\n                cmd,\n                stdout=subprocess.PIPE,\n                stderr=subprocess.STDOUT,\n                text=True,\n                bufsize=1,\n                start_new_session=True\n            )\n            active_task["proc"] = proc\n\n            progress_re = re.compile(r"(\\d+(?:\\.\\d+)?)%")\n            speed_re = re.compile(r"(\\d+(?:\\.\\d+)?\\s*(?:KiB|MiB|GiB|B)/s)")\n            eta_re = re.compile(r"ETA:\\s*([0-9:]+)")\n\n            for line in proc.stdout:\n                line_str = line.strip()\n                if not line_str:\n                    continue\n                active_task["log_line"] = line_str[-60:]\n\n                # Check for progress percentage\n                p_match = progress_re.search(line_str)\n                if p_match:\n                    try:\n                        active_task["percent"] = float(p_match.group(1))\n                    except ValueError:\n                        pass\n\n                # Check for transfer rate\n                s_match = speed_re.search(line_str)\n                if s_match:\n                    active_task["speed"] = s_match.group(1)\n\n                # Check for ETA\n                e_match = eta_re.search(line_str)\n                if e_match:\n                    active_task["eta"] = e_match.group(1)\n\n            proc.wait()\n            if cmd_type == "uninstall":\n                game_catalog.cleanup_stale_installed_record(app_name)\n                for base in [Path("/home/gamer/Games") / game_title.replace(" ", ""), Path("/home/gamer/Games") / app_name]:\n                    if base.exists():\n                        shutil.rmtree(base, ignore_errors=True)\n\n            if proc.returncode == 0:\n                active_task["status_text"] = f"Finished {cmd_type} for {game_title}"\n                active_task["percent"] = 100.0\n            else:\n                active_task["status_text"] = f"{cmd_type.capitalize()} finished (code {proc.returncode})"\n\n            time.sleep(1)\n            load_library()\n        except Exception as exc:\n            active_task["status_text"] = f"Error: {str(exc)[:40]}"\n        finally:\n            active_task["proc"] = None\n            active_task["type"] = "idle"\n\n    threading.Thread(target=task_worker, daemon=True).start()\n    return f"Started {cmd_type} for {game_title}."\n\n\ndef cancel_active_task():\n    global active_task\n    proc = active_task.get("proc")\n    if proc is not None and proc.poll() is None:\n        try:\n            os.killpg(os.getpgid(proc.pid), 15)\n            active_task["status_text"] = "Task cancelled by user"\n            active_task["type"] = "idle"\n            return "Task cancelled."\n        except OSError as e:\n            return f"Cancel failed: {e}"\n    return "No active task."\n\n\n# Global Catalog and State\nepic_user = game_catalog.get_epic_user()\nepic_games: List[Dict[str, Any]] = []\nlocal_list: List[Dict[str, Any]] = []\n\ntab_index = 0             # 0: EPIC GAMES, 1: LOCAL GAMES\nfilter_mode = "ALL"       # "ALL" or "INSTALLED"\nselected_game_index = 0\ncarousel_page = 0\nPAGE_SIZE = 4\n\nmessage = "Select a game to view details or launch"\nactive_proc: Optional[subprocess.Popen] = None\nactive_game_title = ""\n\ntoggles = {\n    "boost": True,\n    "direct_stream": True,\n    "gamepad": True\n}\n\n\ndef load_library():\n    global epic_games, local_list, selected_game_index, epic_user\n    epic_user = game_catalog.get_epic_user()\n    epic_games = game_catalog.epic_library_games()\n    local_list = game_catalog.local_games()\n    selected_game_index = 0\n\n\nload_library()\n\n\ndef get_current_games_list() -> List[Dict[str, Any]]:\n    if tab_index == 0:\n        if filter_mode == "INSTALLED":\n            return [g for g in epic_games if g.get("is_installed")]\n        return epic_games\n    return local_list\n\n\ndef get_selected_game() -> Optional[Dict[str, Any]]:\n    games = get_current_games_list()\n    if not games:\n        return None\n    idx = max(0, min(selected_game_index, len(games) - 1))\n    return games[idx]\n\n\ndef launch_game(item: Dict[str, Any]) -> str:\n    global active_proc, active_game_title\n    if active_proc is not None and active_proc.poll() is None:\n        return f"{active_game_title} is already running."\n\n    action = item.get("action")\n    if action == "legendary" and not item.get("is_installed"):\n        return f"{item[\'title\']} is not installed. Click INSTALL GAME to download."\n\n    try:\n        if action == "legendary":\n            command = ["python3", str(BENCH / "gpu_runtime.py"), "launch", item["app_name"], wine_runner()]\n        elif action == "build":\n            path = item["install_path"]\n            if path.lower().endswith(".py"):\n                command = ["python3", path]\n            elif path.lower().endswith(".sh"):\n                command = ["bash", path]\n            else:\n                command = [path]\n        else:\n            return "Unsupported launch action."\n\n        log = (LOGS / "dashboard-launch.log").open("a", encoding="utf-8", buffering=1)\n        log.write(f"\\n--- {time.strftime(\'%Y-%m-%d %H:%M:%S\')} · Launching {item[\'title\']} ---\\n")\n        active_proc = subprocess.Popen(\n            command,\n            env=game_env(),\n            stdin=subprocess.DEVNULL,\n            stdout=log,\n            stderr=subprocess.STDOUT,\n            start_new_session=True\n        )\n        active_game_title = item["title"]\n\n        try:\n            from session_control import write_active_game\n            write_active_game(active_proc.pid, os.getpgid(active_proc.pid), active_game_title, item.get("store", "7BZ"))\n        except Exception:\n            pass\n\n        def watch():\n            active_proc.wait()\n            try:\n                from session_control import clear_active_game\n                clear_active_game(active_proc.pid)\n            except Exception:\n                pass\n\n        threading.Thread(target=watch, daemon=True).start()\n        return f"Launching {active_game_title} on Direct Xorg..."\n    except Exception as exc:\n        return f"Launch failed: {str(exc)[:60]}"\n\n\ndef stop_active_game() -> str:\n    global active_proc, active_game_title\n    if active_proc is None or active_proc.poll() is not None:\n        return "No game currently running."\n    try:\n        os.killpg(os.getpgid(active_proc.pid), 15)\n        title = active_game_title\n        active_game_title = ""\n        return f"Stopped {title}."\n    except OSError as exc:\n        return f"Error stopping: {exc}"\n\n\ndef draw_corner_brackets():\n    color = CHASSIS_LINE\n    pygame.draw.lines(screen, color, False, [(15, 35), (15, 15), (35, 15)], 2)\n    pygame.draw.lines(screen, color, False, [(WIDTH - 35, 15), (WIDTH - 15, 15), (WIDTH - 15, 35)], 2)\n    pygame.draw.lines(screen, color, False, [(15, HEIGHT - 35), (15, HEIGHT - 15), (35, HEIGHT - 15)], 2)\n    pygame.draw.lines(screen, color, False, [(WIDTH - 35, HEIGHT - 15), (WIDTH - 15, HEIGHT - 15), (WIDTH - 15, HEIGHT - 35)], 2)\n\n\ndef draw_header():\n    """Draw 7BZ Logo, Real Dual GPU/CPU Telemetry, and Active Process Status."""\n    # 1. Official 7BZ Logo\n    logo = embedded_assets.get_surface("7bz_logo")\n    if logo:\n        screen.blit(logo, (24, 15))\n    else:\n        screen.blit(font_logo.render("7BZCLOUD", True, CYAN_NEON), (30, 16))\n        screen.blit(font_logo.render("GAMING", True, MAGENTA_NEON), (190, 16))\n\n    # 2. Real Telemetry Pod (Center-Right)\n    pod_rect = pygame.Rect(WIDTH - 730, 14, 520, 40)\n    pygame.draw.rect(screen, (14, 19, 30), pod_rect, border_radius=8)\n    pygame.draw.rect(screen, (28, 38, 58), pod_rect, width=1, border_radius=8)\n\n    # Formatted Telemetry Metrics\n    m_cpu = f"CPU: {telemetry[\'cpu\']}%"\n    m_g0 = f"G0: {telemetry[\'g0_util\']}% {telemetry[\'g0_temp\']}C"\n    m_g1 = f"G1: {telemetry[\'g1_util\']}%"\n    m_ram = f"RAM: {telemetry[\'ram_used\']}G/{int(telemetry[\'ram_total\'])}G"\n    m_fps = "60 FPS"\n\n    metrics = [\n        (m_cpu, CYAN_NEON),\n        (m_g0, CYAN_NEON),\n        (m_g1, (160, 185, 220)),\n        (m_ram, MAGENTA_NEON),\n        (m_fps, LIME_GREEN),\n    ]\n\n    x = pod_rect.x + 12\n    y = pod_rect.y + 12\n    for i, (txt, col) in enumerate(metrics):\n        lbl = font_hud.render(txt, True, col)\n        screen.blit(lbl, (x, y))\n        x += lbl.get_width() + 10\n        if i < len(metrics) - 1:\n            pygame.draw.line(screen, (35, 48, 70), (x, y + 2), (x, y + 14))\n            x += 10\n\n    # 3. Active Process / Status Pod (Far Right)\n    status_rect = pygame.Rect(WIDTH - 195, 14, 180, 40)\n    if active_proc is not None and active_proc.poll() is None:\n        pygame.draw.rect(screen, (180, 25, 40), status_rect, border_radius=8)\n        lbl_stop = font_btn.render("STOP GAME", True, TEXT_WHITE)\n        screen.blit(lbl_stop, (status_rect.centerx - lbl_stop.get_width() // 2, status_rect.centery - lbl_stop.get_height() // 2))\n    else:\n        pygame.draw.rect(screen, (14, 19, 30), status_rect, border_radius=8)\n        pygame.draw.rect(screen, (28, 38, 58), status_rect, width=1, border_radius=8)\n        pygame.draw.circle(screen, LIME_GREEN, (status_rect.x + 16, status_rect.centery), 4)\n        lbl_rdy = font_tiny.render("NODE READY", True, LIME_GREEN)\n        screen.blit(lbl_rdy, (status_rect.x + 26, status_rect.centery - 6))\n        lbl_clock = font_hud.render(time.strftime("%H:%M:%S"), True, TEXT_WHITE)\n        screen.blit(lbl_clock, (status_rect.right - lbl_clock.get_width() - 10, status_rect.centery - 7))\n\n\ndef draw_tabs():\n    """Draw the 2 primary tabs: EPIC GAMES and LOCAL GAMES."""\n    tab_w = 210\n    total_w = 2 * tab_w + 16\n    start_x = (WIDTH - total_w) // 2\n    y = 66\n    h = 42\n\n    for i, title in enumerate(TABS):\n        rect = pygame.Rect(start_x + i * (tab_w + 16), y, tab_w, h)\n        is_active = (i == tab_index)\n\n        bg_col = (20, 28, 44) if is_active else (15, 20, 32)\n        pygame.draw.rect(screen, bg_col, rect, border_radius=8)\n\n        if is_active:\n            pygame.draw.rect(screen, CYAN_NEON, rect, width=2, border_radius=8)\n            text_col = CYAN_NEON\n        else:\n            pygame.draw.rect(screen, (28, 38, 56), rect, width=1, border_radius=8)\n            text_col = TEXT_WHITE\n\n        lbl = font_title.render(title, True, text_col)\n        screen.blit(lbl, (rect.centerx - lbl.get_width() // 2, rect.centery - lbl.get_height() // 2))\n\n\ndef draw_toggle(rect: pygame.Rect, state: bool):\n    pill = pygame.Rect(rect.x, rect.y, 36, 18)\n    if state:\n        pygame.draw.rect(screen, CYAN_NEON, pill, border_radius=9)\n        pygame.draw.circle(screen, (10, 15, 25), (pill.right - 9, pill.centery), 6)\n    else:\n        pygame.draw.rect(screen, (30, 40, 58), pill, border_radius=9)\n        pygame.draw.circle(screen, (140, 155, 175), (pill.x + 9, pill.centery), 6)\n\n\ndef draw_left_card():\n    """Draw CLOUD STATUS, ACCOUNT & STORAGE card."""\n    card = pygame.Rect(25, 120, 245, 575)\n    pygame.draw.rect(screen, PANEL_BG, card, border_radius=14)\n    pygame.draw.rect(screen, PANEL_BORDER, card, width=1, border_radius=14)\n\n    # Title\n    screen.blit(font_title.render("CLOUD STATUS & STORAGE", True, HEADER_MUTED), (40, 136))\n\n    # 1. Epic Games Account Profile\n    y = 168\n    pygame.draw.circle(screen, (16, 32, 54), (56, y + 16), 18)\n    pygame.draw.circle(screen, CYAN_NEON, (56, y + 16), 18, 1)\n    lbl_usr_ico = font_title.render("E", True, CYAN_NEON)\n    screen.blit(lbl_usr_ico, (51, y + 7))\n\n    user_name = epic_user.get("displayName", "Gamerhbaza")\n    screen.blit(font_body.render(user_name.upper(), True, TEXT_WHITE), (86, y + 4))\n    screen.blit(font_tiny.render("EPIC ONLINE CONNECTED", True, LIME_GREEN), (86, y + 22))\n\n    # 2. Host Storage Pool Section (Real Kaggle Storage)\n    y = 232\n    screen.blit(font_body.render("HOST STORAGE POOL", True, HEADER_MUTED), (40, y))\n\n    # Real Progress bar\n    bar_rect = pygame.Rect(40, y + 22, 215, 8)\n    pygame.draw.rect(screen, (24, 33, 48), bar_rect, border_radius=4)\n    fill_w = int(215 * (telemetry["disk_pct"] / 100.0))\n    pygame.draw.rect(screen, CYAN_NEON, (40, y + 22, max(4, fill_w), 8), border_radius=4)\n\n    free_str = f"{telemetry[\'disk_free\']} GB FREE"\n    tot_str = f" / {int(telemetry[\'disk_total\'])} GB"\n    screen.blit(font_body.render(free_str, True, CYAN_NEON), (40, y + 36))\n    screen.blit(font_small.render(tot_str, True, TEXT_MUTED), (40 + font_body.size(free_str)[0], y + 37))\n\n    # 3. View Filter Buttons (ALL vs INSTALLED)\n    y = 305\n    screen.blit(font_body.render("LIBRARY VIEW", True, HEADER_MUTED), (40, y))\n\n    btn_all = pygame.Rect(40, y + 20, 102, 28)\n    btn_inst = pygame.Rect(148, y + 20, 107, 28)\n\n    all_col = (26, 38, 60) if filter_mode == "ALL" else (18, 24, 36)\n    inst_col = (26, 38, 60) if filter_mode == "INSTALLED" else (18, 24, 36)\n\n    pygame.draw.rect(screen, all_col, btn_all, border_radius=6)\n    pygame.draw.rect(screen, inst_col, btn_inst, border_radius=6)\n\n    if filter_mode == "ALL":\n        pygame.draw.rect(screen, CYAN_NEON, btn_all, width=1, border_radius=6)\n    else:\n        pygame.draw.rect(screen, CYAN_NEON, btn_inst, width=1, border_radius=6)\n\n    lbl_all = font_tiny.render(f"ALL ({len(epic_games)})", True, TEXT_WHITE)\n    inst_count = len([g for g in epic_games if g.get("is_installed")])\n    lbl_inst = font_tiny.render(f"INSTALLED ({inst_count})", True, TEXT_WHITE)\n\n    screen.blit(lbl_all, (btn_all.centerx - lbl_all.get_width() // 2, btn_all.centery - lbl_all.get_height() // 2))\n    screen.blit(lbl_inst, (btn_inst.centerx - lbl_inst.get_width() // 2, btn_inst.centery - lbl_inst.get_height() // 2))\n\n    # 4. Interactive Feature Toggles\n    y = 385\n    screen.blit(font_body.render("CLOUD OPTIMIZATIONS", True, HEADER_MUTED), (40, y))\n    y += 24\n\n    toggles_list = [\n        ("boost", "GPU BOOST (G0/G1)"),\n        ("direct_stream", "DIRECT XORG STREAM"),\n        ("gamepad", "GAMEPAD INTERPOSER")\n    ]\n    for key, label in toggles_list:\n        screen.blit(font_small.render(label, True, TEXT_WHITE), (40, y + 2))\n        draw_toggle(pygame.Rect(card.right - 54, y, 36, 18), toggles[key])\n        y += 36\n\n    # 5. Sync Account Button\n    btn_sync = pygame.Rect(40, 638, 215, 36)\n    pygame.draw.rect(screen, (22, 32, 50), btn_sync, border_radius=8)\n    pygame.draw.rect(screen, (35, 50, 75), btn_sync, width=1, border_radius=8)\n    lbl_sync = font_btn.render("SYNC EPIC LIBRARY", True, CYAN_NEON)\n    screen.blit(lbl_sync, (btn_sync.centerx - lbl_sync.get_width() // 2, btn_sync.centery - lbl_sync.get_height() // 2))\n\n\ndef draw_center_hero():\n    """Draw featured/selected game hero banner with official 16:9 box art and direct actions."""\n    hero_rect = pygame.Rect(285, 120, 675, 265)\n    pygame.draw.rect(screen, (14, 18, 28), hero_rect, border_radius=14)\n    pygame.draw.rect(screen, PANEL_BORDER, hero_rect, width=1, border_radius=14)\n\n    game = get_selected_game()\n    if not game:\n        lbl_empty = font_title.render("No games found in selected library view.", True, TEXT_MUTED)\n        screen.blit(lbl_empty, (hero_rect.centerx - lbl_empty.get_width() // 2, hero_rect.centery))\n        return\n\n    # 1. Wide Banner Artwork\n    wide_surf = game_catalog.get_artwork(\n        game["app_name"], game.get("wide_art", ""), "wide", 673, 263, game["title"], game.get("store", "Epic")\n    )\n    if wide_surf:\n        screen.blit(wide_surf, (286, 121))\n\n    # Translucent gradient scrim on bottom for high contrast action buttons\n    scrim = pygame.Surface((673, 90), pygame.SRCALPHA)\n    for row in range(90):\n        alpha = int((row / 90.0) * 225)\n        pygame.draw.line(scrim, (10, 14, 22, alpha), (0, row), (673, row))\n    screen.blit(scrim, (286, 294))\n\n    # Top-Left Studio / Publisher Badge on Banner\n    tag_rect = pygame.Rect(hero_rect.x + 16, hero_rect.y + 14, 210, 24)\n    pygame.draw.rect(screen, (15, 22, 35), tag_rect, border_radius=6)\n    pygame.draw.rect(screen, (35, 50, 75), tag_rect, width=1, border_radius=6)\n    dev = game.get("developer", "Bethesda Softworks")\n    lbl_tag = font_tiny.render(f"STUDIO: {dev[:22].upper()}", True, CYAN_NEON)\n    screen.blit(lbl_tag, (tag_rect.centerx - lbl_tag.get_width() // 2, tag_rect.centery - lbl_tag.get_height() // 2))\n\n    # Top-Right Installed / Cloud Status Pill on banner\n    is_inst = game.get("is_installed", False)\n    pill_rect = pygame.Rect(hero_rect.right - 145, hero_rect.y + 14, 130, 24)\n    pill_bg = (16, 40, 28) if is_inst else (18, 28, 44)\n    pill_border = LIME_GREEN if is_inst else CYAN_NEON\n    pill_txt = "INSTALLED" if is_inst else "OWNED IN CLOUD"\n    pill_col = LIME_GREEN if is_inst else CYAN_NEON\n\n    pygame.draw.rect(screen, pill_bg, pill_rect, border_radius=12)\n    pygame.draw.rect(screen, pill_border, pill_rect, width=1, border_radius=12)\n    pygame.draw.circle(screen, pill_col, (pill_rect.x + 14, pill_rect.centery), 3)\n    lbl_pill = font_tiny.render(pill_txt, True, pill_col)\n    screen.blit(lbl_pill, (pill_rect.x + 24, pill_rect.centery - lbl_pill.get_height() // 2))\n\n    # Game Meta line (above buttons)\n    size_str = game.get("size", "44.5 GiB")\n    ver = game.get("version", "1.3.1")\n    meta_line = f"FOOTPRINT: {size_str}   •   VERSION: {ver}   •   DIRECT XORG :1 (TESLA T4)"\n    screen.blit(font_small.render(meta_line, True, (210, 225, 250)), (310, 305))\n\n    # Interactive Action Buttons\n    btn_y = 330\n\n    if is_inst:\n        # 1. PLAY NOW Button\n        btn_play = pygame.Rect(310, btn_y, 140, 38)\n        pygame.draw.rect(screen, PLAY_CYAN, btn_play, border_radius=8)\n        lbl_play = font_btn.render("PLAY NOW", True, (10, 15, 25))\n        screen.blit(lbl_play, (btn_play.centerx - lbl_play.get_width() // 2, btn_play.centery - lbl_play.get_height() // 2))\n\n        # 2. VERIFY Button\n        btn_ver = pygame.Rect(462, btn_y, 95, 38)\n        pygame.draw.rect(screen, (22, 32, 50), btn_ver, border_radius=8)\n        pygame.draw.rect(screen, CYAN_NEON, btn_ver, width=1, border_radius=8)\n        lbl_ver = font_btn.render("VERIFY", True, CYAN_NEON)\n        screen.blit(lbl_ver, (btn_ver.centerx - lbl_ver.get_width() // 2, btn_ver.centery - lbl_ver.get_height() // 2))\n\n        # 3. UNINSTALL Button\n        btn_un = pygame.Rect(568, btn_y, 105, 38)\n        pygame.draw.rect(screen, (32, 22, 28), btn_un, border_radius=8)\n        pygame.draw.rect(screen, DANGER_RED, btn_un, width=1, border_radius=8)\n        lbl_un = font_btn.render("UNINSTALL", True, DANGER_RED)\n        screen.blit(lbl_un, (btn_un.centerx - lbl_un.get_width() // 2, btn_un.centery - lbl_un.get_height() // 2))\n    else:\n        # INSTALL GAME Button\n        btn_install = pygame.Rect(310, btn_y, 160, 38)\n        pygame.draw.rect(screen, BLUE_ACCENT, btn_install, border_radius=8)\n        lbl_inst = font_btn.render("INSTALL GAME", True, TEXT_WHITE)\n        screen.blit(lbl_inst, (btn_install.centerx - lbl_inst.get_width() // 2, btn_install.centery - lbl_inst.get_height() // 2))\n\n\ndef draw_quick_launch():\n    """Draw the 4 poster cards of the user\'s actual game library with carousel paging."""\n    games = get_current_games_list()\n    total_games = len(games)\n    total_pages = max(1, math.ceil(total_games / PAGE_SIZE))\n\n    # Header Row\n    lbl_ql = font_title.render("QUICK-LAUNCH LIBRARY", True, TEXT_WHITE)\n    screen.blit(lbl_ql, (288, 404))\n\n    # Page info & Chevrons\n    page_str = f"Page {carousel_page + 1} of {total_pages}"\n    lbl_page = font_small.render(page_str, True, TEXT_MUTED)\n    screen.blit(lbl_page, (820, 406))\n\n    # Chevron buttons\n    lbl_prev = font_title.render("<", True, CYAN_NEON if carousel_page > 0 else (60, 75, 95))\n    lbl_next = font_title.render(">", True, CYAN_NEON if carousel_page < total_pages - 1 else (60, 75, 95))\n    screen.blit(lbl_prev, (915, 403))\n    screen.blit(lbl_next, (942, 403))\n\n    w = 155\n    h = 246\n    start_x = 288\n    gap = 18\n    y = 432\n\n    page_start = carousel_page * PAGE_SIZE\n    page_games = games[page_start:page_start + PAGE_SIZE]\n\n    for i in range(PAGE_SIZE):\n        x = start_x + i * (w + gap)\n        card_rect = pygame.Rect(x, y, w, h)\n        idx = page_start + i\n        is_selected = (idx == selected_game_index)\n\n        if i < len(page_games):\n            g = page_games[i]\n            # Poster Artwork\n            tall_surf = game_catalog.get_artwork(\n                g["app_name"], g.get("tall_art", ""), "tall", w, h, g["title"], g.get("store", "Epic")\n            )\n            if tall_surf:\n                screen.blit(tall_surf, (x, y))\n\n            # Bottom gradient scrim for card title\n            c_scrim = pygame.Surface((w, 65), pygame.SRCALPHA)\n            for r in range(65):\n                alpha = int((r / 65.0) * 220)\n                pygame.draw.line(c_scrim, (10, 14, 22, alpha), (0, r), (w, r))\n            screen.blit(c_scrim, (x, y + h - 65))\n\n            # Title\n            t = g["title"]\n            if len(t) > 17:\n                t = t[:15] + "..."\n            lbl_t = font_body.render(t, True, TEXT_WHITE)\n            screen.blit(lbl_t, (x + 8, y + h - 38))\n\n            # Store / Platform\n            lbl_st = font_tiny.render(g.get("store", "EPIC"), True, (140, 160, 190))\n            screen.blit(lbl_st, (x + 8, y + h - 20))\n\n            # Status Badge Pill (Top-Right)\n            is_inst = g.get("is_installed", False)\n            badge_rect = pygame.Rect(card_rect.right - 62, card_rect.y + 8, 54, 18)\n            badge_bg = (16, 40, 28) if is_inst else (18, 28, 44)\n            badge_col = LIME_GREEN if is_inst else CYAN_NEON\n            badge_txt = "INSTALLED" if is_inst else "OWNED"\n\n            pygame.draw.rect(screen, badge_bg, badge_rect, border_radius=4)\n            pygame.draw.rect(screen, badge_col, badge_rect, width=1, border_radius=4)\n            lbl_b = font_tiny.render(badge_txt, True, badge_col)\n            screen.blit(lbl_b, (badge_rect.centerx - lbl_b.get_width() // 2, badge_rect.centery - lbl_b.get_height() // 2))\n        else:\n            pygame.draw.rect(screen, (15, 20, 32), card_rect, border_radius=12)\n            pygame.draw.rect(screen, (24, 32, 48), card_rect, width=1, border_radius=12)\n\n        # Selection / Hover Border\n        if is_selected:\n            pygame.draw.rect(screen, CYAN_NEON, card_rect, width=2, border_radius=12)\n        else:\n            pygame.draw.rect(screen, (28, 38, 56), card_rect, width=1, border_radius=12)\n\n\ndef draw_right_card():\n    """Draw REAL EPIC TASK & DOWNLOAD MANAGER card (no fake mockup!)."""\n    card = pygame.Rect(975, 120, 280, 575)\n    pygame.draw.rect(screen, PANEL_BG, card, border_radius=14)\n    pygame.draw.rect(screen, PANEL_BORDER, card, width=1, border_radius=14)\n\n    # Title\n    screen.blit(font_title.render("EPIC TASK & DOWNLOAD MANAGER", True, HEADER_MUTED), (992, 136))\n\n    # 1. Real Operations Section\n    y = 172\n    is_busy = (active_task["proc"] is not None and active_task["proc"].poll() is None)\n\n    if is_busy:\n        t_type = active_task["type"].upper()\n        screen.blit(font_body.render(f"{t_type} IN PROGRESS", True, CYAN_NEON), (992, y))\n\n        t_title = active_task["title"]\n        if len(t_title) > 28:\n            t_title = t_title[:26] + "..."\n        screen.blit(font_small.render(t_title, True, TEXT_WHITE), (992, y + 20))\n\n        # Progress bar\n        bar_rect = pygame.Rect(992, y + 42, 246, 8)\n        pygame.draw.rect(screen, (24, 33, 48), bar_rect, border_radius=4)\n        pct = max(0.0, min(100.0, active_task["percent"]))\n        fill_w = int(246 * (pct / 100.0))\n        pygame.draw.rect(screen, CYAN_NEON, (992, y + 42, max(4, fill_w), 8), border_radius=4)\n\n        stats_str = f"{pct:.1f}%  •  {active_task[\'speed\']}  •  ETA: {active_task[\'eta\']}"\n        screen.blit(font_tiny.render(stats_str, True, (180, 200, 230)), (992, y + 56))\n\n        # Cancel Button\n        btn_cancel = pygame.Rect(992, y + 78, 246, 26)\n        pygame.draw.rect(screen, (36, 22, 28), btn_cancel, border_radius=6)\n        pygame.draw.rect(screen, DANGER_RED, btn_cancel, width=1, border_radius=6)\n        lbl_c = font_tiny.render("CANCEL TASK", True, DANGER_RED)\n        screen.blit(lbl_c, (btn_cancel.centerx - lbl_c.get_width() // 2, btn_cancel.centery - lbl_c.get_height() // 2))\n        y += 120\n    else:\n        screen.blit(font_body.render("STATUS: ALL TASKS IDLE", True, LIME_GREEN), (992, y))\n        screen.blit(font_small.render("Cloud Queue Ready", True, (140, 160, 190)), (992, y + 22))\n        screen.blit(font_tiny.render(active_task["status_text"], True, TEXT_MUTED), (992, y + 42))\n        y += 75\n\n    # Divider line\n    pygame.draw.line(screen, (28, 38, 56), (992, y), (card.right - 20, y))\n    y += 16\n\n    # 2. Selected Game Technical Specifications Inspector\n    screen.blit(font_body.render("TECHNICAL SPECIFICATIONS", True, HEADER_MUTED), (992, y))\n    y += 24\n\n    game = get_selected_game()\n    if game:\n        specs = [\n            ("TITLE", game["title"][:22]),\n            ("APP ID", game["app_name"][:18] + "..."),\n            ("DEVELOPER", game.get("developer", "Bethesda")[:22]),\n            ("FOOTPRINT", game.get("size", "44.5 GiB")),\n            ("VERSION", game.get("version", "1.3.1")),\n            ("STATUS", "INSTALLED" if game.get("is_installed") else "OWNED IN CLOUD"),\n            ("RENDER GPU", "GPU 0 (Tesla T4 Direct :1)"),\n            ("ENCODE GPU", "GPU 1 (Tesla T4 NVENC)"),\n            ("AUDIO SINK", "PulseAudio Native 48kHz"),\n            ("RUNNER", "Wine-GE-Proton8-26"),\n        ]\n\n        for lbl, val in specs:\n            screen.blit(font_tiny.render(lbl, True, (120, 140, 170)), (992, y))\n            val_col = LIME_GREEN if lbl == "STATUS" and val == "INSTALLED" else TEXT_WHITE\n            lbl_val = font_tiny.render(val, True, val_col)\n            screen.blit(lbl_val, (card.right - lbl_val.get_width() - 20, y))\n            y += 24\n    else:\n        screen.blit(font_small.render("Select a game to inspect specs.", True, TEXT_MUTED), (992, y))\n\n\ndef handle_click(pos: Tuple[int, int]):\n    global tab_index, selected_game_index, carousel_page, filter_mode, message\n    mx, my = pos\n\n    # 1. Stop Game Button (Header top-right)\n    if active_proc is not None and active_proc.poll() is None:\n        status_rect = pygame.Rect(WIDTH - 195, 14, 180, 40)\n        if status_rect.collidepoint((mx, my)):\n            message = stop_active_game()\n            return\n\n    # 2. Platform Tabs (y: 66-108)\n    if 66 <= my <= 108:\n        tab_w = 210\n        total_w = 2 * tab_w + 16\n        start_x = (WIDTH - total_w) // 2\n        for i in range(2):\n            t_rect = pygame.Rect(start_x + i * (tab_w + 16), 66, tab_w, 42)\n            if t_rect.collidepoint((mx, my)):\n                tab_index = i\n                selected_game_index = 0\n                carousel_page = 0\n                return\n\n    # 3. Left Panel Controls\n    if 25 <= mx <= 270:\n        # View Filter Buttons (ALL vs INSTALLED)\n        if 325 <= my <= 355:\n            if mx <= 145:\n                filter_mode = "ALL"\n            else:\n                filter_mode = "INSTALLED"\n            selected_game_index = 0\n            carousel_page = 0\n            return\n\n        # Optimization Toggles\n        if 400 <= my <= 430:\n            toggles["boost"] = not toggles["boost"]\n        elif 435 <= my <= 465:\n            toggles["direct_stream"] = not toggles["direct_stream"]\n        elif 470 <= my <= 500:\n            toggles["gamepad"] = not toggles["gamepad"]\n\n        # Sync Library Button\n        if 638 <= my <= 674:\n            message = "Syncing Epic Games Library..."\n            load_library()\n            return\n\n    # 4. Center Hero Action Buttons\n    game = get_selected_game()\n    if game and 330 <= my <= 368:\n        is_inst = game.get("is_installed", False)\n        if is_inst:\n            # Play Now Button (310 to 450)\n            if 310 <= mx <= 450:\n                message = launch_game(game)\n                return\n            # Verify Button (462 to 557)\n            if 462 <= mx <= 557:\n                message = run_legendary_task("verify", game["app_name"], game["title"])\n                return\n            # Uninstall Button (568 to 673)\n            if 568 <= mx <= 673:\n                message = run_legendary_task("uninstall", game["app_name"], game["title"])\n                return\n        else:\n            # Install Button (310 to 470)\n            if 310 <= mx <= 470:\n                message = run_legendary_task("install", game["app_name"], game["title"])\n                return\n\n    # 5. Quick-Launch Carousel Navigation & Cards\n    # Chevrons\n    if 395 <= my <= 425:\n        games = get_current_games_list()\n        total_pages = max(1, math.ceil(len(games) / PAGE_SIZE))\n        if 905 <= mx <= 930 and carousel_page > 0:\n            carousel_page -= 1\n            selected_game_index = carousel_page * PAGE_SIZE\n            return\n        if 935 <= mx <= 960 and carousel_page < total_pages - 1:\n            carousel_page += 1\n            selected_game_index = carousel_page * PAGE_SIZE\n            return\n\n    # Card Clicks\n    if 432 <= my <= 678:\n        w = 155\n        start_x = 288\n        gap = 18\n        for i in range(PAGE_SIZE):\n            cx = start_x + i * (w + gap)\n            if cx <= mx <= cx + w:\n                idx = carousel_page * PAGE_SIZE + i\n                games = get_current_games_list()\n                if idx < len(games):\n                    selected_game_index = idx\n                return\n\n    # 6. Right Panel Cancel Button\n    if active_task["proc"] is not None and 975 <= mx <= 1255:\n        if 250 <= my <= 280:\n            message = cancel_active_task()\n            return\n\n\ndef main():\n    global display, tab_index, selected_game_index, carousel_page, message\n    if not pygame:\n        print("Pygame is required to run dev_launcher.")\n        return\n\n    running = True\n    while running:\n        update_telemetry()\n        screen.fill(BG_COLOR)\n\n        draw_corner_brackets()\n        draw_header()\n        draw_tabs()\n        draw_left_card()\n        draw_center_hero()\n        draw_quick_launch()\n        draw_right_card()\n\n        for event in pygame.event.get():\n            if event.type == pygame.QUIT:\n                running = False\n            elif event.type == pygame.VIDEORESIZE:\n                display = pygame.display.set_mode((max(640, event.w), max(360, event.h)), pygame.RESIZABLE)\n            elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:\n                mx = round(event.pos[0] * WIDTH / max(1, display.get_width()))\n                my = round(event.pos[1] * HEIGHT / max(1, display.get_height()))\n                print(f"[Input] Mouse click at ({mx}, {my})", flush=True)\n                handle_click((mx, my))\n            elif event.type == pygame.KEYDOWN:\n                games = get_current_games_list()\n                total_games = len(games)\n                if event.key in (pygame.K_LEFT, pygame.K_a):\n                    if selected_game_index > 0:\n                        selected_game_index -= 1\n                        carousel_page = selected_game_index // PAGE_SIZE\n                elif event.key in (pygame.K_RIGHT, pygame.K_d):\n                    if selected_game_index < total_games - 1:\n                        selected_game_index += 1\n                        carousel_page = selected_game_index // PAGE_SIZE\n                elif event.key == pygame.K_TAB:\n                    tab_index = (tab_index + 1) % len(TABS)\n                    selected_game_index = 0\n                    carousel_page = 0\n                elif event.key in (pygame.K_RETURN, pygame.K_SPACE):\n                    game = get_selected_game()\n                    if game:\n                        if game.get("is_installed"):\n                            launch_game(game)\n                        else:\n                            run_legendary_task("install", game["app_name"], game["title"])\n                elif event.key == pygame.K_ESCAPE:\n                    stop_active_game()\n            elif event.type == pygame.JOYBUTTONDOWN:\n                if event.button == 0:  # Gamepad A\n                    game = get_selected_game()\n                    if game and game.get("is_installed"):\n                        launch_game(game)\n                elif event.button == 1:  # Gamepad B\n                    stop_active_game()\n\n        output = screen if display.get_size() == (WIDTH, HEIGHT) else pygame.transform.smoothscale(screen, display.get_size())\n        display.blit(output, (0, 0))\n        pygame.display.flip()\n\n        clock.tick(30)\n\n    pygame.quit()\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(BENCH / 'embedded_assets.py').write_text('"""Embedded UI art assets for 7BZ Cloud Gaming dashboard."""\nimport base64\nimport io\nimport pygame\n\nASSETS_B64 = {\n    "7bz_logo": "...",\n    "card_apex": "...",\n    "card_elden_ring": "...",\n    "card_forza_5": "...",\n    "card_hades": "...",\n    "hero_cyberpunk": "...",\n}\n\n_cache = {}\n\ndef get_surface(name: str):\n    if name in _cache:\n        return _cache[name]\n    if name in ASSETS_B64:\n        raw = base64.b64decode(ASSETS_B64[name])\n        surf = pygame.image.load(io.BytesIO(raw)).convert_alpha()\n        _cache[name] = surf\n        return surf\n    return None\n', encoding='utf-8')
(BENCH / 'game_catalog.py').write_text('"""Game Catalog and Epic Games Storefront Integration for 7BZ Cloud Gaming.\n\nIntegrates directly with Legendary and Heroic metadata without separate windows:\n- Queries official Epic user account and cloud library (117+ owned titles).\n- Queries installed games (Wolfenstein: The New Order, etc.).\n- Asynchronously caches and renders official Epic Games CDN artwork (DieselGameBox & DieselGameBoxTall).\n- Discovers local standalone builds and development scripts.\n"""\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport threading\nimport time\nfrom typing import Any, Dict, List, Optional, Tuple\nimport urllib.request\n\ntry:\n    import pygame\nexcept ImportError:\n    pygame = None\n\nHOME = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\nBUILDS = Path(os.environ.get("GAME_BUILDS_DIR", "/tmp/game-builds"))\nCACHE_DIR = HOME / ".cache" / "7bz-dev-cloud-game" / "artwork"\nCACHE_DIR.mkdir(parents=True, exist_ok=True)\n\n# In-memory surface cache: (cache_key, target_w, target_h) -> pygame.Surface\n_surface_cache: Dict[Tuple[str, int, int], Any] = {}\n_download_queue = set()\n_executor = ThreadPoolExecutor(max_workers=6, thread_name_prefix="art_downloader")\n\n\ndef _read_json(path: Path) -> Any:\n    try:\n        return json.loads(path.read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        return None\n\n\ndef get_epic_user() -> Dict[str, Any]:\n    """Return Epic account username and login details."""\n    user_file = HOME / ".config" / "legendary" / "user.json"\n    data = _read_json(user_file)\n    if isinstance(data, dict) and data.get("displayName"):\n        return {\n            "displayName": data["displayName"],\n            "accountId": data.get("account_id", ""),\n            "status": "Connected"\n        }\n    return {\n        "displayName": "Gamerhbaza",\n        "accountId": "EpicCloudUser",\n        "status": "Connected"\n    }\n\n\ndef _legendary_metadata(app_name: str) -> Dict[str, Any]:\n    candidates = (\n        HOME / ".config" / "legendary" / "metadata" / f"{app_name}.json",\n        HOME / ".config" / "heroic" / "legendaryConfig" / "legendary" / "metadata" / f"{app_name}.json",\n    )\n    for path in candidates:\n        data = _read_json(path)\n        if isinstance(data, dict):\n            return data\n    return {}\n\n\ndef _extract_images(metadata_dict: Dict[str, Any]) -> Tuple[str, str]:\n    """Return (wide_url, tall_url) from Epic metadata."""\n    meta = metadata_dict.get("metadata") or metadata_dict\n    images = meta.get("keyImages") or meta.get("key_images") or []\n    if not isinstance(images, list):\n        return "", ""\n\n    wide_url = ""\n    tall_url = ""\n\n    # Priority for wide (Hero banner)\n    for pref in ("DieselGameBox", "DieselGameBoxWide", "OfferImageWide", "Featured", "Thumbnail"):\n        for img in images:\n            if isinstance(img, dict) and img.get("type") == pref and img.get("url"):\n                wide_url = img["url"]\n                break\n        if wide_url:\n            break\n\n    # Priority for tall (Poster cards)\n    for pref in ("DieselGameBoxTall", "OfferImageTall", "DieselStoreFrontTall"):\n        for img in images:\n            if isinstance(img, dict) and img.get("type") == pref and img.get("url"):\n                tall_url = img["url"]\n                break\n        if tall_url:\n            break\n\n    if not wide_url and tall_url:\n        wide_url = tall_url\n    if not tall_url and wide_url:\n        tall_url = wide_url\n\n    return wide_url, tall_url\n\n\ndef cleanup_stale_installed_record(app_name: str):\n    """Remove a stale or uninstalled game record from both Legendary and Heroic installed.json files."""\n    candidates = (\n        HOME / ".config" / "heroic" / "legendaryConfig" / "legendary" / "installed.json",\n        HOME / ".config" / "legendary" / "installed.json",\n    )\n    for path in candidates:\n        if path.is_file():\n            try:\n                data = json.loads(path.read_text(encoding="utf-8"))\n                if isinstance(data, dict) and app_name in data:\n                    del data[app_name]\n                    path.write_text(json.dumps(data, indent=2), encoding="utf-8")\n            except Exception:\n                pass\n\n\ndef legendary_installed_games() -> List[Dict[str, Any]]:\n    """Return all verified installed Epic games that actually exist on disk."""\n    candidates = (\n        HOME / ".config" / "heroic" / "legendaryConfig" / "legendary" / "installed.json",\n        HOME / ".config" / "legendary" / "installed.json",\n    )\n    installed_dict: Dict[str, Any] = {}\n    for path in candidates:\n        raw = _read_json(path)\n        if isinstance(raw, dict):\n            installed_dict.update(raw)\n\n    games: List[Dict[str, Any]] = []\n    for app_name, record in list(installed_dict.items()):\n        if not isinstance(record, dict):\n            continue\n\n        raw_ip = record.get("install_path") or f"/home/gamer/Games/{app_name}"\n        install_path = Path(str(raw_ip))\n        raw_exe = record.get("executable") or ""\n\n        # Verification check: The game directory MUST exist and contain game files\n        if not install_path.is_dir():\n            cleanup_stale_installed_record(str(app_name))\n            continue\n\n        try:\n            has_files = any(install_path.iterdir())\n        except OSError:\n            has_files = False\n\n        if not has_files:\n            cleanup_stale_installed_record(str(app_name))\n            continue\n\n        # If an executable is specified or expected, ensure at least one .exe exists\n        if raw_exe:\n            exe_target = install_path / raw_exe\n            if not exe_target.is_file():\n                # Check recursive search\n                found = list(install_path.rglob("*.exe"))\n                if not found:\n                    cleanup_stale_installed_record(str(app_name))\n                    continue\n                else:\n                    raw_exe = found[0].name\n        else:\n            found = list(install_path.rglob("*.exe"))\n            if not found:\n                cleanup_stale_installed_record(str(app_name))\n                continue\n            raw_exe = found[0].name\n\n        metadata = _legendary_metadata(str(app_name))\n        meta_info = metadata.get("metadata", {})\n        title = str(record.get("title") or metadata.get("app_title") or app_name)\n        wide_url, tall_url = _extract_images(metadata)\n\n        size_bytes = record.get("install_size") or 0\n        size_str = f"{size_bytes / (1024**3):.1f} GiB" if size_bytes > 0 else "Installed"\n\n        games.append({\n            "id": f"epic:{app_name}",\n            "app_name": str(app_name),\n            "title": title,\n            "store": "Epic Games",\n            "is_installed": True,\n            "status": "Installed",\n            "install_path": str(install_path),\n            "executable": str(raw_exe),\n            "size": size_str,\n            "version": str(record.get("version") or "1.0"),\n            "developer": meta_info.get("developer") or "Epic Games",\n            "publisher": meta_info.get("publisher") or "Epic Games",\n            "wide_art": wide_url,\n            "tall_art": tall_url,\n            "action": "legendary",\n        })\n    return games\n\n\ndef epic_library_games() -> List[Dict[str, Any]]:\n    """Return the complete cloud library of owned Epic games."""\n    installed = {g["app_name"]: g for g in legendary_installed_games()}\n    meta_dir = HOME / ".config" / "legendary" / "metadata"\n    heroic_meta_dir = HOME / ".config" / "heroic" / "legendaryConfig" / "legendary" / "metadata"\n\n    seen_apps = set()\n    library: List[Dict[str, Any]] = []\n\n    # Put installed games first\n    for g in installed.values():\n        library.append(g)\n        seen_apps.add(g["app_name"])\n\n    # Scan metadata files for owned games\n    search_dirs = [meta_dir, heroic_meta_dir]\n    for mdir in search_dirs:\n        if not mdir.is_dir():\n            continue\n        for p in sorted(mdir.glob("*.json")):\n            app_name = p.stem\n            if app_name in seen_apps:\n                continue\n            seen_apps.add(app_name)\n\n            data = _read_json(p)\n            if not isinstance(data, dict):\n                continue\n\n            title = data.get("app_title", "")\n            if not title or title.startswith("[REDACTED]") or title.lower().startswith("ue4"):\n                continue\n\n            meta_info = data.get("metadata", {})\n            wide_url, tall_url = _extract_images(data)\n\n            # Skip items without any artwork\n            if not wide_url and not tall_url:\n                continue\n\n            library.append({\n                "id": f"epic:{app_name}",\n                "app_name": app_name,\n                "title": title,\n                "store": "Epic Games",\n                "is_installed": False,\n                "status": "Ready to Install",\n                "install_path": f"/home/gamer/Games/{title.replace(\' \', \'\')}",\n                "executable": "Auto",\n                "size": "Available in Cloud",\n                "version": "Latest",\n                "developer": meta_info.get("developer") or "Epic Games",\n                "publisher": meta_info.get("publisher") or "Epic Games",\n                "wide_art": wide_url,\n                "tall_art": tall_url,\n                "action": "legendary",\n            })\n\n    # Sort remaining non-installed games alphabetically\n    installed_count = len(installed)\n    library[installed_count:] = sorted(library[installed_count:], key=lambda x: x["title"].lower())\n    return library\n\n\ndef local_games() -> List[Dict[str, Any]]:\n    """Return local custom builds, development binaries, and bench tools."""\n    items: List[Dict[str, Any]] = []\n\n    # 1. Custom build directory\n    if BUILDS.is_dir():\n        for path in sorted(BUILDS.iterdir(), key=lambda item: item.name.lower()):\n            if path.is_file() and (os.access(path, os.X_OK) or path.suffix.lower() in (".sh", ".appimage", ".x86_64", ".bin", ".exe")):\n                items.append({\n                    "id": f"build:{path.name}",\n                    "app_name": path.name,\n                    "title": path.stem.replace("_", " ").title(),\n                    "store": "Local Build",\n                    "is_installed": True,\n                    "status": "Local Binary",\n                    "install_path": str(path),\n                    "executable": path.name,\n                    "size": f"{path.stat().st_size / (1024**2):.1f} MB",\n                    "version": "1.0-dev",\n                    "developer": "Local Development",\n                    "publisher": "7BZ Bench",\n                    "wide_art": "",\n                    "tall_art": "",\n                    "action": "build",\n                })\n\n    # 2. Check /home/gamer/Games for installed standalone games or Wine directories\n    games_dir = HOME / "Games"\n    if games_dir.is_dir():\n        for sub in sorted(games_dir.iterdir()):\n            if sub.is_dir() and not sub.name.startswith("."):\n                exes = list(sub.glob("*.exe")) + list(sub.glob("*.sh"))\n                exe_name = exes[0].name if exes else "Main"\n                items.append({\n                    "id": f"local:{sub.name}",\n                    "app_name": sub.name,\n                    "title": sub.name.replace("_", " ").title(),\n                    "store": "Local Game",\n                    "is_installed": True,\n                    "status": "Standalone Directory",\n                    "install_path": str(sub),\n                    "executable": exe_name,\n                    "size": "Local Disk",\n                    "version": "Direct",\n                    "developer": "Custom / Wine",\n                    "publisher": "Local User",\n                    "wide_art": "",\n                    "tall_art": "",\n                    "action": "build",\n                })\n\n    # 3. Built-in Bench Utilities for testing and development\n    bench_dir = Path("/tmp/kaggle-game-dev-bench")\n    diag_tools = [\n        ("GPU 3D Diagnostic", "test_gpu_runtime.py", "Tesla T4 3D / DXVK / Vulkan probe"),\n        ("idTech 5 / OpenGL Probe", "test_idtech5.py", "OpenGL 4.5 Compatibility profile verification"),\n        ("Direct Stream Inspector", "run_inspection.py", "Selkies low-latency WebRTC metrics test"),\n    ]\n    for title, script_name, desc in diag_tools:\n        script_path = bench_dir / script_name\n        items.append({\n            "id": f"bench:{script_name}",\n            "app_name": script_name,\n            "title": title,\n            "store": "Bench Tool",\n            "is_installed": True,\n            "status": "Ready",\n            "install_path": str(script_path),\n            "executable": script_name,\n            "size": "System Script",\n            "version": "2.0-bench",\n            "developer": "7BZ Cloud Gaming",\n            "publisher": desc,\n            "wide_art": "",\n            "tall_art": "",\n            "action": "build",\n        })\n\n    return items\n\n\ndef _download_worker(app_name: str, url: str, kind: str):\n    """Download image in background thread and cache to disk."""\n    if not url:\n        return\n    ext = ".png" if ".png" in url.lower() else ".jpg"\n    dst = CACHE_DIR / f"{app_name}_{kind}{ext}"\n    if dst.is_file() and dst.stat().st_size > 1000:\n        return\n    try:\n        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})\n        with urllib.request.urlopen(req, timeout=12) as resp, dst.open("wb") as out:\n            out.write(resp.read())\n    except Exception:\n        pass\n    finally:\n        _download_queue.discard((app_name, kind))\n        # Invalidate in-memory cache so next frame immediately loads official artwork\n        for k in list(_surface_cache.keys()):\n            if k[0].startswith(f"{app_name}_{kind}"):\n                _surface_cache.pop(k, None)\n\n\ndef _draw_fallback_card(title: str, store: str, target_w: int, target_h: int) -> Any:\n    """Procedurally render a sleek cybernetic fallback poster."""\n    if not pygame:\n        return None\n    surf = pygame.Surface((target_w, target_h), pygame.SRCALPHA)\n    pygame.draw.rect(surf, (16, 22, 34), (0, 0, target_w, target_h), border_radius=10)\n    pygame.draw.rect(surf, (28, 40, 60), (0, 0, target_w, target_h), width=1, border_radius=10)\n\n    # Accent top bar\n    pygame.draw.rect(surf, (0, 240, 255), (10, 10, target_w - 20, 3), border_radius=2)\n\n    try:\n        font_big = pygame.font.SysFont("Liberation Sans", 24, bold=True)\n        font_txt = pygame.font.SysFont("Liberation Sans", 11, bold=True)\n        font_sub = pygame.font.SysFont("Liberation Sans", 9)\n\n        initials = "".join(w[0] for w in title.split() if w)[:3].upper()\n        lbl_init = font_big.render(initials, True, (0, 240, 255))\n        surf.blit(lbl_init, (target_w // 2 - lbl_init.get_width() // 2, target_h // 2 - 30))\n\n        clean_title = title if len(title) <= 18 else title[:16] + "..."\n        lbl_title = font_txt.render(clean_title, True, (245, 248, 255))\n        surf.blit(lbl_title, (target_w // 2 - lbl_title.get_width() // 2, target_h // 2 + 15))\n\n        lbl_store = font_sub.render(store.upper(), True, (130, 145, 170))\n        surf.blit(lbl_store, (target_w // 2 - lbl_store.get_width() // 2, target_h // 2 + 32))\n    except Exception:\n        pass\n\n    return surf\n\n\ndef get_artwork(app_name: str, url: str, kind: str, target_w: int, target_h: int, title: str = "", store: str = "Epic Games") -> Any:\n    """Return a cached Pygame surface scaled to target dimensions."""\n    if not pygame:\n        return None\n\n    cache_key = (f"{app_name}_{kind}", target_w, target_h)\n    if cache_key in _surface_cache:\n        return _surface_cache[cache_key]\n\n    # Check disk cache\n    for ext in (".jpg", ".png", ".jpeg"):\n        file_path = CACHE_DIR / f"{app_name}_{kind}{ext}"\n        if file_path.is_file() and file_path.stat().st_size > 1000:\n            try:\n                raw_surf = pygame.image.load(str(file_path))\n                scaled = pygame.transform.smoothscale(raw_surf, (target_w, target_h))\n                _surface_cache[cache_key] = scaled\n                return scaled\n            except Exception:\n                pass\n\n    # Not cached on disk yet: queue background download\n    if url and (app_name, kind) not in _download_queue:\n        _download_queue.add((app_name, kind))\n        _executor.submit(_download_worker, app_name, url, kind)\n\n    # Return fallback (do not cache in _surface_cache so real image takes over when ready)\n    return _draw_fallback_card(title or app_name, store, target_w, target_h)\n\n\ndef pre_cache_top_games():\n    """Background task to pre-download artwork for top library titles."""\n    games = epic_library_games()\n    for g in games[:24]:\n        app = g["app_name"]\n        if g.get("tall_art") and (app, "tall") not in _download_queue:\n            _download_queue.add((app, "tall"))\n            _executor.submit(_download_worker, app, g["tall_art"], "tall")\n        if g.get("wide_art") and (app, "wide") not in _download_queue:\n            _download_queue.add((app, "wide"))\n            _executor.submit(_download_worker, app, g["wide_art"], "wide")\n\n\ndef get_catalog() -> Dict[str, List[Dict[str, Any]]]:\n    """Helper returning complete structured catalog for dev_launcher."""\n    return {\n        "epic": epic_library_games(),\n        "local": local_games(),\n    }\n\n\nthreading.Thread(target=pre_cache_top_games, daemon=True).start()\n', encoding='utf-8')
(BENCH / 'session_control.py').write_text('"""Local control service for the 7BZ streaming HUD and dashboard."""\nimport asyncio\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport time\nfrom typing import Any, Dict, List\n\nfrom aiohttp import web\n\nfrom game_catalog import discover_games\n\n\nBENCH = Path("/tmp/kaggle-game-dev-bench")\nSTATE = BENCH / "active-game.json"\nPORT = int(os.environ.get("SEVEN_BZ_CONTROL_PORT", "8766"))\n\n\ndef get_gpu_metrics() -> List[Dict[str, Any]]:\n    """Query live metrics for all available NVIDIA GPUs."""\n    try:\n        res = subprocess.run(\n            ["nvidia-smi", "--query-gpu=index,name,utilization.gpu,memory.used,memory.total",\n             "--format=csv,noheader,nounits"],\n            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, timeout=2\n        )\n        if res.returncode != 0:\n            return []\n        gpus = []\n        for line in res.stdout.strip().splitlines():\n            parts = [p.strip() for p in line.split(",")]\n            if len(parts) >= 5:\n                idx = int(parts[0])\n                role = "Render (Game)" if idx == 0 else "Encode (Stream)"\n                gpus.append({\n                    "index": idx,\n                    "name": parts[1],\n                    "role": role,\n                    "utilization": int(parts[2]),\n                    "memory_used": int(parts[3]),\n                    "memory_total": int(parts[4]),\n                })\n        return gpus\n    except Exception:\n        return []\n\n\ndef write_active_game(pid: int, pgid: int, title: str, source: str) -> None:\n    """Record a dashboard-launched process after validating its identity."""\n    STATE.write_text(json.dumps({"pid": pid, "pgid": pgid, "title": title,\n                                 "source": source, "started": time.time()}), encoding="utf-8")\n\n\ndef clear_active_game(pid: int = 0) -> None:\n    """Remove the active record when it still belongs to the given process."""\n    try:\n        current = json.loads(STATE.read_text(encoding="utf-8"))\n        if not pid or int(current.get("pid", 0)) == pid:\n            STATE.unlink(missing_ok=True)\n    except (OSError, ValueError, TypeError):\n        STATE.unlink(missing_ok=True)\n\n\ndef _state() -> Dict[str, Any]:\n    state_data: Dict[str, Any] = {"running": False, "title": "No active game"}\n    try:\n        data = json.loads(STATE.read_text(encoding="utf-8"))\n        pid = int(data.get("pid", 0))\n        if pid > 1 and Path(f"/proc/{pid}").exists():\n            data["running"] = True\n            state_data = data\n    except (OSError, ValueError, TypeError):\n        pass\n    state_data["gpus"] = get_gpu_metrics()\n    return state_data\n\n\ndef _owned_process(pid: int) -> bool:\n    """Confirm a recorded process exists and belongs to this service\'s user."""\n    try:\n        return Path(f"/proc/{pid}").stat().st_uid == os.getuid()\n    except OSError:\n        return False\n\n\nasync def _end_group(pgid: int, sig: signal.Signals) -> bool:\n    try:\n        os.killpg(pgid, sig)\n        return True\n    except (ProcessLookupError, PermissionError, OSError):\n        return False\n\n\nasync def reset_session() -> Dict[str, Any]:\n    """Stop the tracked game group, Wine servers, and restore the dashboard."""\n    state = _state()\n    stopped = False\n    pid = int(state.get("pid", 0) or 0)\n    pgid = int(state.get("pgid", 0) or 0)\n    if pid > 1 and pgid > 1 and _owned_process(pid):\n        stopped = await _end_group(pgid, signal.SIGTERM)\n        for _ in range(12):\n            if not Path(f"/proc/{pid}").exists():\n                break\n            await asyncio.sleep(0.25)\n        if Path(f"/proc/{pid}").exists():\n            await _end_group(pgid, signal.SIGKILL)\n    clear_active_game(pid)\n    subprocess.run(["wineserver", "-k"], check=False, stdout=subprocess.DEVNULL,\n                   stderr=subprocess.DEVNULL, timeout=5)\n    subprocess.run(["wmctrl", "-a", "7BZ Dev Cloud Game"], check=False,\n                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=3)\n    return {"ok": True, "stopped": stopped, "message": "Game stopped. Dashboard restored."}\n\n\nasync def health(_request: web.Request) -> web.Response:\n    return web.json_response({"ok": True, "service": "7bz-control"})\n\n\nasync def status(_request: web.Request) -> web.Response:\n    return web.json_response(_state())\n\n\nasync def catalog(_request: web.Request) -> web.Response:\n    return web.json_response({"games": discover_games()})\n\n\nINPUT_MODE_FILE = BENCH / "input-mode.json"\n\n\ndef _get_input_mode() -> str:\n    try:\n        if INPUT_MODE_FILE.is_file():\n            data = json.loads(INPUT_MODE_FILE.read_text(encoding="utf-8"))\n            return data.get("mode", "mouse_keyboard")\n    except Exception:\n        pass\n    return "mouse_keyboard"\n\n\ndef _set_input_mode(mode: str) -> str:\n    valid = "gamepad" if mode == "gamepad" else "mouse_keyboard"\n    try:\n        INPUT_MODE_FILE.write_text(json.dumps({"mode": valid, "updated": time.time()}), encoding="utf-8")\n    except Exception:\n        pass\n    return valid\n\n\nasync def input_mode_get(_request: web.Request) -> web.Response:\n    return web.json_response({"mode": _get_input_mode()})\n\n\nasync def input_mode_post(request: web.Request) -> web.Response:\n    if request.headers.get("X-7BZ-Action") != "hud":\n        raise web.HTTPForbidden(text="Missing HUD action header")\n    try:\n        body = await request.json()\n        target_mode = body.get("mode", "")\n    except Exception:\n        target_mode = ""\n    if not target_mode:\n        current = _get_input_mode()\n        target_mode = "gamepad" if current == "mouse_keyboard" else "mouse_keyboard"\n    new_mode = _set_input_mode(target_mode)\n    return web.json_response({"ok": True, "mode": new_mode, "message": f"Input mode set to {new_mode}"})\n\n\nasync def reset(request: web.Request) -> web.Response:\n    if request.headers.get("X-7BZ-Action") != "hud":\n        raise web.HTTPForbidden(text="Missing HUD action header")\n    return web.json_response(await reset_session())\n\n\nasync def gpus_handler(_request: web.Request) -> web.Response:\n    return web.json_response({"gpus": get_gpu_metrics()})\n\n\ndef main() -> None:\n    """Serve narrow session actions on the loopback-only control port."""\n    app = web.Application(client_max_size=16 * 1024)\n    app.add_routes([web.get("/health", health), web.get("/status", status),\n                    web.get("/catalog", catalog), web.get("/gpus", gpus_handler),\n                    web.post("/reset", reset),\n                    web.get("/input-mode", input_mode_get), web.post("/input-mode", input_mode_post)])\n    web.run_app(app, host="127.0.0.1", port=PORT, access_log=None)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(BENCH / 'install_web_client.py').write_text('"""Create a branded Selkies web root with the 7BZ streaming HUD."""\nfrom importlib import resources\nfrom pathlib import Path\nimport shutil\n\n\nBENCH = Path("/tmp/kaggle-game-dev-bench")\nWEB_ROOT = BENCH / "web-root"\nASSETS = BENCH / "web_client"\nMARKER = "7BZ Dev Cloud Game HUD"\n\n\ndef _source_root() -> Path:\n    import selkies\n    direct = Path(selkies.__file__).resolve().parent / "selkies_web"\n    if direct.is_dir():\n        return direct\n    package = resources.files("selkies.selkies_web")\n    with resources.as_file(package) as extracted:\n        if extracted.is_dir():\n            temporary = BENCH / "selkies-web-source"\n            shutil.copytree(extracted, temporary, dirs_exist_ok=True)\n            return temporary\n    raise RuntimeError("Selkies web assets were not found")\n\n\ndef install() -> Path:\n    """Copy the installed client, add branded assets, and return its web root."""\n    import re, time\n    v = int(time.time())\n    source = _source_root()\n    shutil.copytree(source, WEB_ROOT, dirs_exist_ok=True)\n    shutil.copy2(ASSETS / "hud.css", WEB_ROOT / "7bz-hud.css")\n    shutil.copy2(ASSETS / "hud.js", WEB_ROOT / "7bz-hud.js")\n    index = WEB_ROOT / "index.html"\n    text = index.read_text(encoding="utf-8")\n    if MARKER not in text:\n        head = f\'<!-- {MARKER} -->\\n<link rel="stylesheet" href="7bz-hud.css?v={v}">\\n\'\n        script = f\'<script defer src="7bz-hud.js?v={v}"></script>\\n\'\n        text = text.replace("</head>", head + "</head>")\n        text = text.replace("</body>", script + "</body>")\n    else:\n        text = re.sub(r\'7bz-hud\\.css(\\?v=\\d+)?\', f\'7bz-hud.css?v={v}\', text)\n        text = re.sub(r\'7bz-hud\\.js(\\?v=\\d+)?\', f\'7bz-hud.js?v={v}\', text)\n    text = text.replace("<title>Selkies</title>", "<title>7BZ Dev Cloud Game</title>")\n    index.write_text(text, encoding="utf-8")\n    return WEB_ROOT\n\n\nif __name__ == "__main__":\n    print(install())\n', encoding='utf-8')
(BENCH / 'web_client' / 'hud.css').write_text(':root {\n  --seven-bg: rgba(7, 12, 25, .72);\n  --seven-panel: rgba(17, 25, 47, .76);\n  --seven-line: rgba(137, 183, 255, .18);\n  --seven-text: #f7f9ff;\n  --seven-muted: #9aa8c4;\n  --seven-cyan: #48d7ff;\n  --seven-violet: #8f7cff;\n  --seven-danger: #ff526b;\n  --seven-good: #4ee6a8;\n}\n\n#seven-bz-hud, #seven-bz-hud * { box-sizing: border-box; }\n\n#seven-bz-hud {\n  position: fixed;\n  z-index: 2147483000;\n  top: 14px;\n  left: 50%;\n  width: min(1080px, calc(100vw - 28px));\n  transform: translateX(-50%);\n  color: var(--seven-text);\n  font-family: Inter, ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;\n  pointer-events: none;\n}\n\n.seven-shell {\n  pointer-events: auto;\n  overflow: hidden;\n  border: 1px solid var(--seven-line);\n  border-radius: 18px;\n  background: linear-gradient(120deg, rgba(12, 20, 40, .88), var(--seven-bg));\n  box-shadow: 0 18px 55px rgba(0, 0, 0, .45), inset 0 1px rgba(255,255,255,.08);\n  backdrop-filter: blur(22px) saturate(150%);\n  -webkit-backdrop-filter: blur(22px) saturate(150%);\n}\n\n.seven-topbar { display: flex; align-items: center; min-height: 48px; padding: 8px 12px 8px 16px; gap: 14px; }\n.seven-brand { display: flex; align-items: center; gap: 10px; min-width: 215px; font-weight: 760; letter-spacing: .02em; }\n.seven-mark { width: 24px; height: 24px; border-radius: 8px; background: linear-gradient(140deg, var(--seven-cyan), var(--seven-violet)); box-shadow: 0 0 22px rgba(72,215,255,.32); }\n.seven-live { display: inline-flex; align-items: center; gap: 6px; color: var(--seven-good); font-size: 11px; font-weight: 700; letter-spacing: .12em; }\n.seven-live::before { content: ""; width: 7px; height: 7px; border-radius: 50%; background: currentColor; box-shadow: 0 0 12px currentColor; }\n.seven-quick { display: flex; flex: 1; align-items: center; justify-content: center; gap: 7px; }\n.seven-chip { min-width: 86px; padding: 6px 9px; border-radius: 10px; background: rgba(255,255,255,.055); border: 1px solid rgba(255,255,255,.055); text-align: center; }\n.seven-chip b { display: block; font-size: 13px; line-height: 14px; font-weight: 720; }\n.seven-chip span { color: var(--seven-muted); font-size: 9px; text-transform: uppercase; letter-spacing: .1em; }\n\n.seven-button { appearance: none; border: 1px solid rgba(255,255,255,.1); border-radius: 10px; padding: 8px 11px; background: rgba(255,255,255,.07); color: var(--seven-text); font: inherit; font-size: 12px; font-weight: 650; cursor: pointer; transition: .16s ease; white-space: nowrap; }\n.seven-button:hover, .seven-button:focus-visible { transform: translateY(-1px); border-color: rgba(72,215,255,.55); background: rgba(72,215,255,.14); outline: none; }\n.seven-button.is-active { color: #07111f; background: linear-gradient(120deg, var(--seven-cyan), #75f1d0); border-color: transparent; }\n.seven-button.seven-danger { color: #ffeef1; border-color: rgba(255,82,107,.38); background: rgba(255,82,107,.15); }\n.seven-button.seven-danger:hover { background: rgba(255,82,107,.3); border-color: var(--seven-danger); }\n.seven-icon-button { width: 34px; height: 34px; padding: 0; font-size: 17px; }\n\n.seven-details { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; max-height: 340px; padding: 0 12px 12px; transition: max-height .22s ease, padding .22s ease, opacity .18s ease; }\n.seven-details.is-collapsed { max-height: 0; padding-top: 0; padding-bottom: 0; opacity: 0; pointer-events: none; }\n.seven-card { border: 1px solid rgba(255,255,255,.07); border-radius: 14px; background: var(--seven-panel); padding: 12px; }\n.seven-card-title { display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; color: var(--seven-muted); font-size: 10px; font-weight: 740; letter-spacing: .12em; text-transform: uppercase; }\n.seven-metrics { display: grid; grid-template-columns: repeat(4, 1fr); gap: 7px; }\n.seven-metric { padding: 9px; border-radius: 11px; background: rgba(255,255,255,.04); }\n.seven-metric strong { display: block; font-size: 16px; }\n.seven-metric small { color: var(--seven-muted); font-size: 9px; text-transform: uppercase; letter-spacing: .08em; }\n.seven-actions { display: flex; flex-wrap: wrap; gap: 7px; }\n.seven-optimize { display: flex; align-items: center; justify-content: space-between; gap: 12px; margin-bottom: 10px; }\n.seven-optimize-copy b { display: block; font-size: 14px; }\n.seven-optimize-copy span { color: var(--seven-muted); font-size: 11px; }\n.seven-notice { grid-column: 1 / -1; display: flex; align-items: center; justify-content: space-between; gap: 12px; min-height: 34px; padding: 7px 12px; border: 1px solid rgba(143,124,255,.2); border-radius: 11px; background: rgba(143,124,255,.09); color: #dcd7ff; font-size: 11px; }\n.seven-toast { position: absolute; top: calc(100% + 8px); right: 8px; max-width: 380px; padding: 10px 14px; border-radius: 11px; background: rgba(7,12,25,.94); border: 1px solid var(--seven-line); box-shadow: 0 12px 30px rgba(0,0,0,.36); font-size: 12px; opacity: 0; transform: translateY(-5px); transition: .18s ease; pointer-events: none; }\n.seven-toast.is-visible { opacity: 1; transform: none; }\n\n@media (max-width: 850px) {\n  .seven-brand { min-width: auto; }\n  .seven-brand-name { display: none; }\n  .seven-quick .seven-chip:nth-child(n+4) { display: none; }\n  .seven-details { grid-template-columns: 1fr; }\n  .seven-notice { grid-column: 1; }\n}\n\n@media (max-width: 560px) {\n  #seven-bz-hud { top: 6px; width: calc(100vw - 12px); }\n  .seven-topbar { gap: 6px; padding: 6px 8px; }\n  .seven-live { display: none; }\n  .seven-quick { justify-content: flex-start; overflow: hidden; }\n  .seven-chip { min-width: 67px; padding: 5px; }\n  .seven-quick .seven-chip:nth-child(n+3) { display: none; }\n  .seven-metrics { grid-template-columns: repeat(2, 1fr); }\n}\n', encoding='utf-8')
(BENCH / 'web_client' / 'hud.js').write_text('/** 7BZ Dev Cloud Game streaming HUD. */\n(() => {\n  "use strict";\n  if (window.__sevenBzHud) return;\n  window.__sevenBzHud = true;\n\n  const post = (message) => window.postMessage(message, window.location.origin);\n  const value = (input, fallback = 0) => Number.isFinite(Number(input)) ? Number(input) : fallback;\n  const format = (number, digits = 0) => value(number).toFixed(digits);\n  const state = {\n    expanded: false, qualityInitialized: false,\n    rtts: [], audio: true, pointer: false, esc: false, toastTimer: 0,\n    hasGamepad: false, inputMode: "mouse_keyboard",\n  };\n\n  const hud = document.createElement("section");\n  hud.id = "seven-bz-hud";\n  hud.setAttribute("aria-label", "7BZ streaming controls");\n  hud.innerHTML = `\n    <div class="seven-shell">\n      <div class="seven-topbar">\n        <div class="seven-brand"><i class="seven-mark"></i><span class="seven-brand-name">7BZ Dev Cloud Game</span><span class="seven-live">LIVE</span></div>\n        <div class="seven-quick">\n          <div class="seven-chip"><b data-stat="fps">--</b><span>Stream FPS</span></div>\n          <div class="seven-chip"><b data-stat="rtt">--</b><span>Stream RTT</span></div>\n          <div class="seven-chip"><b data-stat="rate">--</b><span>Video rate</span></div>\n          <div class="seven-chip"><b data-stat="pad">None</b><span>Gamepad</span></div>\n          <div class="seven-chip"><b data-stat="gpu">--</b><span>GPU (G0|G1)</span></div>\n          <div class="seven-chip"><b data-stat="cpu">--</b><span>CPU load</span></div>\n        </div>\n        <button class="seven-button seven-icon-button" data-action="expand" aria-label="Open controls" title="Open controls">⌄</button>\n      </div>\n      <div class="seven-details is-collapsed">\n        <div class="seven-card">\n          <div class="seven-card-title"><span>Live performance</span><span data-stat="transport">WebSocket</span></div>\n          <div class="seven-metrics">\n            <div class="seven-metric"><strong data-stat="gpu0_metric">--</strong><small>🎮 GPU 0 Render</small></div>\n            <div class="seven-metric"><strong data-stat="gpu1_metric">--</strong><small>📡 GPU 1 Stream</small></div>\n            <div class="seven-metric"><strong data-stat="jitter">--</strong><small>RTT jitter</small></div>\n            <div class="seven-metric"><strong data-stat="loss">--</strong><small>Frame loss</small></div>\n            <div class="seven-metric"><strong data-stat="packet">N/A</strong><small>Packet loss</small></div>\n          </div>\n        </div>\n        <div class="seven-card">\n          <div class="seven-optimize">\n            <div class="seven-optimize-copy"><b>Automatic stream quality</b><span data-stat="profile">720p · 60 FPS · adaptive bitrate</span></div>\n            <span class="seven-live">AUTO</span>\n          </div>\n          <div class="seven-actions">\n            <button class="seven-button" data-action="fullscreen">Fullscreen</button>\n            <button class="seven-button" data-action="pointer">Gaming Mode (Lock Mouse)</button>\n            <button class="seven-button" data-action="mode" data-stat="modename">🖱️ Mode: M+K</button>\n            <button class="seven-button" data-action="esc">Esc Lock</button>\n            <button class="seven-button" data-action="audio">Audio Unmute</button>\n            <button class="seven-button seven-danger" data-action="reset">Force Kill / Exit Frozen Game</button>\n          </div>\n        </div>\n        <div class="seven-notice"><span>Input tip: In FPS games (Blood West), use Gaming Mode for unconstrained 360° mouse view. If using Gamepad, press any button on the controller to connect.</span><span>Ctrl+Shift+M: HUD</span></div>\n      </div>\n    </div>\n    <div class="seven-toast" role="status"></div>`;\n  document.body.appendChild(hud);\n\n  const one = (selector) => hud.querySelector(selector);\n  const stat = (name, text) => { const target = one(`[data-stat="${name}"]`); if (target) target.textContent = text; };\n  const button = (name) => one(`[data-action="${name}"]`);\n  const toast = (message) => {\n    const target = one(".seven-toast");\n    target.textContent = message;\n    target.classList.add("is-visible");\n    clearTimeout(state.toastTimer);\n    state.toastTimer = setTimeout(() => target.classList.remove("is-visible"), 3200);\n  };\n\n  function controlBase() {\n    const replaced = window.location.pathname.replace(/\\/proxy\\/(?:8080|8081)(?:\\/.*)?$/, "/proxy/8766");\n    return replaced === window.location.pathname ? "/proxy/8766" : replaced;\n  }\n\n  function checkGamepads() {\n    try {\n      const pads = navigator.getGamepads ? Array.from(navigator.getGamepads()).filter(Boolean) : [];\n      if (pads.length > 0) {\n        const p = pads[0];\n        const name = p.id ? p.id.split("(")[0].trim() : "Controller";\n        stat("pad", name.slice(0, 11) || "Connected");\n        if (!state.hasGamepad) {\n          state.hasGamepad = true;\n          toast(`🎮 Controller Connected: ${name.slice(0, 25)}`);\n          post({ type: "gamepadControl", enabled: true });\n        }\n      } else {\n        stat("pad", "None");\n        state.hasGamepad = false;\n      }\n    } catch (_) {}\n  }\n\n  function current() {\n    const latest = window.stream_stats?.latest || {};\n    const client = window.stream_client || {};\n    const rtt = value(latest.rtt_ms);\n    if (rtt > 0) {\n      state.rtts.push(rtt);\n      if (state.rtts.length > 14) state.rtts.shift();\n    }\n    const mean = state.rtts.reduce((sum, item) => sum + item, 0) / Math.max(1, state.rtts.length);\n    const jitter = Math.sqrt(state.rtts.reduce((sum, item) => sum + ((item - mean) ** 2), 0) / Math.max(1, state.rtts.length));\n    const lost = value(latest.lost_frames ?? latest.frames_not_shown);\n    return {\n      latest, client, rtt, jitter, lost,\n      fps: value(latest.fps, value(window.fps)),\n      rate: value(latest.video_mbps, value(latest.mbps)),\n    };\n  }\n\n  let lastGpuCheck = 0;\n  async function syncGpus() {\n    const now = Date.now();\n    if (now - lastGpuCheck < 2000) return;\n    lastGpuCheck = now;\n    try {\n      const response = await fetch(`${controlBase()}/gpus`, { credentials: "same-origin" });\n      if (response.ok) {\n        const data = await response.json();\n        const gpus = data.gpus || [];\n        if (gpus.length >= 2) {\n          const g0 = gpus[0], g1 = gpus[1];\n          stat("gpu", `${g0.utilization}% | ${g1.utilization}%`);\n          stat("gpu0_metric", `${g0.utilization}% · ${(g0.memory_used / 1024).toFixed(1)} GB`);\n          stat("gpu1_metric", `${g1.utilization}% · ${(g1.memory_used / 1024).toFixed(1)} GB`);\n        } else if (gpus.length === 1) {\n          const g0 = gpus[0];\n          stat("gpu", `${g0.utilization}%`);\n          stat("gpu0_metric", `${g0.utilization}% · ${(g0.memory_used / 1024).toFixed(1)} GB`);\n          stat("gpu1_metric", "Inactive");\n        }\n      }\n    } catch (_) {}\n  }\n\n  function render() {\n    const sample = current();\n    const latest = sample.latest;\n    stat("fps", sample.fps ? `${format(sample.fps)} FPS` : "--");\n    stat("rtt", sample.rtt ? `${format(sample.rtt)} ms` : "--");\n    stat("rate", sample.rate ? `${format(sample.rate, 1)} Mbps` : "--");\n    if (!lastGpuCheck) {\n      stat("gpu", latest.gpu_percent !== undefined ? `${format(latest.gpu_percent)}%` : "--");\n    }\n    stat("cpu", latest.cpu_percent !== undefined ? `${format(latest.cpu_percent)}%` : "--");\n    stat("jitter", state.rtts.length > 2 ? `${format(sample.jitter, 1)} ms` : "--");\n    stat("loss", Number.isFinite(Number(sample.lost)) ? String(sample.lost) : "--");\n    const used = value(latest.gpu_mem_used), total = value(latest.gpu_mem_total);\n    const gib = 1024 ** 3;\n    if (!lastGpuCheck && total) {\n      stat("gpu0_metric", `${format(used / gib, 1)} / ${format(total / gib, 1)} GB`);\n    }\n    const transport = sample.client.transport || window.__SELKIES_STREAMING_MODE__ ||\n      document.getElementById("streamModeSelect")?.value || "websockets";\n    stat("transport", transport === "webrtc" ? "WebRTC" : "WebSocket");\n    stat("packet", transport === "webrtc" && latest.packet_loss_percent !== undefined\n      ? `${format(latest.packet_loss_percent, 2)}%` : "N/A (WS)");\n    checkGamepads();\n    syncGpus();\n  }\n\n  function toggleExpanded() {\n    state.expanded = !state.expanded;\n    one(".seven-details").classList.toggle("is-collapsed", !state.expanded);\n    button("expand").textContent = state.expanded ? "⌃" : "⌄";\n    post({ type: "statsOpen", open: true });\n  }\n\n  function togglePointer() {\n    // Selkies owns fullscreen, pointer lock, and the relative-motion handler.\n    // A second requestPointerLock from this HUD can steal its input target.\n    post({ type: "requestGamingMode" });\n    toast("Gaming Mode requested. Press Esc to exit.");\n  }\n\n  async function toggleInputMode() {\n    const btn = button("mode");\n    if (!btn) return;\n    btn.disabled = true;\n    try {\n      const response = await fetch(`${controlBase()}/input-mode`, {\n        method: "POST",\n        credentials: "same-origin",\n        headers: { "X-7BZ-Action": "hud", "Content-Type": "application/json" },\n        body: JSON.stringify({})\n      });\n      if (!response.ok) throw new Error(`HTTP ${response.status}`);\n      const result = await response.json();\n      state.inputMode = result.mode;\n      updateInputModeUI();\n      toast(result.mode === "mouse_keyboard"\n        ? "Mouse & Keyboard selected. Restart the game to remove virtual controllers."\n        : "Gamepad selected. Restart the game to expose Xbox controllers.");\n    } catch (error) {\n      toast(`Input mode switch failed: ${error.message}`);\n    } finally {\n      btn.disabled = false;\n    }\n  }\n\n  function updateInputModeUI() {\n    const isKbm = state.inputMode === "mouse_keyboard";\n    const label = isKbm ? "🖱️ Mode: M+K" : "🎮 Mode: Gamepad";\n    stat("modename", label);\n    const btn = button("mode");\n    if (btn) {\n      btn.textContent = label;\n      btn.classList.toggle("is-active", !isKbm);\n    }\n  }\n\n  async function syncInputMode() {\n    try {\n      const response = await fetch(`${controlBase()}/input-mode`, { credentials: "same-origin" });\n      if (response.ok) {\n        const data = await response.json();\n        state.inputMode = data.mode || "mouse_keyboard";\n        updateInputModeUI();\n      }\n    } catch (_) {}\n  }\n\n  async function toggleEsc() {\n    if (!navigator.keyboard?.lock) { toast("Esc Lock requires a Chromium browser in fullscreen."); return; }\n    try {\n      if (state.esc) { navigator.keyboard.unlock(); state.esc = false; }\n      else { await navigator.keyboard.lock(["Escape"]); state.esc = true; }\n      button("esc").classList.toggle("is-active", state.esc);\n      toast(state.esc ? "Escape is captured by the remote game." : "Escape lock released.");\n    } catch (_error) { toast("Enter fullscreen first, then enable Esc Lock."); }\n  }\n\n  async function resetGame() {\n    const control = button("reset");\n    control.disabled = true;\n    control.textContent = "Stopping…";\n    try {\n      const response = await fetch(`${controlBase()}/reset`, { method: "POST", credentials: "same-origin",\n        headers: { "X-7BZ-Action": "hud" } });\n      if (!response.ok) throw new Error(`HTTP ${response.status}`);\n      const result = await response.json();\n      toast(result.message || "Dashboard restored.");\n      if (document.pointerLockElement) document.exitPointerLock();\n    } catch (error) { toast(`Reset failed: ${error.message}`); }\n    finally { control.disabled = false; control.textContent = "Force Kill / Exit Frozen Game"; }\n  }\n\n  hud.addEventListener("click", (event) => {\n    const action = event.target.closest("[data-action]")?.dataset.action;\n    if (!action) return;\n    if (action === "expand") toggleExpanded();\n    if (action === "fullscreen") post({ type: "requestFullscreen" });\n    if (action === "pointer") togglePointer();\n    if (action === "mode") toggleInputMode();\n    if (action === "esc") toggleEsc();\n    if (action === "audio") {\n      state.audio = true;\n      post({ type: "pipelineControl", pipeline: "audio", enabled: true });\n      post({ type: "setMute", value: false });\n      post({ type: "setVolume", value: 1 });\n      button("audio").classList.add("is-active");\n      toast("Audio enabled at full volume.");\n    }\n    if (action === "reset") resetGame();\n  });\n\n  document.addEventListener("pointerlockchange", () => {\n    state.pointer = Boolean(document.pointerLockElement);\n    button("pointer").classList.toggle("is-active", state.pointer);\n  });\n  window.addEventListener("message", (event) => {\n    if (event.source !== window || event.origin !== window.location.origin) return;\n    if (event.data?.type === "serverSettings" && !state.qualityInitialized) {\n      // An earlier HUD saved 45 FPS and 4 Mbps in Selkies\' browser storage.\n      // Reset the chosen ceiling once, after the dashboard restores its settings.\n      state.qualityInitialized = true;\n      setTimeout(() => post({ type: "settings", settings: {\n        rate_control_mode: "cbr", video_bitrate: 3000, framerate: 60,\n      } }), 1500);\n      return;\n    }\n    if (event.data?.type !== "gamingModeUpdate") return;\n    state.pointer = Boolean(event.data.active);\n    button("pointer").classList.toggle("is-active", state.pointer);\n    if (state.pointer) toast("Gaming Mode active: relative mouse input enabled.");\n  });\n  document.addEventListener("fullscreenchange", () => button("fullscreen").classList.toggle("is-active", Boolean(document.fullscreenElement)));\n  window.addEventListener("keydown", (event) => {\n    if (event.ctrlKey && event.shiftKey && event.code === "KeyM") {\n      event.preventDefault();\n      toggleExpanded();\n    }\n  }, true);\n\n  window.addEventListener("gamepadconnected", (event) => {\n    state.hasGamepad = true;\n    toast(`🎮 Gamepad Connected: ${event.gamepad.id.slice(0, 30)}`);\n    post({ type: "gamepadControl", enabled: true });\n    checkGamepads();\n  });\n  window.addEventListener("gamepaddisconnected", () => {\n    state.hasGamepad = false;\n    toast("Gamepad disconnected");\n    checkGamepads();\n  });\n\n  post({ type: "statsOpen", open: true });\n  syncInputMode();\n  setInterval(render, 1000);\n  setTimeout(render, 700);\n})();\n', encoding='utf-8')

# Write tint2 panel configuration for desktop taskbar
tint2rc = """# Minimal dark tint2 panel configuration
panel_position = bottom center horizontal
panel_size = 100% 32
panel_margin = 0 0
panel_padding = 4 2 4
panel_background_id = 1
wm_menu = 1
panel_dock = 0
panel_layer = top

taskbar_mode = single_desktop
taskbar_padding = 2 0 2
taskbar_background_id = 0
taskbar_active_background_id = 2

task_icon = 1
task_text = 1
task_width = 190
task_maximum_size = 220 28
task_padding = 6 2
task_font = DejaVu Sans 10
task_font_color = #d8e4f5 100
task_active_font_color = #4dcdf2 100

time1_font = DejaVu Sans 10
clock_font_color = #9bdcf5 100
time1_format = %H:%M:%S

rounded = 0
border_width = 1
background_color = #090e1e 100
border_color = #172b4d 100

rounded = 4
border_width = 1
background_color = #152744 100
border_color = #4dcdf2 100
"""
(BENCH / 'tint2rc').write_text(tint2rc, encoding='utf-8')

# Steam launch wrapper - executes under unprivileged 'gamer' user
steam_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
export LD_PRELOAD='/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so'
export SELKIES_JS_SOCKET_PATH="/tmp"
export SDL_JOYSTICK_DISABLE_UDEV=1
export SDL_JOYSTICK_DEVICE=/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003

mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/steam.log"
STEAM_BIN=$(which steam 2>/dev/null || echo /usr/games/steam)

echo "Starting Steam as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    LD_PRELOAD="$LD_PRELOAD" \
    SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \
    SDL_JOYSTICK_DISABLE_UDEV=1 \
    SDL_JOYSTICK_DEVICE="$SDL_JOYSTICK_DEVICE" \
    HOME="/home/gamer" \
    "$STEAM_BIN" -no-cef-sandbox -cef-disable-gpu -cef-force-software-rendering "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_steam.sh').write_text(steam_sh, encoding='utf-8')

# Heroic launch wrapper (Epic Games & GOG) - executes under unprivileged 'gamer' user
heroic_sh = '#!/bin/bash\nexport DISPLAY="${DISPLAY:-:0}"\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\nexport SDL_AUDIODRIVER=pulse\nexport LD_PRELOAD=\'/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so\'\nexport SELKIES_JS_SOCKET_PATH="/tmp"\nexport SDL_JOYSTICK_DISABLE_UDEV=1\nexport SDL_JOYSTICK_DEVICE=/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003\n\nmkdir -p /home/gamer/logs\nLOG="/home/gamer/logs/heroic.log"\nHEROIC_BIN=$(which heroic 2>/dev/null || echo /usr/bin/heroic)\n\n# Source auto-optimizer Wine settings if available\nif [ -f /home/gamer/.config/gamelab_wine_env.sh ]; then\n    source /home/gamer/.config/gamelab_wine_env.sh\nfi\n\n# Automatically sanitize Heroic configuration and clean DXVK DLLs before launch\npython3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py >> "$LOG" 2>&1\nchown -R gamer:gamer /home/gamer 2>/dev/null || true\nchmod -R u+rwX /home/gamer 2>/dev/null || true\n\necho "Starting Heroic as user gamer at $(date)..." >> "$LOG"\nHEROIC_PREFIX=()\nVGL_TARGET=$(PYTHONPATH=/tmp/kaggle-game-dev-bench python3 -c \'from gpu_runtime import diagnose; d=diagnose(); print(d["virtualgl_display"] if d["virtualgl"] and d["nvidia_32bit_gl"] else "")\' 2>/dev/null)\nif [ -n "$VGL_TARGET" ]; then\n    HEROIC_PREFIX=(/opt/VirtualGL/bin/vglrun -d "$VGL_TARGET")\nfi\nif [ "$(id -u)" -eq 0 ]; then\n    USER_PREFIX=(runuser -u gamer --)\nelse\n    USER_PREFIX=()\nfi\nexec "${USER_PREFIX[@]}" env \\\n    DISPLAY="$DISPLAY" \\\n    PATH="$PATH" \\\n    SDL_AUDIODRIVER=pulse \\\n    LD_PRELOAD="$LD_PRELOAD" \\\n    SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \\\n    SDL_JOYSTICK_DISABLE_UDEV=1 \\\n    SDL_JOYSTICK_DEVICE="$SDL_JOYSTICK_DEVICE" \\\n    HOME="/home/gamer" \\\n    "${HEROIC_PREFIX[@]}" "$HEROIC_BIN" --no-sandbox --disable-gpu --disable-gpu-compositing "$@" >> "$LOG" 2>&1\n'
(BENCH / 'launch_heroic.sh').write_text(heroic_sh, encoding='utf-8')

# Minigalaxy launch wrapper (GOG dedicated) - executes under unprivileged 'gamer' user
mini_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/minigalaxy.log"
MINI_BIN=$(which minigalaxy 2>/dev/null || echo /usr/games/minigalaxy)

echo "Starting Minigalaxy as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    HOME="/home/gamer" \
    "$MINI_BIN" "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_minigalaxy.sh').write_text(mini_sh, encoding='utf-8')

# Lutris launch wrapper - executes under unprivileged 'gamer' user
lutris_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export SDL_AUDIODRIVER=pulse
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/lutris.log"
LUTRIS_BIN=$(which lutris 2>/dev/null || echo /usr/games/lutris)

# Match Ubuntu noble C extension ABI (_dbus_bindings) using python3.12 if available
if [ -x /usr/bin/python3.12 ]; then
    PY_BIN="/usr/bin/python3.12"
else
    PY_BIN="python3"
fi

echo "Starting Lutris with $PY_BIN as user gamer at $(date)..." >> "$LOG"
exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    SDL_AUDIODRIVER=pulse \
    HOME="/home/gamer" \
    "$PY_BIN" "$LUTRIS_BIN" "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_lutris.sh').write_text(lutris_sh, encoding='utf-8')

# EA App launch wrapper - executes under unprivileged 'gamer' user in isolated wineprefix
ea_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export WINEPREFIX="/home/gamer/ea-prefix"
export WINEARCH=win64
export WINEDLLOVERRIDES="mscoree="
mkdir -p "$WINEPREFIX" /tmp/ea-store /home/gamer/logs
chown -R gamer:gamer "$WINEPREFIX" /home/gamer/logs 2>/dev/null || true

LOG="/home/gamer/logs/ea.log"
echo "Starting EA App runner at $(date)..." >> "$LOG"

EA_INSTALLER="/tmp/ea-store/EAappInstaller.exe"
if [ ! -f "$EA_INSTALLER" ]; then
    echo "Downloading EAappInstaller.exe..." >> "$LOG"
    wget -q https://origin-a.akamaihd.net/EA-Desktop-Client-Download/installer-releases/EAappInstaller.exe -O "$EA_INSTALLER"
fi

EA_INSTALLED="$WINEPREFIX/drive_c/Program Files/Electronic Arts/EA Desktop/EA Desktop/EADesktop.exe"
if [ -f "$EA_INSTALLED" ]; then
    echo "Running installed EA Desktop: $EA_INSTALLED" >> "$LOG"
    TARGET="$EA_INSTALLED"
else
    echo "Running EA App Installer via Wine64: $EA_INSTALLER" >> "$LOG"
    TARGET="$EA_INSTALLER"
fi

exec env \
    DISPLAY="$DISPLAY" \
    PATH="$PATH" \
    WINEPREFIX="$WINEPREFIX" \
    WINEARCH=win64 \
    WINEDLLOVERRIDES="$WINEDLLOVERRIDES" \
    HOME="/home/gamer" \
    wine "$TARGET" "$@" >> "$LOG" 2>&1
"""
(BENCH / 'launch_ea.sh').write_text(ea_sh, encoding='utf-8')

# Fix Heroic Config Python script
(BENCH / 'fix_heroic_config.py').write_text('"""Apply Heroic settings according to the live, measured graphics backend."""\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nfrom datetime import datetime, timezone\nfrom gpu_runtime import diagnose\n\nHOME = Path(os.environ.get(\'KAGGLE_GAMER_HOME\', \'/home/gamer\'))\nCONFIG = HOME / \'.config/heroic\'\nGAME_CONFIG = CONFIG / \'GamesConfig\'\nRUNNERS = CONFIG / \'tools/wine\'\nBACKUPS = HOME / \'logs/heroic-config-backups\'\nBACKUPS.mkdir(parents=True, exist_ok=True)\nGPU_FACTS = diagnose()\nGPU_DXVK = (GPU_FACTS[\'vulkan_window\'] and GPU_FACTS[\'nvidia_32bit_gl\']\n            and GPU_FACTS[\'vulkan_32bit_loader\'] and GPU_FACTS[\'dxvk_files\'])\n\n# Select best available Wine runner (Prefer Wine-GE over system wine)\nrunner = {\'bin\': \'/usr/bin/wine\', \'wineserver\': \'/usr/bin/wineserver\',\n          \'wineboot\': \'/usr/bin/wineboot\', \'name\': \'Wine Default\', \'type\': \'wine\'}\nfor candidate in [RUNNERS / \'Wine-GE-Proton8-26/bin/wine\', *RUNNERS.glob(\'*/bin/wine\')]:\n    if (candidate.is_file() and os.access(candidate, os.X_OK) and\n            candidate.with_name(\'wineserver\').is_file()):\n        runner = {\'bin\': str(candidate), \'wineserver\': str(candidate.with_name(\'wineserver\')),\n                  \'wineboot\': str(candidate.with_name(\'wineboot\')),\n                  \'name\': candidate.parent.parent.name, \'type\': \'wine\'}\n        break\n\nGAME_SETTINGS_KEYS = (\n    \'autoInstallDxvk\', \'wineVersion\', \'winePrefix\', \'autoInstallVkd3d\',\n    \'preferSystemLibs\', \'launcherArgs\', \'enableEsync\', \'enableFsync\'\n)\n\ndef clean_prefix_dxvk(prefix: Path):\n    """Safely disarm DXVK DLL overrides so WineD3D (OpenGL) renders without Vulkan hardware crash."""\n    for sub in (\'drive_c/windows/system32\', \'drive_c/windows/syswow64\'):\n        sys_dir = prefix / sub\n        if not sys_dir.is_dir():\n            continue\n        for dll_name in (\'dxgi.dll\', \'d3d11.dll\', \'d3d10core.dll\', \'d3d9.dll\'):\n            target = sys_dir / dll_name\n            if target.is_file():\n                try:\n                    bak = target.with_name(target.name + \'.dxvk_bak\')\n                    if not bak.exists():\n                        target.rename(bak)\n                    else:\n                        target.unlink()\n                    print(f\'Disarmed DXVK dll in prefix: {target}\')\n                except OSError as err:\n                    print(f\'Notice disarming {target}: {err}\')\n\ndef repair(path, global_config=False):\n    if not path.exists():\n        return \'missing\'\n    try:\n        old_text = path.read_text(encoding=\'utf-8\')\n        data = json.loads(old_text)\n        if not isinstance(data, dict):\n            return \'unexpected JSON shape\'\n    except (OSError, json.JSONDecodeError) as exc:\n        return f\'not edited: {exc}\'\n\n    settings_objects = []\n    if global_config:\n        # Migrate flat config if found\n        if not isinstance(data.get(\'defaultSettings\'), dict):\n            keys = (\'autoInstallDxvk\', \'autoInstallVkd3d\', \'autoInstallDxvkNvapi\',\n                    \'enableEsync\', \'enableFsync\', \'disableUMU\', \'wineVersion\')\n            data[\'defaultSettings\'] = {key: data.pop(key) for key in keys if key in data}\n        if isinstance(data.get(\'defaultSettings\'), dict):\n            settings_objects.append(data[\'defaultSettings\'])\n        else:\n            settings_objects.append(data)\n    else:\n        # Per-game config: handle stem key, arbitrary single key, multiple keys, or flat\n        if path.stem in data and isinstance(data[path.stem], dict):\n            settings_objects.append(data[path.stem])\n        else:\n            child_matches = [\n                v for v in data.values()\n                if isinstance(v, dict) and any(k in v for k in GAME_SETTINGS_KEYS)\n            ]\n            if child_matches:\n                settings_objects.extend(child_matches)\n            elif any(k in data for k in GAME_SETTINGS_KEYS):\n                settings_objects.append(data)\n            elif len(data) == 1 and isinstance(list(data.values())[0], dict):\n                settings_objects.append(list(data.values())[0])\n\n    if not settings_objects:\n        return \'unexpected settings shape\'\n\n    for settings in settings_objects:\n        for key in (\'autoInstallVkd3d\', \'autoInstallDxvkNvapi\',\n                    \'enableEsync\', \'enableFsync\'):\n            settings[key] = False\n        settings[\'autoInstallDxvk\'] = bool(GPU_DXVK)\n        settings[\'disableUMU\'] = True\n\n        # Assign Wine runner\n        selected = settings.get(\'wineVersion\')\n        if (not isinstance(selected, dict) or selected.get(\'type\') != \'wine\' or\n                not os.path.isfile(selected.get(\'bin\', \'\')) or\n                not os.path.isfile(selected.get(\'wineserver\', \'\'))):\n            settings[\'wineVersion\'] = runner\n\n        # Ensure WINEDLLOVERRIDES forces WineD3D builtins\n        env_opts = settings.get(\'enviromentOptions\')\n        if not isinstance(env_opts, list):\n            env_opts = []\n        found_override = False\n        for opt in env_opts:\n            if isinstance(opt, dict) and opt.get(\'key\') == \'WINEDLLOVERRIDES\':\n                opt[\'value\'] = (\'d3d11,dxgi,d3d9,d3d10core=n,b\' if GPU_DXVK\n                                else \'d3d11,dxgi,d3d9,d3d10core=b\')\n                found_override = True\n                break\n        if not found_override:\n            env_opts.append({\'key\': \'WINEDLLOVERRIDES\', \'value\':\n                             \'d3d11,dxgi,d3d9,d3d10core=n,b\' if GPU_DXVK\n                             else \'d3d11,dxgi,d3d9,d3d10core=b\'})\n        settings[\'enviromentOptions\'] = env_opts\n\n        # Clean prefix DXVK overrides if prefix exists\n        pfx = settings.get(\'winePrefix\')\n        if pfx and os.path.isdir(pfx) and not GPU_DXVK:\n            clean_prefix_dxvk(Path(pfx))\n\n    new_text = json.dumps(data, indent=2) + \'\\n\'\n    if new_text == old_text:\n        return \'already configured\'\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')\n    backup = BACKUPS / f\'{path.name}.{stamp}.bak\'\n    shutil.copy2(path, backup)\n    tmp = path.with_suffix(path.suffix + \'.new\')\n    tmp.write_text(new_text, encoding=\'utf-8\')\n    os.replace(tmp, path)\n    return f\'updated; backup {backup}\'\n\ndef sync_legendary():\n    """Seamlessly share Epic Games auth tokens & installed library between Heroic and Legendary CLI."""\n    heroic_leg = CONFIG / \'legendaryConfig/legendary\'\n    std_leg = HOME / \'.config/legendary\'\n    heroic_leg.mkdir(parents=True, exist_ok=True)\n    std_leg.mkdir(parents=True, exist_ok=True)\n\n    synced = []\n    for fname in (\'user.json\', \'installed.json\', \'config.ini\'):\n        h_f = heroic_leg / fname\n        s_f = std_leg / fname\n        try:\n            if h_f.is_file() and not s_f.is_file():\n                shutil.copy2(h_f, s_f)\n                synced.append(f\'{fname} (Heroic -> Legendary)\')\n            elif s_f.is_file() and not h_f.is_file():\n                shutil.copy2(s_f, h_f)\n                synced.append(f\'{fname} (Legendary -> Heroic)\')\n            elif h_f.is_file() and s_f.is_file():\n                if h_f.stat().st_mtime > s_f.stat().st_mtime:\n                    shutil.copy2(h_f, s_f)\n                    synced.append(f\'{fname} (Heroic -> Legendary, updated)\')\n                elif s_f.stat().st_mtime > h_f.stat().st_mtime:\n                    shutil.copy2(s_f, h_f)\n                    synced.append(f\'{fname} (Legendary -> Heroic, updated)\')\n        except Exception as e:\n            print(f\'Notice syncing {fname}: {e}\')\n    if synced:\n        print(\'Legendary/Heroic synced:\', \', \'.join(synced))\n    else:\n        print(\'Legendary/Heroic auth status: synchronized\')\n\nprint(\'Heroic runner:\', runner[\'bin\'])\nglobal_file = CONFIG / \'config.json\'\nprint(global_file, repair(global_file, global_config=True))\nfor path in sorted(GAME_CONFIG.glob(\'*.json\')):\n    print(path, repair(path))\nfor log in sorted(GAME_CONFIG.glob(\'*-lastPlay.log\'), key=lambda p: p.stat().st_mtime, reverse=True)[:5]:\n    print(f\'Latest play log: {log}\')\n    print(\'\\n\'.join(log.read_text(errors=\'replace\').splitlines()[-25:]))\n\nsync_legendary()\n', encoding='utf-8')

# Fix Heroic Shell wrapper for launcher
fix_heroic_sh = '#!/bin/bash\nexport DISPLAY="${DISPLAY:-:0}"\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\nmkdir -p /home/gamer/logs\nLOG="/home/gamer/logs/heroic_fix.log"\nexec xterm -geometry 95x25 -title "Heroic Config Repair" -e bash -c "echo \'=== Checking GPU and repairing Heroic settings ===\' | tee -a \'$LOG\'; python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>&1 | tee -a \'$LOG\'; chown -R gamer:gamer /home/gamer 2>/dev/null || true; chmod -R u+rwX /home/gamer 2>/dev/null || true; echo \'\'; echo \'Settings updated for the detected graphics backend.\'; echo \'\'; read -p \'Press Enter to close window...\'"\n'
(BENCH / 'fix_heroic.sh').write_text(fix_heroic_sh, encoding='utf-8')

# Legendary interactive menu and graphical interface
(BENCH / 'legendary_menu.sh').write_text('#!/bin/bash\nexport DISPLAY="${DISPLAY:-:0}"\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\nexport SDL_AUDIODRIVER=pulse\nexport HOME="/home/gamer"\nexport LEGENDARY_CONFIG_PATH="$HOME/.config/heroic/legendaryConfig/legendary"\nexport BROWSER="/bin/true"\nmkdir -p "$LEGENDARY_CONFIG_PATH" "$HOME/.config/legendary" "$HOME/logs"\nset -o pipefail\n\n# Source auto-optimizer Wine settings if available\nif [ -f "$HOME/.config/gamelab_wine_env.sh" ]; then\n    source "$HOME/.config/gamelab_wine_env.sh"\nfi\n\n# Ensure credentials and configs are synced on startup\npython3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py >/dev/null 2>&1 || true\n\nwhile true; do\n    clear\n    echo "=================================================================="\n    echo "           LEGENDARY CLI - EPIC GAMES NATIVE MANAGER              "\n    echo "=================================================================="\n    echo " Epic Library Location : $LEGENDARY_CONFIG_PATH"\n    echo " Rendering Backend     : WineD3D (OpenGL software rendering / llvmpipe)"\n    echo " Shared Session        : Synced automatically with Heroic Games"\n    echo "------------------------------------------------------------------"\n    echo " [1] List installed games and launchable apps"\n    echo " [2] Quick Launch an installed game (Brotato, Wolfenstein, etc)"\n    echo " [3] Log in to Epic Games (or check login status)"\n    echo " [4] List all games in your Epic Games library"\n    echo " [5] Install a game from your Epic library"\n    echo " [6] Fix Heroic settings (Disable DXVK / Enable WineD3D)"\n    echo " [7] Open an interactive Bash shell as gamer user"\n    echo " [8] Print a game\'s launch command without starting it"\n    echo " [0] Exit"\n    echo "=================================================================="\n    read -p "Select option [0-8]: " choice\n\n    case "$choice" in\n        1)\n            echo ""\n            echo "Syncing Epic library from Heroic..."\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            echo ""\n            legendary list-installed\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        2)\n            echo ""\n            echo "Installed games:"\n            legendary list-installed\n            echo ""\n            read -p "Enter Game App Name (from the list above): " appname\n            if [ -n "$appname" ]; then\n                WINE_BIN=""\n                for candidate in "/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine" \\\n                                 /home/gamer/.config/heroic/tools/wine/*/bin/wine \\\n                                 /usr/bin/wine /usr/local/bin/wine /usr/games/wine; do\n                    if [ -f "$candidate" ] && [ -x "$candidate" ]; then\n                        WINE_BIN="$candidate"\n                        break\n                    fi\n                done\n                if [ -z "$WINE_BIN" ]; then\n                    echo "Error: No Wine runner found on system!"\n                else\n                    echo "Checking GPU 3D renderer before launching $appname using runner: $WINE_BIN"\n                    export WINEPREFIX="/home/gamer/.wine"\n                    \n                    # Auto-heal prefix if kernel32.dll is missing\n                    if [ ! -f "$WINEPREFIX/drive_c/windows/system32/kernel32.dll" ]; then\n                        echo "Initializing Wine prefix with Wine-GE..."\n                        WINEBOOT_BIN="$(dirname "$WINE_BIN")/wineboot"\n                        if [ -x "$WINEBOOT_BIN" ]; then\n                            WINEDLLOVERRIDES="mscoree,mshtml=d" "$WINEBOOT_BIN" --init 2>/dev/null || true\n                        fi\n                    fi\n                    \n                    python3 /tmp/kaggle-game-dev-bench/gpu_runtime.py launch "$appname" "$WINE_BIN" 2>&1 | tee -a /home/gamer/logs/legendary-game.log\n                    echo "Legendary exit status: ${PIPESTATUS[0]}" | tee -a /home/gamer/logs/legendary-game.log\n                fi\n            fi\n            read -p "Press Enter to return to menu..."\n            ;;\n        3)\n            echo ""\n            echo "Checking Epic Games login status..."\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            if [ -f "$HOME/.config/heroic/legendaryConfig/legendary/user.json" ] || [ -f "$HOME/.config/legendary/user.json" ]; then\n                echo "✓ Found existing Epic Games session!"\n                legendary status 2>/dev/null\n                echo ""\n                read -p "Already logged in! Do you want to re-authenticate with a new code? (y/N): " reauth\n                if [[ ! "$reauth" =~ ^[Yy]$ ]]; then\n                    read -p "Press Enter to return to menu..."\n                    continue\n                fi\n            fi\n\n            echo ""\n            echo "=================================================================="\n            echo "                   EPIC GAMES LOGIN INSTRUCTIONS                  "\n            echo "=================================================================="\n            echo " 1. Open this URL in your PC browser (Chrome/Edge/Firefox):"\n            echo "    https://legendary.gl/epiclogin"\n            echo ""\n            echo " 2. Sign in with your Epic Games account."\n            echo ""\n            echo " 3. The page will display a JSON code containing \'authorizationCode\'."\n            echo "    Example: {\\"authorizationCode\\": \\"3a7b9c1d2e...\\"}"\n            echo ""\n            echo " 4. Copy the authorizationCode value (or the whole JSON text)"\n            echo "    and paste it into the prompt below."\n            echo "=================================================================="\n            echo ""\n            legendary auth\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        4)\n            echo ""\n            legendary list\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        5)\n            echo ""\n            read -p "Enter Game App Name to install: " install_app\n            if [ -n "$install_app" ]; then\n                legendary install "$install_app"\n                python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py 2>/dev/null || true\n            fi\n            read -p "Press Enter to return to menu..."\n            ;;\n        6)\n            echo ""\n            python3 /tmp/kaggle-game-dev-bench/fix_heroic_config.py\n            echo ""\n            read -p "Press Enter to return to menu..."\n            ;;\n        7)\n            echo "Dropping into shell as user gamer. Type \'exit\' to return."\n            bash\n            ;;\n        8)\n            legendary list-installed\n            read -p "Enter Game App Name: " appname\n            if [ -n "$appname" ]; then\n                WINE_BIN=""\n                for candidate in "/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine" \\\n                                 /home/gamer/.config/heroic/tools/wine/*/bin/wine \\\n                                 /usr/bin/wine /usr/local/bin/wine /usr/games/wine; do\n                    if [ -f "$candidate" ] && [ -x "$candidate" ]; then\n                        WINE_BIN="$candidate"\n                        break\n                    fi\n                done\n                legendary launch "$appname" --wine "${WINE_BIN:-wine}" --dry-run 2>&1 | tee -a /home/gamer/logs/legendary-dry-run.log\n                echo "Legendary dry-run exit status: ${PIPESTATUS[0]}" | tee -a /home/gamer/logs/legendary-dry-run.log\n            fi\n            read -p "Press Enter to return to menu..."\n            ;;\n        0)\n            break\n            ;;\n        *)\n            echo "Invalid option."\n            sleep 1\n            ;;\n    esac\ndone\n', encoding='utf-8')
(BENCH / 'legendary_gui.py').write_text('"""Modern Gaming Storefront & Manager for Legendary Epic Games.\n\nFeatures:\n- Dark modern storefront aesthetics with responsive hover effects.\n- Master-detail game showcase with rich metadata and storage badges.\n- Game size display with accurate disk space calculations.\n- Clean uninstallation workflow with confirmation and disk cleanup.\n- File verification and repair via Legendary.\n- Dedicated Storage & Disk Manager tab.\n- Full Epic Games library browsing and installation.\n- Real-time color-coded launch console.\n"""\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport tkinter as tk\nfrom tkinter import ttk, messagebox, scrolledtext\n\nHOME = Path(os.environ.get(\'KAGGLE_GAMER_HOME\', \'/home/gamer\'))\nCONFIG = HOME / \'.config/heroic\'\nHEROIC_LEG = CONFIG / \'legendaryConfig/legendary\'\nSTD_LEG = HOME / \'.config/legendary\'\nBENCH = Path(\'/tmp/kaggle-game-dev-bench\')\nLOGS = HOME / \'logs\'\nLOGS.mkdir(parents=True, exist_ok=True)\n\n# Ensure config paths exist\nHEROIC_LEG.mkdir(parents=True, exist_ok=True)\nSTD_LEG.mkdir(parents=True, exist_ok=True)\n\n# Theme Palette (Deep Cyberpunk / Modern Dark Storefront)\nC_CANVAS = "#0a0e17"       # Main background\nC_PANEL = "#111827"        # Headers & toolbars\nC_CARD = "#1a2234"         # Card background\nC_CARD_HOVER = "#232e44"   # Hover state for cards\nC_BORDER = "#253248"       # Subtle border lines\nC_ACCENT = "#8b5cf6"       # Epic Games purple accent\nC_CYAN = "#06b6d4"         # Neon cyan highlight\nC_FG = "#f1f5f9"           # Primary text\nC_MUTED = "#94a3b8"        # Secondary muted text\nC_GREEN = "#10b981"        # Emerald play button\nC_GREEN_HOVER = "#059669"  # Hover emerald\nC_RED = "#ef4444"          # Danger / uninstall button\nC_RED_HOVER = "#dc2626"    # Hover danger\nC_BLUE = "#3b82f6"         # Info / verify button\nC_BLUE_HOVER = "#2563eb"   # Hover blue\nC_YELLOW = "#f59e0b"       # Warning gold\nC_BTN_SEC = "#242e42"      # Secondary button bg\nC_BTN_SEC_HOV = "#33415c"  # Secondary button hover\n\n\ndef find_best_runner():\n    candidates = [\n        HOME / \'.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine\',\n        *HOME.glob(\'.config/heroic/tools/wine/*/bin/wine\'),\n        Path(\'/usr/bin/wine\'),\n        Path(\'/usr/local/bin/wine\'),\n        Path(\'/usr/games/wine\'),\n    ]\n    for c in candidates:\n        if c.is_file() and os.access(c, os.X_OK):\n            return str(c)\n    return \'/usr/bin/wine\'\n\n\ndef format_size(bytes_val):\n    try:\n        b = float(bytes_val)\n        if b >= 1024 * 1024 * 1024:\n            return f"{b / (1024**3):.2f} GiB"\n        elif b >= 1024 * 1024:\n            return f"{b / (1024**2):.1f} MiB"\n        elif b > 0:\n            return f"{b / 1024:.0f} KiB"\n    except (ValueError, TypeError):\n        pass\n    return "0 B"\n\n\ndef get_disk_info(path="/home/gamer"):\n    try:\n        usage = shutil.disk_usage(path)\n        return {\n            "total": usage.total,\n            "used": usage.used,\n            "free": usage.free,\n            "total_str": format_size(usage.total),\n            "used_str": format_size(usage.used),\n            "free_str": format_size(usage.free),\n            "percent": int((usage.used / usage.total) * 100) if usage.total > 0 else 0\n        }\n    except Exception:\n        return {"total": 0, "used": 0, "free": 0, "total_str": "Unknown", "used_str": "Unknown", "free_str": "Unknown", "percent": 0}\n\n\ndef make_hover_button(parent, text, normal_bg, hover_bg, normal_fg="#ffffff", hover_fg="#ffffff",\n                      font=("Liberation Sans", 10, "bold"), padx=12, pady=6, command=None, cursor="hand2"):\n    """Helper to create polished interactive buttons with hover glow."""\n    btn = tk.Button(parent, text=text, bg=normal_bg, fg=normal_fg, activebackground=hover_bg,\n                    activeforeground=hover_fg, relief="flat", font=font, padx=padx, pady=pady,\n                    cursor=cursor, command=command, borderwidth=0, highlightthickness=0)\n    btn.bind("<Enter>", lambda e: btn.configure(bg=hover_bg, fg=hover_fg))\n    btn.bind("<Leave>", lambda e: btn.configure(bg=normal_bg, fg=normal_fg))\n    return btn\n\n\nclass LegendaryApp(tk.Tk):\n    def __init__(self):\n        super().__init__()\n        self.title("7BZ Dev Cloud Gaming · Legendary Storefront")\n        self.geometry("1060x650")\n        self.minsize(900, 550)\n        self.configure(bg=C_CANVAS)\n\n        # Active process tracker\n        self.active_proc = None\n        self.proc_thread = None\n        self.installed_games_data = {}\n        self.selected_game_app = None\n\n        # Apply dark theme styling\n        self.setup_styles()\n\n        # Build UI layout\n        self.create_header()\n        self.create_tabs()\n        self.create_bottom_bar()\n\n        # Initial data load\n        self.refresh_all()\n\n    def setup_styles(self):\n        self.style = ttk.Style(self)\n        try:\n            self.style.theme_use("clam")\n        except tk.TclError:\n            pass\n\n        self.style.configure(".", background=C_CANVAS, foreground=C_FG, font=("Liberation Sans", 10))\n        self.style.configure("TNotebook", background=C_CANVAS, borderwidth=0)\n        self.style.configure("TNotebook.Tab", background=C_PANEL, foreground=C_MUTED,\n                             padding=[18, 9], font=("Liberation Sans", 10, "bold"), borderwidth=0)\n        self.style.map("TNotebook.Tab",\n                       background=[("selected", C_CARD), ("active", C_CARD_HOVER)],\n                       foreground=[("selected", C_CYAN), ("active", C_FG)])\n\n        # Treeview styling (clean modern table)\n        self.style.configure("Treeview", background="#121722", foreground=C_FG,\n                             fieldbackground="#121722", rowheight=36, font=("Liberation Sans", 10),\n                             borderwidth=0)\n        self.style.configure("Treeview.Heading", background=C_PANEL, foreground="#ffffff",\n                             font=("Liberation Sans", 10, "bold"), padding=[8, 8], relief="flat")\n        self.style.map("Treeview",\n                       background=[("selected", "#2a3754"), ("active", "#1b253b")],\n                       foreground=[("selected", "#ffffff"), ("active", C_FG)])\n\n        self.style.configure("Horizontal.TProgressbar", background=C_CYAN, troughcolor=C_PANEL,\n                             borderwidth=0, thickness=12)\n\n    def create_header(self):\n        header = tk.Frame(self, bg=C_PANEL, height=68, highlightbackground=C_BORDER, highlightthickness=1)\n        header.pack(fill="x", side="top", padx=10, pady=(8, 4))\n\n        # Brand / Logo frame\n        brand_frame = tk.Frame(header, bg=C_PANEL)\n        brand_frame.pack(side="left", padx=14, pady=8)\n\n        lbl_badge = tk.Label(brand_frame, text=" EPIC STORE ", bg=C_ACCENT, fg="#ffffff",\n                             font=("Liberation Sans", 9, "bold"), padx=6, pady=2)\n        lbl_badge.pack(side="left", padx=(0, 8))\n\n        lbl_title = tk.Label(brand_frame, text="Legendary Game Center", bg=C_PANEL, fg=C_FG,\n                             font=("Liberation Sans", 13, "bold"))\n        lbl_title.pack(side="left")\n\n        # Stats & Metrics Pills (Center)\n        stats_frame = tk.Frame(header, bg=C_PANEL)\n        stats_frame.pack(side="left", padx=20, pady=8)\n\n        self.lbl_storage_pill = tk.Label(stats_frame, text="💾 Free: Calculating...", bg=C_CARD, fg=C_CYAN,\n                                         font=("Liberation Sans", 9, "bold"), padx=10, pady=4)\n        self.lbl_storage_pill.pack(side="left", padx=4)\n\n        self.lbl_games_pill = tk.Label(stats_frame, text="🎮 Games: 0", bg=C_CARD, fg=C_FG,\n                                       font=("Liberation Sans", 9), padx=10, pady=4)\n        self.lbl_games_pill.pack(side="left", padx=4)\n\n        # Quick Actions & Account (Right)\n        actions_frame = tk.Frame(header, bg=C_PANEL)\n        actions_frame.pack(side="right", padx=12, pady=8)\n\n        btn_refresh = make_hover_button(actions_frame, "🔄 Refresh", C_BTN_SEC, C_BTN_SEC_HOV,\n                                        normal_fg=C_FG, padx=10, pady=4, command=self.sync_and_refresh)\n        btn_refresh.pack(side="left", padx=4)\n\n        btn_fix = make_hover_button(actions_frame, "🛠️ Fix Wine", C_BTN_SEC, C_BTN_SEC_HOV,\n                                    normal_fg=C_FG, padx=10, pady=4, command=self.run_sanitizer)\n        btn_fix.pack(side="left", padx=4)\n\n        btn_login = make_hover_button(actions_frame, "🔑 Login", C_CARD, C_CARD_HOVER,\n                                      normal_fg=C_CYAN, padx=10, pady=4, command=self.open_login_dialog)\n        btn_login.pack(side="left", padx=4)\n\n        self.lbl_account = tk.Label(actions_frame, text="Checking...", bg=C_PANEL, fg=C_YELLOW,\n                                    font=("Liberation Sans", 9, "bold"))\n        self.lbl_account.pack(side="left", padx=(10, 4))\n\n    def create_tabs(self):\n        self.notebook = ttk.Notebook(self)\n        self.notebook.pack(fill="both", expand=True, padx=10, pady=4)\n\n        # Tab 1: Installed Games (Master-Detail Storefront View)\n        self.tab_installed = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_installed, text=" 🎮 Installed Games ")\n        self.build_installed_tab()\n\n        # Tab 2: Full Epic Store & Cloud Library\n        self.tab_library = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_library, text=" 🛒 Epic Cloud Library ")\n        self.build_library_tab()\n\n        # Tab 3: Storage & Disk Manager\n        self.tab_storage = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_storage, text=" 📊 Storage & Disk ")\n        self.build_storage_tab()\n\n        # Tab 4: Launch Console & Live Logs\n        self.tab_console = ttk.Frame(self.notebook)\n        self.notebook.add(self.tab_console, text=" 📜 Launch Console & Logs ")\n        self.build_console_tab()\n\n    def build_installed_tab(self):\n        container = tk.Frame(self.tab_installed, bg=C_CANVAS)\n        container.pack(fill="both", expand=True, padx=4, pady=4)\n\n        # Split: Left (Game Catalog Table), Right (Hero Game Details & Command Hub)\n        paned = tk.PanedWindow(container, orient="horizontal", bg=C_CANVAS, sashwidth=4, bd=0)\n        paned.pack(fill="both", expand=True)\n\n        # LEFT PANE: Games List\n        left_frame = tk.Frame(paned, bg=C_CANVAS)\n        paned.add(left_frame, minsize=460, stretch="always")\n\n        # Search Bar\n        search_bar = tk.Frame(left_frame, bg=C_PANEL, highlightbackground=C_BORDER, highlightthickness=1)\n        search_bar.pack(fill="x", side="top", pady=(0, 6), ipady=3)\n\n        tk.Label(search_bar, text=" 🔍 ", bg=C_PANEL, fg=C_MUTED, font=("Liberation Sans", 10)).pack(side="left", padx=4)\n        self.ent_search_installed = tk.Entry(search_bar, bg=C_PANEL, fg="#ffffff", insertbackground="#ffffff",\n                                             relief="flat", font=("Liberation Sans", 10), borderwidth=0)\n        self.ent_search_installed.pack(side="left", fill="x", expand=True, padx=4)\n        self.ent_search_installed.bind("<KeyRelease>", self.filter_installed_games)\n\n        self.lbl_installed_count = tk.Label(search_bar, text="0 games", bg=C_PANEL, fg=C_MUTED,\n                                            font=("Liberation Sans", 9), padx=8)\n        self.lbl_installed_count.pack(side="right")\n\n        # Games Treeview\n        tree_frame = tk.Frame(left_frame, bg=C_CANVAS)\n        tree_frame.pack(fill="both", expand=True)\n\n        columns = ("title", "size", "version", "status")\n        self.tree_installed = ttk.Treeview(tree_frame, columns=columns, show="headings", selectmode="browse")\n        self.tree_installed.heading("title", text="Game Title")\n        self.tree_installed.heading("size", text="Installed Size")\n        self.tree_installed.heading("version", text="Version")\n        self.tree_installed.heading("status", text="Status")\n\n        self.tree_installed.column("title", width=250, anchor="w")\n        self.tree_installed.column("size", width=110, anchor="center")\n        self.tree_installed.column("version", width=90, anchor="center")\n        self.tree_installed.column("status", width=90, anchor="center")\n\n        scroll = ttk.Scrollbar(tree_frame, orient="vertical", command=self.tree_installed.yview)\n        self.tree_installed.configure(yscrollcommand=scroll.set)\n\n        self.tree_installed.pack(side="left", fill="both", expand=True)\n        scroll.pack(side="right", fill="y")\n\n        self.tree_installed.bind("<<TreeviewSelect>>", self.on_installed_select)\n        self.tree_installed.bind("<Double-1>", lambda e: self.launch_selected())\n\n        # RIGHT PANE: Hero Game Showcase Card\n        self.right_frame = tk.Frame(paned, bg=C_CARD, highlightbackground=C_BORDER, highlightthickness=1)\n        paned.add(self.right_frame, minsize=380, stretch="never")\n\n        self.build_game_showcase(self.right_frame)\n\n    def build_game_showcase(self, parent):\n        """Build the rich game card in the right pane."""\n        # Top banner frame\n        self.card_header = tk.Frame(parent, bg="#151d2d", padx=16, pady=14)\n        self.card_header.pack(fill="x", side="top")\n\n        self.lbl_card_badge = tk.Label(self.card_header, text="READY TO PLAY", bg=C_GREEN, fg="#ffffff",\n                                       font=("Liberation Sans", 8, "bold"), padx=6, pady=2)\n        self.lbl_card_badge.pack(anchor="w")\n\n        self.lbl_card_title = tk.Label(self.card_header, text="Select a Game", bg="#151d2d", fg=C_FG,\n                                       font=("Liberation Sans", 14, "bold"), wraplength=340, justify="left")\n        self.lbl_card_title.pack(anchor="w", pady=(6, 2))\n\n        self.lbl_card_app_id = tk.Label(self.card_header, text="App Name: --", bg="#151d2d", fg=C_MUTED,\n                                        font=("Liberation Sans Mono", 8))\n        self.lbl_card_app_id.pack(anchor="w")\n\n        # Details Content Frame\n        self.card_body = tk.Frame(parent, bg=C_CARD, padx=16, pady=12)\n        self.card_body.pack(fill="both", expand=True)\n\n        # Storage & Size Box (Feature requested: display game size)\n        size_box = tk.Frame(self.card_body, bg=C_PANEL, highlightbackground=C_BORDER, highlightthickness=1, padx=12, pady=10)\n        size_box.pack(fill="x", pady=6)\n\n        tk.Label(size_box, text="STORAGE FOOTPRINT", bg=C_PANEL, fg=C_CYAN,\n                 font=("Liberation Sans", 8, "bold")).pack(anchor="w")\n\n        self.lbl_card_size = tk.Label(size_box, text="-- GiB", bg=C_PANEL, fg="#ffffff",\n                                      font=("Liberation Sans", 16, "bold"))\n        self.lbl_card_size.pack(anchor="w", pady=(2, 2))\n\n        self.lbl_card_path = tk.Label(size_box, text="Path: --", bg=C_PANEL, fg=C_MUTED,\n                                      font=("Liberation Sans", 8), wraplength=320, justify="left")\n        self.lbl_card_path.pack(anchor="w")\n\n        # Game Properties Grid\n        props_frame = tk.Frame(self.card_body, bg=C_CARD)\n        props_frame.pack(fill="x", pady=8)\n\n        tk.Label(props_frame, text="Version:", bg=C_CARD, fg=C_MUTED, font=("Liberation Sans", 9)).grid(row=0, column=0, sticky="w", pady=2)\n        self.lbl_prop_version = tk.Label(props_frame, text="--", bg=C_CARD, fg=C_FG, font=("Liberation Sans", 9, "bold"))\n        self.lbl_prop_version.grid(row=0, column=1, sticky="w", padx=10, pady=2)\n\n        tk.Label(props_frame, text="Platform:", bg=C_CARD, fg=C_MUTED, font=("Liberation Sans", 9)).grid(row=1, column=0, sticky="w", pady=2)\n        self.lbl_prop_platform = tk.Label(props_frame, text="Windows 64-bit", bg=C_CARD, fg=C_FG, font=("Liberation Sans", 9))\n        self.lbl_prop_platform.grid(row=1, column=1, sticky="w", padx=10, pady=2)\n\n        tk.Label(props_frame, text="Graphics Engine:", bg=C_CARD, fg=C_MUTED, font=("Liberation Sans", 9)).grid(row=2, column=0, sticky="w", pady=2)\n        self.lbl_prop_engine = tk.Label(props_frame, text="NVIDIA Direct Xorg", bg=C_CARD, fg=C_CYAN, font=("Liberation Sans", 9, "bold"))\n        self.lbl_prop_engine.grid(row=2, column=1, sticky="w", padx=10, pady=2)\n\n        tk.Label(props_frame, text="Executable:", bg=C_CARD, fg=C_MUTED, font=("Liberation Sans", 9)).grid(row=3, column=0, sticky="w", pady=2)\n        self.lbl_prop_exe = tk.Label(props_frame, text="--", bg=C_CARD, fg=C_FG, font=("Liberation Sans Mono", 8))\n        self.lbl_prop_exe.grid(row=3, column=1, sticky="w", padx=10, pady=2)\n\n        # Action Buttons Container (Bottom of right card)\n        actions_container = tk.Frame(parent, bg=C_CARD, padx=16, pady=14)\n        actions_container.pack(fill="x", side="bottom")\n\n        # 1. Primary Action: Launch Game\n        self.btn_launch_hero = make_hover_button(\n            actions_container, "▶  LAUNCH GAME", C_GREEN, C_GREEN_HOVER,\n            font=("Liberation Sans", 11, "bold"), pady=8, command=self.launch_selected\n        )\n        self.btn_launch_hero.pack(fill="x", pady=(0, 6))\n\n        # Secondary Actions Row (Verify, Open Folder)\n        sec_row = tk.Frame(actions_container, bg=C_CARD)\n        sec_row.pack(fill="x", pady=(0, 6))\n\n        self.btn_verify = make_hover_button(\n            sec_row, "🔍 Verify Files", C_BTN_SEC, C_BTN_SEC_HOV,\n            normal_fg=C_FG, font=("Liberation Sans", 9), padx=8, pady=5, command=self.verify_selected\n        )\n        self.btn_verify.pack(side="left", fill="x", expand=True, padx=(0, 4))\n\n        self.btn_folder = make_hover_button(\n            sec_row, "📁 Open Folder", C_BTN_SEC, C_BTN_SEC_HOV,\n            normal_fg=C_FG, font=("Liberation Sans", 9), padx=8, pady=5, command=self.open_selected_folder\n        )\n        self.btn_folder.pack(side="right", fill="x", expand=True, padx=(4, 0))\n\n        # 2. Danger Action: Uninstall Game (Feature requested: Delete game)\n        self.btn_uninstall = make_hover_button(\n            actions_container, "🗑️  UNINSTALL / DELETE GAME", C_RED, C_RED_HOVER,\n            font=("Liberation Sans", 10, "bold"), pady=6, command=self.uninstall_selected\n        )\n        self.btn_uninstall.pack(fill="x")\n\n    def on_installed_select(self, event=None):\n        """Update the hero showcase when a game is selected in the list."""\n        sel = self.tree_installed.selection()\n        if not sel:\n            return\n        item = self.tree_installed.item(sel[0])\n        app_name = item.get("tags", [""])[0] if item.get("tags") else ""\n        if not app_name or app_name not in self.installed_games_data:\n            return\n\n        self.selected_game_app = app_name\n        g = self.installed_games_data[app_name]\n        title = g.get(\'title\') or app_name\n        version = g.get(\'version\', \'Unknown\')\n        install_size = g.get(\'install_size\', 0)\n        path_str = g.get(\'install_path\', \'N/A\')\n        exe_str = g.get(\'executable\', \'N/A\')\n\n        self.lbl_card_title.config(text=title)\n        self.lbl_card_app_id.config(text=f"App ID: {app_name}")\n        self.lbl_card_size.config(text=format_size(install_size))\n        self.lbl_card_path.config(text=f"Location: {path_str}")\n        self.lbl_prop_version.config(text=f"v{version}")\n        self.lbl_prop_exe.config(text=exe_str)\n\n        # Detect special renderer (e.g. Wolfenstein idTech5 OpenGL)\n        if exe_str.lower() == "wolfneworder_x64.exe":\n            self.lbl_prop_engine.config(text="idTech5 Direct Xorg (Verified 60 FPS)", fg=C_GREEN)\n        else:\n            self.lbl_prop_engine.config(text="Direct NVIDIA Xorg / DXVK", fg=C_CYAN)\n\n    def filter_installed_games(self, event=None):\n        query = self.ent_search_installed.get().strip().lower()\n        for item in self.tree_installed.get_children():\n            self.tree_installed.delete(item)\n\n        filtered_count = 0\n        for app_name, g in sorted(self.installed_games_data.items(), key=lambda x: str(x[1].get(\'title\', \'\')).lower()):\n            title = g.get(\'title\') or app_name\n            if query and query not in title.lower() and query not in app_name.lower():\n                continue\n            version = g.get(\'version\', \'Unknown\')\n            size_str = format_size(g.get(\'install_size\', 0))\n            status = "✓ Ready"\n            self.tree_installed.insert("", "end", values=(title, size_str, version, status), tags=(app_name,))\n            filtered_count += 1\n\n        self.lbl_installed_count.config(text=f"{filtered_count} games")\n        children = self.tree_installed.get_children()\n        if children:\n            self.tree_installed.selection_set(children[0])\n            self.on_installed_select()\n\n    def build_library_tab(self):\n        container = tk.Frame(self.tab_library, bg=C_CANVAS)\n        container.pack(fill="both", expand=True, padx=4, pady=4)\n\n        toolbar = tk.Frame(container, bg=C_PANEL, highlightbackground=C_BORDER, highlightthickness=1)\n        toolbar.pack(fill="x", side="top", pady=(0, 6), ipady=4, padx=2)\n\n        btn_load = make_hover_button(toolbar, "🔍 Fetch Cloud Library", C_ACCENT, "#7c3aed",\n                                     font=("Liberation Sans", 9, "bold"), padx=12, pady=5, command=self.fetch_remote_library)\n        btn_load.pack(side="left", padx=8)\n\n        btn_install = make_hover_button(toolbar, "📥 Install Game", C_GREEN, C_GREEN_HOVER,\n                                        font=("Liberation Sans", 9, "bold"), padx=12, pady=5, command=self.install_selected)\n        btn_install.pack(side="left", padx=4)\n\n        # Filter\n        tk.Label(toolbar, text="Filter:", bg=C_PANEL, fg=C_MUTED, font=("Liberation Sans", 9)).pack(side="left", padx=(16, 4))\n        self.ent_filter_lib = tk.Entry(toolbar, bg=C_CARD, fg="#ffffff", insertbackground="#ffffff", relief="flat", font=("Liberation Sans", 9))\n        self.ent_filter_lib.pack(side="left", fill="x", expand=True, padx=(0, 10))\n        self.ent_filter_lib.bind("<KeyRelease>", self.filter_library)\n\n        columns = ("title", "app_name", "status")\n        self.tree_library = ttk.Treeview(container, columns=columns, show="headings", selectmode="browse")\n        self.tree_library.heading("title", text="Game Title")\n        self.tree_library.heading("app_name", text="Epic App ID")\n        self.tree_library.heading("status", text="Installation Status")\n\n        self.tree_library.column("title", width=420, anchor="w")\n        self.tree_library.column("app_name", width=280, anchor="w")\n        self.tree_library.column("status", width=140, anchor="center")\n\n        scroll = ttk.Scrollbar(container, orient="vertical", command=self.tree_library.yview)\n        self.tree_library.configure(yscrollcommand=scroll.set)\n\n        self.tree_library.pack(side="left", fill="both", expand=True)\n        scroll.pack(side="right", fill="y")\n\n    def build_storage_tab(self):\n        """Dedicated Storage & Disk Manager tab with visual usage meter."""\n        container = tk.Frame(self.tab_storage, bg=C_CANVAS, padx=12, pady=10)\n        container.pack(fill="both", expand=True)\n\n        # Top Storage Meter Card\n        meter_card = tk.Frame(container, bg=C_CARD, highlightbackground=C_BORDER, highlightthickness=1, padx=16, pady=14)\n        meter_card.pack(fill="x", side="top", pady=(0, 10))\n\n        tk.Label(meter_card, text="HOST DISK USAGE & ALLOCATION", bg=C_CARD, fg=C_CYAN,\n                 font=("Liberation Sans", 9, "bold")).pack(anchor="w")\n\n        self.storage_progress = ttk.Progressbar(meter_card, orient="horizontal", mode="determinate",\n                                               style="Horizontal.TProgressbar")\n        self.storage_progress.pack(fill="x", pady=(8, 8))\n\n        stats_row = tk.Frame(meter_card, bg=C_CARD)\n        stats_row.pack(fill="x")\n\n        self.lbl_storage_total = tk.Label(stats_row, text="Total: --", bg=C_CARD, fg=C_FG, font=("Liberation Sans", 10, "bold"))\n        self.lbl_storage_total.pack(side="left", padx=(0, 20))\n\n        self.lbl_storage_used = tk.Label(stats_row, text="Used: --", bg=C_CARD, fg=C_YELLOW, font=("Liberation Sans", 10, "bold"))\n        self.lbl_storage_used.pack(side="left", padx=(0, 20))\n\n        self.lbl_storage_free = tk.Label(stats_row, text="Available Free: --", bg=C_CARD, fg=C_GREEN, font=("Liberation Sans", 10, "bold"))\n        self.lbl_storage_free.pack(side="left", padx=(0, 20))\n\n        # Action Buttons on Storage\n        btn_cleanup = make_hover_button(stats_row, "🧹 Cleanup Old Cache & Manifests", C_BTN_SEC, C_BTN_SEC_HOV,\n                                        font=("Liberation Sans", 9), padx=10, pady=4, command=self.run_cleanup)\n        btn_cleanup.pack(side="right")\n\n        # Table of installed games with exact disk consumption\n        tk.Label(container, text="Installed Games Breakdown by Size", bg=C_CANVAS, fg=C_FG,\n                 font=("Liberation Sans", 11, "bold")).pack(anchor="w", pady=(8, 4))\n\n        columns = ("title", "size_bytes", "size_fmt", "path")\n        self.tree_storage = ttk.Treeview(container, columns=columns, show="headings", selectmode="browse")\n        self.tree_storage.heading("title", text="Game Title")\n        self.tree_storage.heading("size_bytes", text="Raw Bytes")\n        self.tree_storage.heading("size_fmt", text="Disk Footprint")\n        self.tree_storage.heading("path", text="Installed Folder")\n\n        self.tree_storage.column("title", width=280, anchor="w")\n        self.tree_storage.column("size_bytes", width=120, anchor="center")\n        self.tree_storage.column("size_fmt", width=120, anchor="center")\n        self.tree_storage.column("path", width=360, anchor="w")\n\n        s_scroll = ttk.Scrollbar(container, orient="vertical", command=self.tree_storage.yview)\n        self.tree_storage.configure(yscrollcommand=s_scroll.set)\n\n        self.tree_storage.pack(side="left", fill="both", expand=True)\n        s_scroll.pack(side="right", fill="y")\n\n    def build_console_tab(self):\n        container = tk.Frame(self.tab_console, bg=C_CANVAS)\n        container.pack(fill="both", expand=True, padx=4, pady=4)\n\n        toolbar = tk.Frame(container, bg=C_PANEL, highlightbackground=C_BORDER, highlightthickness=1)\n        toolbar.pack(fill="x", side="top", pady=(0, 6), ipady=3, padx=2)\n\n        btn_stop = make_hover_button(toolbar, "⏹  STOP RUNNING GAME", C_RED, C_RED_HOVER,\n                                     font=("Liberation Sans", 9, "bold"), padx=12, pady=4, command=self.stop_active_game)\n        btn_stop.pack(side="left", padx=8)\n\n        btn_clear = make_hover_button(toolbar, "🧹 Clear Log", C_BTN_SEC, C_BTN_SEC_HOV,\n                                      normal_fg=C_FG, font=("Liberation Sans", 9), padx=10, pady=4,\n                                      command=lambda: self.log_text.delete("1.0", tk.END))\n        btn_clear.pack(side="left", padx=4)\n\n        self.lbl_status = tk.Label(toolbar, text="System Ready · Idle", bg=C_PANEL, fg=C_MUTED,\n                                   font=("Liberation Sans", 9, "bold"))\n        self.lbl_status.pack(side="right", padx=12)\n\n        self.log_text = scrolledtext.ScrolledText(container, bg="#0d1117", fg="#c9d1d9",\n                                                 insertbackground="#ffffff", font=("Liberation Mono", 9),\n                                                 relief="flat", borderwidth=0)\n        self.log_text.pack(fill="both", expand=True)\n\n        # Configure color tags for rich log output\n        self.log_text.tag_configure("info", foreground="#58a6ff")\n        self.log_text.tag_configure("success", foreground=C_GREEN)\n        self.log_text.tag_configure("warn", foreground=C_YELLOW)\n        self.log_text.tag_configure("error", foreground=C_RED)\n        self.log_text.tag_configure("dim", foreground="#8b949e")\n\n    def create_bottom_bar(self):\n        bar = tk.Frame(self, bg=C_PANEL, height=28, highlightbackground=C_BORDER, highlightthickness=1)\n        bar.pack(fill="x", side="bottom", padx=10, pady=(2, 6))\n\n        lbl_tip = tk.Label(bar, text="⚡ Direct NVIDIA Xorg: Tesla T4 hardware acceleration enabled · H.264 60 FPS",\n                           bg=C_PANEL, fg=C_MUTED, font=("Liberation Sans", 8))\n        lbl_tip.pack(side="left", padx=10, pady=4)\n\n        self.lbl_runner_status = tk.Label(bar, text="🍷 Wine-GE-Proton8-26", bg=C_PANEL, fg=C_CYAN,\n                                          font=("Liberation Sans", 8, "bold"))\n        self.lbl_runner_status.pack(side="right", padx=10, pady=4)\n\n    def log(self, text, tag="dim"):\n        def _append():\n            self.log_text.insert(tk.END, text + "\\n", tag)\n            self.log_text.see(tk.END)\n        self.after(0, _append)\n\n    def refresh_all(self):\n        self.refresh_account_status()\n        self.refresh_runner_status()\n        self.refresh_disk_status()\n        self.refresh_installed_games()\n\n    def refresh_disk_status(self):\n        info = get_disk_info()\n        self.lbl_storage_pill.config(text=f"💾 Free: {info[\'free_str\']} / {info[\'total_str\']}")\n        self.storage_progress["value"] = info["percent"]\n        self.lbl_storage_total.config(text=f"Total: {info[\'total_str\']}")\n        self.lbl_storage_used.config(text=f"Used: {info[\'used_str\']} ({info[\'percent\']}%)")\n        self.lbl_storage_free.config(text=f"Available Free: {info[\'free_str\']}")\n\n    def refresh_account_status(self):\n        user_file = None\n        for p in [HEROIC_LEG / \'user.json\', STD_LEG / \'user.json\']:\n            if p.is_file():\n                user_file = p\n                break\n\n        if user_file:\n            try:\n                data = json.loads(user_file.read_text(encoding=\'utf-8\'))\n                display_name = data.get(\'displayName\', \'Epic User\')\n                acc_id = data.get(\'account_id\', \'\')\n                short_id = f" ({acc_id[:8]}...)" if acc_id else ""\n                self.lbl_account.config(text=f"🟢 {display_name}{short_id}", fg=C_GREEN)\n                return\n            except Exception:\n                pass\n        self.lbl_account.config(text="🔴 Not Logged In", fg=C_RED)\n\n    def refresh_runner_status(self):\n        runner = find_best_runner()\n        name = "Wine-GE-Proton8-26" if "Wine-GE" in runner else Path(runner).name\n        self.lbl_runner_status.config(text=f"🍷 {name} (NVIDIA Direct GLX/DXVK)")\n\n    def refresh_installed_games(self):\n        self.installed_games_data = {}\n        for item in self.tree_installed.get_children():\n            self.tree_installed.delete(item)\n        for item in self.tree_storage.get_children():\n            self.tree_storage.delete(item)\n\n        installed_file = None\n        for p in [HEROIC_LEG / \'installed.json\', STD_LEG / \'installed.json\']:\n            if p.is_file():\n                installed_file = p\n                break\n\n        games = {}\n        if installed_file:\n            try:\n                games = json.loads(installed_file.read_text(encoding=\'utf-8\'))\n            except Exception as e:\n                self.log(f"Notice reading installed.json: {e}", "warn")\n\n        if not games:\n            # Fallback: Query legendary list-installed --json\n            try:\n                res = subprocess.run(["legendary", "list-installed", "--json"], capture_output=True, text=True, timeout=5)\n                if res.returncode == 0 and res.stdout.strip():\n                    data = json.loads(res.stdout)\n                    if isinstance(data, list):\n                        for g in data:\n                            games[g.get(\'app_name\', \'\')] = g\n                    elif isinstance(data, dict):\n                        games = data\n            except Exception:\n                pass\n\n        self.installed_games_data = games\n        total_game_bytes = 0\n\n        for app_name, g in sorted(games.items(), key=lambda x: str(x[1].get(\'title\', \'\')).lower()):\n            title = g.get(\'title\') or app_name\n            version = g.get(\'version\', \'Unknown\')\n            install_size = g.get(\'install_size\', 0)\n            total_game_bytes += install_size\n            size_str = format_size(install_size)\n            path_str = g.get(\'install_path\', \'N/A\')\n            status = "✓ Ready"\n\n            self.tree_installed.insert("", "end", values=(title, size_str, version, status), tags=(app_name,))\n            self.tree_storage.insert("", "end", values=(title, f"{install_size:,} bytes", size_str, path_str), tags=(app_name,))\n\n        count = len(games)\n        self.lbl_installed_count.config(text=f"{count} games")\n        self.lbl_games_pill.config(text=f"🎮 {count} Installed ({format_size(total_game_bytes)})")\n\n        children = self.tree_installed.get_children()\n        if children:\n            self.tree_installed.selection_set(children[0])\n            self.tree_installed.focus(children[0])\n            self.on_installed_select()\n        else:\n            self.lbl_card_title.config(text="No Games Installed")\n            self.lbl_card_app_id.config(text="App ID: --")\n            self.lbl_card_size.config(text="0 B")\n            self.lbl_card_path.config(text="Location: None")\n\n    def sync_and_refresh(self):\n        self.log("▶ Syncing Heroic and Legendary configurations...", "info")\n        try:\n            fix_script = BENCH / \'fix_heroic_config.py\'\n            if fix_script.is_file():\n                res = subprocess.run([sys.executable, str(fix_script)], capture_output=True, text=True, timeout=10)\n                self.log(res.stdout, "dim")\n                if res.stderr:\n                    self.log(res.stderr, "warn")\n            self.refresh_all()\n            self.log("✓ Sync completed successfully.", "success")\n        except Exception as exc:\n            self.log(f"Sync error: {exc}", "error")\n\n    def run_sanitizer(self):\n        self.notebook.select(self.tab_console)\n        self.log("▶ Checking live GPU renderer and updating Wine settings...", "info")\n        threading.Thread(target=self._run_sanitizer_worker, daemon=True).start()\n\n    def _run_sanitizer_worker(self):\n        try:\n            fix_script = BENCH / \'fix_heroic_config.py\'\n            if fix_script.is_file():\n                res = subprocess.run([sys.executable, str(fix_script)], capture_output=True, text=True, timeout=15)\n                self.log(res.stdout, "dim")\n            subprocess.run(["chown", "-R", "gamer:gamer", "/home/gamer"], check=False)\n            subprocess.run(["chmod", "-R", "u+rwX", "/home/gamer"], check=False)\n            self.log("✓ Wine settings updated for the detected graphics backend.", "success")\n            self.after(0, self.refresh_all)\n        except Exception as e:\n            self.log(f"Error during sanitization: {e}", "error")\n\n    def launch_selected(self):\n        if not self.selected_game_app or self.selected_game_app not in self.installed_games_data:\n            messagebox.showinfo("No Selection", "Please select a game from the list to launch.")\n            return\n\n        g = self.installed_games_data[self.selected_game_app]\n        title = g.get(\'title\') or self.selected_game_app\n        app_name = self.selected_game_app\n\n        self.notebook.select(self.tab_console)\n        self.log(f"\\n==================================================================", "info")\n        self.log(f"🚀 Launching game: {title} ({app_name})", "success")\n        self.log(f"==================================================================", "info")\n\n        runner = find_best_runner()\n        self.log(f"Runner: {runner}", "dim")\n        self.log("Configuring display and audio...", "dim")\n\n        threading.Thread(target=self._launch_worker, args=(app_name, title, runner), daemon=True).start()\n\n    def _launch_worker(self, app_name, title, runner):\n        env = dict(os.environ)\n        env["DISPLAY"] = os.environ.get("DISPLAY", ":0")\n        env["HOME"] = str(HOME)\n        env["WINEPREFIX"] = str(HOME / ".wine")\n        env["CUDA_VISIBLE_DEVICES"] = "0"\n        direct_xorg = (env["DISPLAY"].split(".", 1)[0] == ":1" and\n                       (BENCH / "gpu-xorg-ready").exists())\n        if direct_xorg:\n            env["GAMELAB_DIRECT_XORG"] = "1"\n            for key in ("VGL_DISPLAY", "MESA_GL_VERSION_OVERRIDE",\n                        "MESA_GLSL_VERSION_OVERRIDE", "__NV_PRIME_RENDER_OFFLOAD"):\n                env.pop(key, None)\n        else:\n            env["VGL_DISPLAY"] = "egl0"\n\n        # Automatically verify and heal Wine prefix before launch if kernel32.dll is missing\n        kernel32 = HOME / ".wine/drive_c/windows/system32/kernel32.dll"\n        if not kernel32.is_file():\n            self.log("▶ Detected uninitialized Wine prefix. Auto-initializing with Wine-GE...", "warn")\n            wineboot_bin = Path(runner).with_name("wineboot")\n            if wineboot_bin.is_file():\n                try:\n                    init_env = dict(env)\n                    init_env["WINEDLLOVERRIDES"] = "mscoree,mshtml=d"\n                    subprocess.run([str(wineboot_bin), "--init"], env=init_env, check=False, timeout=30)\n                    self.log("✓ Wine prefix initialized.", "success")\n                except Exception as wb_err:\n                    self.log(f"Notice auto-initializing prefix: {wb_err}", "warn")\n\n        from gpu_runtime import find_game_binary\n        game_binary = find_game_binary(app_name)\n        if game_binary and game_binary.name.lower() == "wolfneworder_x64.exe":\n            cmd = ["python3", "/tmp/kaggle-game-dev-bench/launch_idtech5.py",\n                   *(["--direct-xorg"] if direct_xorg else []),\n                   "--wine", runner, str(game_binary)]\n        else:\n            cmd = ["python3", "/tmp/kaggle-game-dev-bench/gpu_runtime.py",\n                   "launch", app_name, runner]\n\n        self.lbl_status.config(text=f"▶ Running: {title}", fg=C_GREEN)\n\n        try:\n            self.active_proc = subprocess.Popen(\n                cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env\n            )\n            for line in self.active_proc.stdout:\n                self.log(line.rstrip())\n            self.active_proc.wait()\n            self.log(f"\\nGame process exited with status: {self.active_proc.returncode}", "info")\n        except Exception as exc:\n            self.log(f"Launch exception: {exc}", "error")\n        finally:\n            self.active_proc = None\n            self.lbl_status.config(text="System Ready · Idle", fg=C_MUTED)\n\n    def stop_active_game(self):\n        if self.active_proc and self.active_proc.poll() is None:\n            self.log("Terminating active game process...", "warn")\n            self.active_proc.terminate()\n            time.sleep(1)\n            if self.active_proc.poll() is None:\n                self.active_proc.kill()\n            self.log("Process stopped successfully.", "success")\n        else:\n            self.log("No active game process running.", "dim")\n\n    def uninstall_selected(self):\n        """Feature requested: Delete / Uninstall game with confirmation & disk cleanup."""\n        if not self.selected_game_app or self.selected_game_app not in self.installed_games_data:\n            messagebox.showinfo("No Selection", "Please select an installed game to uninstall.")\n            return\n\n        g = self.installed_games_data[self.selected_game_app]\n        title = g.get(\'title\') or self.selected_game_app\n        app_name = self.selected_game_app\n        size_str = format_size(g.get(\'install_size\', 0))\n        install_path = g.get(\'install_path\', \'\')\n\n        # Confirmation Dialog\n        msg = (\n            f"Are you sure you want to uninstall this game?\\n\\n"\n            f"🎮 Title: {title}\\n"\n            f"📦 Space to be freed: {size_str}\\n"\n            f"📂 Location: {install_path}\\n\\n"\n            f"This will remove the game files from disk."\n        )\n        if not messagebox.askyesno("Confirm Game Uninstallation", msg, icon="warning"):\n            return\n\n        self.notebook.select(self.tab_console)\n        self.log(f"\\n==================================================================", "warn")\n        self.log(f"🗑️ Starting Uninstallation of \'{title}\' ({app_name})...", "warn")\n        self.log(f"==================================================================", "warn")\n\n        threading.Thread(target=self._uninstall_worker, args=(app_name, title, install_path), daemon=True).start()\n\n    def _uninstall_worker(self, app_name, title, install_path):\n        try:\n            self.log(f"Executing: legendary uninstall {app_name} -y", "info")\n            cmd = ["legendary", "uninstall", app_name, "-y"]\n            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n            for line in proc.stdout:\n                self.log(line.rstrip())\n            proc.wait()\n\n            # Clean leftover directory if necessary\n            if install_path and Path(install_path).is_dir():\n                try:\n                    self.log(f"Cleaning remaining directory at {install_path}...", "info")\n                    shutil.rmtree(install_path, ignore_errors=True)\n                except Exception as clean_err:\n                    self.log(f"Notice cleaning directory: {clean_err}", "warn")\n\n            self.log(f"\\n✓ \'{title}\' has been successfully uninstalled.", "success")\n            self.after(0, self.refresh_all)\n            self.after(0, lambda: messagebox.showinfo("Uninstalled", f"\'{title}\' was successfully uninstalled!"))\n        except Exception as exc:\n            self.log(f"Uninstallation error: {exc}", "error")\n\n    def verify_selected(self):\n        """Feature: Verify & repair game files via Legendary."""\n        if not self.selected_game_app:\n            return\n        g = self.installed_games_data.get(self.selected_game_app, {})\n        title = g.get(\'title\') or self.selected_game_app\n        app_name = self.selected_game_app\n\n        self.notebook.select(self.tab_console)\n        self.log(f"\\n▶ Verifying game integrity for \'{title}\'...", "info")\n        threading.Thread(target=self._verify_worker, args=(app_name,), daemon=True).start()\n\n    def _verify_worker(self, app_name):\n        try:\n            cmd = ["legendary", "verify", app_name]\n            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n            for line in proc.stdout:\n                self.log(line.rstrip())\n            proc.wait()\n            self.log(f"Verification finished with status {proc.returncode}.", "success")\n        except Exception as e:\n            self.log(f"Verification error: {e}", "error")\n\n    def open_selected_folder(self):\n        if not self.selected_game_app:\n            return\n        g = self.installed_games_data.get(self.selected_game_app, {})\n        path_str = g.get(\'install_path\', \'\')\n        if path_str and Path(path_str).is_dir():\n            try:\n                subprocess.Popen(["xdg-open", path_str])\n            except Exception:\n                messagebox.showinfo("Game Path", f"Game folder:\\n{path_str}")\n        else:\n            messagebox.showinfo("Path Not Found", f"Game path does not exist on disk:\\n{path_str}")\n\n    def run_cleanup(self):\n        self.notebook.select(self.tab_console)\n        self.log("▶ Cleaning up Legendary cache, temp files, and old manifests...", "info")\n        threading.Thread(target=self._cleanup_worker, daemon=True).start()\n\n    def _cleanup_worker(self):\n        try:\n            res = subprocess.run(["legendary", "cleanup"], capture_output=True, text=True, timeout=30)\n            self.log(res.stdout, "dim")\n            self.log("✓ Cleanup finished successfully.", "success")\n            self.after(0, self.refresh_disk_status)\n        except Exception as e:\n            self.log(f"Cleanup error: {e}", "error")\n\n    def fetch_remote_library(self):\n        self.log("▶ Fetching Epic Games library via legendary list...", "info")\n        self.ent_filter_lib.delete(0, tk.END)\n        for item in self.tree_library.get_children():\n            self.tree_library.delete(item)\n\n        threading.Thread(target=self._fetch_library_worker, daemon=True).start()\n\n    def _fetch_library_worker(self):\n        try:\n            res = subprocess.run(["legendary", "list", "--json"], capture_output=True, text=True, timeout=25)\n            if res.returncode == 0 and res.stdout.strip():\n                items = json.loads(res.stdout)\n                installed_ids = set(self.installed_games_data.keys())\n\n                self.all_library_items = []\n                for item in items:\n                    app = item.get("app_name", "")\n                    title = item.get("app_title") or item.get("title") or app\n                    status = "✓ Installed" if app in installed_ids else "Available"\n                    self.all_library_items.append((title, app, status))\n\n                self.all_library_items.sort(key=lambda x: x[0].lower())\n                self.after(0, self._populate_library_table, self.all_library_items)\n                self.log(f"✓ Found {len(self.all_library_items)} games in your Epic library.", "success")\n            else:\n                self.log(f"Legendary list returned code {res.returncode}:\\n{res.stderr or res.stdout}", "error")\n        except Exception as e:\n            self.log(f"Library fetch error: {e}", "error")\n\n    def _populate_library_table(self, items):\n        for item in self.tree_library.get_children():\n            self.tree_library.delete(item)\n        for row in items:\n            self.tree_library.insert("", "end", values=row)\n\n    def filter_library(self, event=None):\n        query = self.ent_filter_lib.get().strip().lower()\n        if not hasattr(self, \'all_library_items\'):\n            return\n        if not query:\n            self._populate_library_table(self.all_library_items)\n        else:\n            filtered = [r for r in self.all_library_items if query in r[0].lower() or query in r[1].lower()]\n            self._populate_library_table(filtered)\n\n    def install_selected(self):\n        sel = self.tree_library.selection()\n        if not sel:\n            messagebox.showinfo("No Selection", "Please select a game to install from the library.")\n            return\n\n        values = self.tree_library.item(sel[0], "values")\n        title = values[0]\n        app_name = values[1]\n\n        if not messagebox.askyesno("Install Game", f"Install \'{title}\' ({app_name}) via Legendary?"):\n            return\n\n        self.notebook.select(self.tab_console)\n        self.log(f"\\n▶ Starting installation of {title} ({app_name})...", "info")\n        threading.Thread(target=self._install_worker, args=(app_name,), daemon=True).start()\n\n    def _install_worker(self, app_name):\n        try:\n            cmd = ["legendary", "install", app_name, "-y"]\n            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\n            for line in proc.stdout:\n                self.log(line.rstrip())\n            proc.wait()\n            self.log(f"Installation finished with code {proc.returncode}.", "success")\n            self.sync_and_refresh()\n        except Exception as e:\n            self.log(f"Install error: {e}", "error")\n\n    def open_login_dialog(self):\n        dlg = tk.Toplevel(self)\n        dlg.title("Epic Games Authentication")\n        dlg.geometry("560x380")\n        dlg.resizable(False, False)\n        dlg.configure(bg=C_CANVAS)\n        dlg.transient(self)\n        dlg.grab_set()\n\n        tk.Label(dlg, text="Log In to Epic Games Store", bg=C_CANVAS, fg="#ffffff",\n                 font=("Liberation Sans", 13, "bold")).pack(pady=(16, 10))\n\n        frame_steps = tk.Frame(dlg, bg=C_PANEL, padx=14, pady=12, highlightbackground=C_BORDER, highlightthickness=1)\n        frame_steps.pack(fill="x", padx=16, pady=6)\n\n        msg = (\n            "1. Open this URL in your PC browser (Chrome/Edge/Firefox):\\n"\n            "   https://legendary.gl/epiclogin\\n\\n"\n            "2. Sign in with your Epic Games account.\\n\\n"\n            "3. The browser will show text containing \'authorizationCode\'.\\n"\n            "4. Copy and paste the code (or the entire JSON string) below:"\n        )\n        tk.Label(frame_steps, text=msg, bg=C_PANEL, fg=C_FG, justify="left",\n                 font=("Liberation Sans", 9)).pack(anchor="w")\n\n        def _copy_url():\n            self.clipboard_clear()\n            self.clipboard_append("https://legendary.gl/epiclogin")\n            lbl_info.config(text="✓ Link copied to clipboard!", fg=C_GREEN)\n\n        btn_copy = make_hover_button(frame_steps, "📋 Copy Login Link", C_CARD, C_CARD_HOVER,\n                                     font=("Liberation Sans", 9), padx=10, pady=4, command=_copy_url)\n        btn_copy.pack(anchor="w", pady=(8, 0))\n\n        # Input entry\n        lbl_code = tk.Label(dlg, text="Authorization Code or JSON Text:", bg=C_CANVAS, fg=C_MUTED,\n                            font=("Liberation Sans", 9, "bold"))\n        lbl_code.pack(anchor="w", padx=18, pady=(10, 2))\n\n        ent_code = tk.Entry(dlg, bg=C_PANEL, fg="#ffffff", insertbackground="#ffffff",\n                            relief="flat", font=("Liberation Mono", 10), highlightbackground=C_BORDER, highlightthickness=1)\n        ent_code.pack(fill="x", padx=18, ipady=4)\n\n        lbl_info = tk.Label(dlg, text="", bg=C_CANVAS, fg=C_YELLOW, font=("Liberation Sans", 9))\n        lbl_info.pack(pady=4)\n\n        def _submit():\n            raw_input = ent_code.get().strip()\n            if not raw_input:\n                lbl_info.config(text="Please paste an authorization code first.", fg=C_RED)\n                return\n\n            code = raw_input\n            if "{" in raw_input and "authorizationCode" in raw_input:\n                try:\n                    data = json.loads(raw_input)\n                    code = data.get("authorizationCode", raw_input)\n                except Exception:\n                    m = re.search(r\'"authorizationCode"\\s*:\\s*"([a-f0-9]+)"\', raw_input, re.I)\n                    if m:\n                        code = m.group(1)\n\n            lbl_info.config(text="Authenticating with Epic Games...", fg=C_YELLOW)\n            dlg.update()\n\n            def _auth_worker():\n                res = subprocess.run(["legendary", "auth", "--code", code], capture_output=True, text=True)\n                if res.returncode == 0:\n                    self.sync_and_refresh()\n                    self.after(0, lambda: (\n                        messagebox.showinfo("Success", "Epic Games authentication succeeded!"),\n                        dlg.destroy()\n                    ))\n                else:\n                    err_msg = res.stderr or res.stdout\n                    self.after(0, lambda: lbl_info.config(text=f"Auth failed: {err_msg[:60]}", fg=C_RED))\n\n            threading.Thread(target=_auth_worker, daemon=True).start()\n\n        btn_frame = tk.Frame(dlg, bg=C_CANVAS)\n        btn_frame.pack(fill="x", padx=18, pady=12)\n\n        make_hover_button(btn_frame, "Cancel", C_CARD, C_CARD_HOVER, normal_fg=C_MUTED,\n                         padx=12, pady=5, command=dlg.destroy).pack(side="left")\n\n        make_hover_button(btn_frame, "🚀 Complete Login", C_GREEN, C_GREEN_HOVER,\n                         font=("Liberation Sans", 10, "bold"), padx=14, pady=5, command=_submit).pack(side="right")\n\n\nif __name__ == "__main__":\n    app = LegendaryApp()\n    app.mainloop()\n', encoding='utf-8')
(BENCH / 'gpu_runtime.py').write_text('"""Choose a measured graphics path for Legendary/Wine games.\n\nNVENC and nvidia-smi are deliberately not treated as proof of 3D rendering.\nRun ``python3 gpu_runtime.py diagnose`` inside the live desktop session.\n"""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\n\n\ndef run(command, env=None, timeout=12):\n    try:\n        result = subprocess.run(command, env=env, text=True, errors="replace",\n                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                                timeout=timeout, check=False)\n        return result.returncode, result.stdout\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        return 1, str(exc)\n\n\ndef clean_env():\n    env = os.environ.copy()\n    for key in ("LIBGL_ALWAYS_SOFTWARE", "GALLIUM_DRIVER", "MESA_LOADER_DRIVER_OVERRIDE",\n                "MESA_GL_VERSION_OVERRIDE", "LIBGL_DRIVERS_PATH",\n                "__GLX_VENDOR_LIBRARY_NAME"):\n        env.pop(key, None)\n    # The Selkies input interposer must not affect graphics probes.\n    env.pop("LD_PRELOAD", None)\n    return env\n\n\ndef nvidia_renderer(output):\n    vendor = re.search(r"OpenGL vendor string:\\s*(.+)", output, re.I)\n    renderer = re.search(r"OpenGL renderer string:\\s*(.+)", output, re.I)\n    # Tesla renderers often say "Tesla T4/PCIe/SSE2", without "NVIDIA".\n    return bool(vendor and "NVIDIA" in vendor.group(1).upper() and renderer\n                and not re.search(r"llvmpipe|softpipe|software", renderer.group(1), re.I))\n\n\ndef has_32bit_library(name):\n    injected = os.environ.get("GAMELAB_NVIDIA_LIB32", "")\n    if injected:\n        candidate = Path(injected) / name\n        try:\n            with candidate.open("rb") as library:\n                if library.read(5) == b"\\x7fELF\\x01":\n                    return True\n        except OSError:\n            pass\n    rc, out = run(["ldconfig", "-p"], timeout=4)\n    return rc == 0 and any(f"{name} (libc6)" in line\n                           for line in out.splitlines())\n\n\ndef dxvk_available():\n    root = Path("/opt/gamelab-dxvk/dxvk-2.6.2")\n    return all((root / arch / (name + ".dll")).is_file()\n               for arch in ("x32", "x64")\n               for name in ("d3d9", "d3d10core", "d3d11", "dxgi"))\n\n\ndef egl_device_probe(env):\n    """Test the NVIDIA EGL device directly, independent of Xorg and Xvfb."""\n    eglinfo = Path("/opt/VirtualGL/bin/eglinfo")\n    if not eglinfo.is_file():\n        return False, "VirtualGL eglinfo is not installed", ""\n    rc, listing = run([str(eglinfo), "-e"], env, timeout=8)\n    candidates = list(dict.fromkeys(re.findall(r"\\begl\\d+\\b", listing)))[:8] if rc == 0 else []\n    if not candidates:\n        candidates = ["egl0"]\n    last_result = listing[-240:]\n    for target in candidates:\n        rc, out = run([str(eglinfo), target], env, timeout=12)\n        renderer = next((line.strip() for line in out.splitlines()\n                         if line.strip().startswith("OpenGL renderer string:")), "")\n        vendor = next((line.strip() for line in out.splitlines()\n                       if line.strip().startswith("OpenGL vendor string:")), "")\n        if (rc == 0 and "NVIDIA" in vendor.upper() and renderer\n                and not re.search(r"llvmpipe|softpipe|software", renderer, re.I)):\n            return True, renderer, target\n        last_result = renderer or out[-240:]\n    return False, last_result, ""\n\n\ndef cpu_quota_count():\n    """Return the cgroup CPU quota as a whole-core budget, if one exists."""\n    try:\n        quota, period = Path("/sys/fs/cgroup/cpu.max").read_text().split()[:2]\n        if quota != "max":\n            return max(1, math.floor(int(quota) / int(period)))\n    except (OSError, ValueError, ZeroDivisionError):\n        pass\n    try:\n        quota = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text())\n        period = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text())\n        if quota > 0:\n            return max(1, math.floor(quota / period))\n    except (OSError, ValueError, ZeroDivisionError):\n        pass\n    return None\n\n\ndef software_threads():\n    override = os.environ.get("LP_NUM_THREADS", "")\n    if override.isdigit() and 1 <= int(override) <= 32:\n        return int(override)\n    try:\n        cores = len(os.sched_getaffinity(0))\n    except AttributeError:\n        cores = os.cpu_count() or 1\n    quota = cpu_quota_count()\n    if quota is not None:\n        cores = min(cores, quota)\n    reserve = 2 if cores >= 4 else 1\n    return max(1, min(8, cores - reserve))\n\n\ndef lower_software_resolution(env):\n    """Best-effort temporary Xvfb mode change; game settings may differ."""\n    requested = os.environ.get("GAMELAB_SOFTWARE_RESOLUTION", "854x480")\n    if requested == "off" or not shutil.which("xrandr") or not env.get("DISPLAY"):\n        return None\n    if requested not in ("640x360", "854x480", "1024x576", "1280x720"):\n        print(f"Unsupported GAMELAB_SOFTWARE_RESOLUTION={requested!r}; keeping desktop mode.",\n              file=sys.stderr)\n        return None\n    rc, out = run(["xrandr", "--current"], env, timeout=5)\n    match = re.search(r"\\bcurrent (\\d+) x (\\d+)\\b", out)\n    if rc != 0 or not match:\n        return None\n    previous = f"{match.group(1)}x{match.group(2)}"\n    if previous == requested:\n        return None\n    rc, out = run(["xrandr", "-s", requested], env, timeout=5)\n    if rc != 0:\n        print(f"Could not set {requested} desktop mode: {out[-200:]}", file=sys.stderr)\n        return None\n    print(f"Software desktop mode: {requested} (restore {previous} on exit)", flush=True)\n    return previous\n\n\ndef install_dxvk(prefix):\n    """Install both bitnesses without following Wine DLL symlinks."""\n    root = Path("/opt/gamelab-dxvk/dxvk-2.6.2")\n    for arch, subdir in (("x64", "system32"), ("x32", "syswow64")):\n        destination = Path(prefix) / "drive_c/windows" / subdir\n        if not destination.is_dir():\n            raise RuntimeError(f"Wine prefix is incomplete: {destination}")\n        for name in ("d3d9", "d3d10core", "d3d11", "dxgi"):\n            target = destination / (name + ".dll")\n            backup = destination / (name + ".dll.gamelab-original")\n            temporary = destination / (name + ".dll.gamelab-new")\n            shutil.copyfile(root / arch / target.name, temporary)\n            if (target.exists() or target.is_symlink()) and not backup.exists():\n                os.replace(target, backup)\n            os.replace(temporary, target)\n\n\ndef install_vkd3d(prefix):\n    """Install VKD3D-Proton (DirectX 12) if available."""\n    candidates = list(Path("/opt/gamelab-vkd3d").glob("vkd3d-proton-*"))\n    if not candidates:\n        return\n    root = candidates[0]\n    for arch, subdir in (("x64", "system32"), ("x86", "syswow64")):\n        src_dir = root / arch\n        if not src_dir.is_dir():\n            continue\n        dst_dir = Path(prefix) / "drive_c/windows" / subdir\n        if not dst_dir.is_dir():\n            continue\n        for name in ("d3d12.dll", "d3d12core.dll"):\n            src_file = src_dir / name\n            if src_file.is_file():\n                shutil.copyfile(src_file, dst_dir / name)\n\n\ndef diagnose():\n    env = clean_env()\n    facts = {\n        "display": env.get("DISPLAY", ""),\n        "nvidia_control": Path("/dev/nvidiactl").exists(),\n        "nvidia_gpu": Path("/dev/nvidia0").exists(),\n        "nvidia_modeset": Path("/dev/nvidia-modeset").exists(),\n        "nvidia_icd": any(Path(p).exists() for p in (\n            "/etc/vulkan/icd.d/nvidia_icd.json",\n            "/usr/share/vulkan/icd.d/nvidia_icd.json")) or any(\n                Path(p).is_file() and "nvidia" in Path(p).name.lower()\n                for p in (env.get("VK_DRIVER_FILES") or env.get("VK_ICD_FILENAMES") or "").split(":") if p),\n        "direct_gl": False, "virtualgl": False, "virtualgl_display": "",\n        "vulkan_device": False,\n        "vulkan_window": False,\n        "nvidia_32bit_gl": has_32bit_library("libGLX_nvidia.so.0"),\n        "vulkan_32bit_loader": has_32bit_library("libvulkan.so.1"),\n        "dxvk_files": dxvk_available(),\n    }\n    facts["egl_device"], facts["egl_renderer"], facts["egl_device_display"] = egl_device_probe(env)\n    if shutil.which("glxinfo") and env.get("DISPLAY"):\n        rc, out = run(["glxinfo", "-B"], env)\n        facts["direct_gl"] = rc == 0 and nvidia_renderer(out)\n        facts["gl_renderer"] = next((line.strip() for line in out.splitlines()\n                                     if "OpenGL renderer string:" in line), out[-240:])\n        vgl = shutil.which("vglrun") or "/opt/VirtualGL/bin/vglrun"\n        if Path(vgl).is_file() and facts["nvidia_control"] and facts["nvidia_gpu"]:\n            candidates = [facts["egl_device_display"]] if facts["egl_device"] else ["egl0"]\n            if Path("/tmp/kaggle-game-dev-bench/gpu-xorg-ready").exists():\n                candidates.append(":1")\n            for target in candidates:\n                rc, out = run([vgl, "-d", target, "glxinfo", "-B"], env)\n                if rc == 0 and nvidia_renderer(out):\n                    facts["virtualgl"] = True\n                    facts["virtualgl_display"] = target\n                    facts["vgl_renderer"] = next((line.strip() for line in out.splitlines()\n                                                   if "OpenGL renderer string:" in line), out[-240:])\n                    break\n    if facts["nvidia_control"] and facts["nvidia_gpu"] and shutil.which("vulkaninfo"):\n        rc, out = run(["vulkaninfo", "--summary"], env)\n        facts["vulkan_device"] = rc == 0 and bool(re.search(\n            r"deviceName\\s*=.*(?:NVIDIA|Tesla T4)", out, re.I))\n        facts["vulkan_summary"] = out[-500:]\n    if facts["vulkan_device"] and env.get("DISPLAY"):\n        rc, out = run([sys.executable, "-m", "selkies.gpu_probe", "--vulkan-window"], env)\n        facts["vulkan_window"] = rc == 0 and out.strip() == "true"\n    return facts\n\n\ndef legendary_install_path(app):\n    """Return an installed Epic title\'s directory from Legendary metadata."""\n    home = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\n    candidates = (\n        home / ".config/heroic/legendaryConfig/legendary/installed.json",\n        home / ".config/legendary/installed.json",\n    )\n    for installed in candidates:\n        try:\n            record = json.loads(installed.read_text(encoding="utf-8")).get(app, {})\n            path = Path(record.get("install_path") or record.get("installPath") or "")\n            if path.is_dir():\n                return path\n        except (OSError, ValueError, AttributeError):\n            continue\n    return None\n\n\ndef unreal_shipping_binary(app):\n    """Find a UE Shipping executable while excluding prerequisite launchers."""\n    install = legendary_install_path(app)\n    if install is None:\n        return None\n    matches = []\n    for path in install.rglob("*-Win64-Shipping.exe"):\n        lowered = str(path).lower()\n        if "prereq" not in lowered and "redist" not in lowered and "crashreport" not in lowered:\n            matches.append(path)\n    matches.sort(key=lambda path: ("binaries\\\\win64" not in str(path).lower().replace("/", "\\\\"), len(str(path))))\n    return matches[0] if matches else None\n\n\ndef find_game_binary(app):\n    """Find the game\'s actual executable binary (supporting UE, id Tech, Unity, custom engines)."""\n    ue_shipping = unreal_shipping_binary(app)\n    if ue_shipping and ue_shipping.is_file():\n        return ue_shipping\n\n    home = Path(os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer"))\n    install_candidates = []\n\n    # Check installed.json metadata (exact key match, then fuzzy/case-insensitive match)\n    for installed_file in (\n        home / ".config/heroic/legendaryConfig/legendary/installed.json",\n        home / ".config/legendary/installed.json",\n    ):\n        try:\n            data = json.loads(installed_file.read_text(encoding="utf-8"))\n            if app in data:\n                record = data[app]\n                ip = Path(record.get("install_path") or record.get("installPath") or "")\n                ex = record.get("executable") or ""\n                if ip.is_dir():\n                    if ex and (ip / ex).is_file():\n                        return ip / ex\n                    for p in ip.rglob("*.exe"):\n                        low = p.name.lower()\n                        if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                            continue\n                        if ex and p.name.lower() == Path(ex).name.lower():\n                            return p\n                        if "x64" in low or "wolf" in low:\n                            return p\n                    install_candidates.append(ip)\n            for key, record in data.items():\n                if not isinstance(record, dict):\n                    continue\n                k_low = key.lower()\n                a_low = app.lower()\n                t_low = str(record.get("title", "")).lower()\n                if a_low in k_low or k_low in a_low or a_low in t_low:\n                    ip = Path(record.get("install_path") or record.get("installPath") or "")\n                    ex = record.get("executable") or ""\n                    if ip.is_dir():\n                        if ex and (ip / ex).is_file():\n                            return ip / ex\n                        for p in ip.rglob("*.exe"):\n                            low = p.name.lower()\n                            if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                                continue\n                            if ex and p.name.lower() == Path(ex).name.lower():\n                                return p\n                            if "x64" in low or "wolf" in low:\n                                return p\n                        install_candidates.append(ip)\n        except Exception:\n            pass\n\n    install_candidates.extend([\n        home / "game-builds",\n        Path("/kaggle/working"),\n    ])\n\n    # Only include Program Files directories that actually match the app or game name\n    for pf_root in (home / ".wine/drive_c/Program Files", home / ".wine/drive_c/Program Files (x86)"):\n        if pf_root.is_dir():\n            for sub in pf_root.iterdir():\n                if sub.is_dir() and not sub.name.lower().startswith("internet"):\n                    if app.lower() in sub.name.lower() or "wolf" in sub.name.lower():\n                        install_candidates.append(sub)\n\n    SYSTEM_EXES = {"iexplore.exe", "explorer.exe", "notepad.exe", "regedit.exe", "msiexec.exe", "wineboot.exe", "rundll32.exe", "cmd.exe", "taskmgr.exe"}\n    found_exes = []\n    for root in install_candidates:\n        if not root.is_dir():\n            continue\n        try:\n            for p in root.rglob("*.exe"):\n                low = p.name.lower()\n                if low in SYSTEM_EXES:\n                    continue\n                if any(skip in low for skip in ("prereq", "redist", "crash", "unins", "setup", "dxsetup", "vcredist", "epicgameslauncher")):\n                    continue\n                found_exes.append(p)\n        except Exception:\n            pass\n\n    if found_exes:\n        a_low = app.lower()\n        found_exes.sort(key=lambda p: (\n            0 if a_low in p.name.lower() else 1,\n            0 if ("x64" in p.name.lower() or "64" in p.name.lower()) else 1,\n            len(p.parts)\n        ))\n        return found_exes[0]\n    return None\n\n\ndef get_input_mode() -> str:\n    """Read the active input mode (mouse_keyboard or gamepad)."""\n    mode_file = Path("/tmp/kaggle-game-dev-bench/input-mode.json")\n    try:\n        if mode_file.is_file():\n            data = json.loads(mode_file.read_text(encoding="utf-8"))\n            return data.get("mode", "mouse_keyboard")\n    except Exception:\n        pass\n    return os.environ.get("SEVEN_BZ_INPUT_MODE", "mouse_keyboard")\n\n\ndef configure_mouse(wine_bin, env):\n    """Apply verified Wine DirectInput, X11 pointer capture, and raw sensitivity."""\n    if not wine_bin:\n        return\n    values = (\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\DirectInput", "MouseWarpOverride", "force"),\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\X11 Driver", "GrabPointer", "Y"),\n        ("HKEY_CURRENT_USER\\\\Software\\\\Wine\\\\X11 Driver", "UseTakeFocus", "N"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseSpeed", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseThreshold1", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseThreshold2", "0"),\n        ("HKEY_CURRENT_USER\\\\Control Panel\\\\Mouse", "MouseSensitivity", "10"),\n    )\n    for key, name, value in values:\n        try:\n            subprocess.run([wine_bin, "reg", "add", key, "/v", name, "/t", "REG_SZ",\n                            "/d", value, "/f"], env=env, check=False,\n                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=8)\n        except (OSError, subprocess.TimeoutExpired):\n            pass\n\n\ndef select_graphics_backend(facts, preference="auto", is_64bit=True):\n    """Select only paths whose required presentation probe succeeded."""\n    if preference not in ("auto", "dxvk", "wined3d"):\n        raise ValueError("GAMELAB_GRAPHICS_BACKEND must be auto, dxvk, or wined3d")\n    dxvk_ready = (facts["vulkan_window"] and (facts["vulkan_32bit_loader"] or is_64bit)\n                  and facts["dxvk_files"])\n    gl_ready = facts["nvidia_32bit_gl"] or is_64bit\n    if preference == "dxvk":\n        if not dxvk_ready:\n            raise RuntimeError("DXVK requires a working Vulkan X11 swapchain, 32-bit loader, and both DLL sets")\n        return "dxvk"\n    if preference == "auto" and dxvk_ready:\n        return "dxvk"\n    if facts["direct_gl"] and gl_ready:\n        return "direct_gl"\n    if facts["virtualgl"] and gl_ready:\n        return "virtualgl"\n    if preference == "wined3d":\n        if facts["virtualgl"] and gl_ready:\n            return "virtualgl"\n        raise RuntimeError("WineD3D requires verified NVIDIA GL and 32-bit GL libraries")\n    return "software"\n\n\ndef launch(app, wine):\n    facts = diagnose()\n    env = clean_env()\n    direct_xorg = (env.get("DISPLAY", "").split(".", 1)[0] == ":1" and\n                   facts["direct_gl"] and\n                   os.environ.get("GAMELAB_DIRECT_XORG") == "1")\n    env["WINEPREFIX"] = os.environ.get("WINEPREFIX", "/home/gamer/.wine")\n    env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n    env["SDL_AUDIODRIVER"] = "pulse"\n    env["HOME"] = os.environ.get("KAGGLE_GAMER_HOME", "/home/gamer")\n    env["__GL_SYNC_TO_VBLANK"] = "0"\n\n    # Dedicate GPU 0 exclusively to 3D rendering and physics simulation\n    env["CUDA_VISIBLE_DEVICES"] = "0"\n    env["DRI_PRIME"] = "0"\n    if not direct_xorg:\n        env["VGL_DISPLAY"] = "egl0"\n    env["DXVK_FILTER_DEVICE_NAME"] = "Tesla T4"\n    env["WINE_FULLSCREEN_FSR"] = "1"\n    env["WINE_FULLSCREEN_FSR_STRENGTH"] = "2"\n    env["VKD3D_CONFIG"] = "dxr11,dxr"\n    env["VKD3D_FEATURE_LEVEL"] = "12_1"\n\n    # Guarantee full OpenGL 4.5 Compatibility Profile & WGL extensions for Wine/idTech\n    if not direct_xorg:\n        env["MESA_GL_VERSION_OVERRIDE"] = "4.5COMPAT"\n        env["MESA_GLSL_VERSION_OVERRIDE"] = "450"\n        env["__NV_PRIME_RENDER_OFFLOAD"] = "1"\n        env["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"\n    if not env.get("DISPLAY"):\n        env["DISPLAY"] = ":0"\n\n    # Propagate hardware profile to Legendary configuration\n    try:\n        leg_cfg_path = Path(env["HOME"]) / ".config/legendary/config.ini"\n        leg_cfg_path.parent.mkdir(parents=True, exist_ok=True)\n        current_cfg = leg_cfg_path.read_text(encoding="utf-8") if leg_cfg_path.is_file() else ""\n        if "[default.env]" not in current_cfg:\n            with leg_cfg_path.open("a", encoding="utf-8") as f:\n                f.write("\\n[default.env]\\nVGL_DISPLAY = egl0\\nMESA_GL_VERSION_OVERRIDE = 4.5COMPAT\\nMESA_GLSL_VERSION_OVERRIDE = 450\\n__NV_PRIME_RENDER_OFFLOAD = 1\\n__GLX_VENDOR_LIBRARY_NAME = nvidia\\n")\n    except Exception:\n        pass\n\n    if not env.get("PULSE_SERVER"):\n        for candidate in ("/run/user/1000/pulse/native", "/tmp/pulse-PKdhtXMmr18n/native"):\n            if Path(candidate).exists():\n                env["PULSE_SERVER"] = f"unix:{candidate}"\n                break\n        if not env.get("PULSE_SERVER"):\n            try:\n                found = list(Path("/tmp").glob("pulse-*/native")) + list(Path("/run/user").glob("*/pulse/native"))\n                if found:\n                    env["PULSE_SERVER"] = f"unix:{found[0]}"\n            except Exception:\n                pass\n\n    wine_bin = wine or shutil.which("wine")\n    configure_mouse(wine_bin, env)\n\n    # Expose the virtual controller only in gamepad mode. In mouse mode Wine\n    # otherwise sees an Xbox controller even though the HUD says M+K.\n    input_mode = get_input_mode()\n    env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n    if input_mode == "mouse_keyboard":\n        env.pop("SELKIES_JS_SOCKET_PATH", None)\n        env.pop("LD_PRELOAD", None)\n        env["SDL_JOYSTICK_DEVICE"] = ""\n        env["WINEDLLOVERRIDES"] += ";winebus.sys=d"\n    else:\n        env["SELKIES_JS_SOCKET_PATH"] = ("/tmp/kaggle-game-dev-bench/direct-stream-sockets"\n                                         if direct_xorg else "/tmp")\n        env["SDL_JOYSTICK_DEVICE"] = "/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003"\n        interposer = "/usr/$LIB/selkies_input_interposer.so:/usr/$LIB/selkies_fake_udev.so"\n        env["LD_PRELOAD"] = (interposer + ":" + env.get("LD_PRELOAD", "")).rstrip(":")\n\n    target_bin = find_game_binary(app)\n    is_64bit = True\n    if target_bin and "32" in target_bin.name.lower() and "64" not in target_bin.name.lower():\n        is_64bit = False\n    launch_cwd = str(target_bin.parent) if target_bin else None\n\n    # Switch to 64-bit Wine binary for 64-bit binaries to prevent 32-bit wrapper dropping 64-bit VGL faker\n    if is_64bit and wine_bin:\n        wine64_candidate = Path(wine_bin).with_name("wine64")\n        if wine64_candidate.is_file():\n            wine_bin = str(wine64_candidate)\n\n    vgl = shutil.which("vglrun") or "/opt/VirtualGL/bin/vglrun"\n    vgl_ready = (not direct_xorg and Path(vgl).is_file() and\n                 (facts.get("virtualgl") or facts.get("egl_device")))\n    vgl_display = facts.get("virtualgl_display") or facts.get("egl_device_display") or "egl0"\n\n    # vglrun enables its dlopen interposer by default.  -dl is not a valid\n    # option in current VirtualGL and is interpreted as the executable.\n    wrap_script = Path("/tmp/kaggle-game-dev-bench/vgl-wrap.sh")\n    try:\n        wrap_script.write_text(f"#!/bin/bash\\nexec {vgl} -d {vgl_display} \\"$@\\"\\n", encoding="utf-8")\n        wrap_script.chmod(0o755)\n    except Exception:\n        pass\n\n    if vgl_ready:\n        env["VGL_DISPLAY"] = vgl_display\n        env.update(VGL_READBACK="sync", VGL_SPOIL="0", VGL_SYNC="1", VGL_COMPRESS="proxy")\n        vgl_fps = os.environ.get("GAMELAB_VGL_FPS", "60")\n        try:\n            if 0 <= float(vgl_fps) <= 240:\n                env["VGL_FPS"] = vgl_fps\n        except ValueError:\n            pass\n\n    if target_bin and wine_bin:\n        print(f"Target game binary detected: {target_bin}", flush=True)\n        try:\n            configure_mouse(str(wine_bin), env)\n        except Exception:\n            pass\n        command = [wine_bin, str(target_bin)]\n        if "wolf" in target_bin.name.lower() or "idtech" in target_bin.name.lower():\n            command.extend([\n                "+set", "com_skipIntroVideo", "1",\n                "+set", "com_skipBootSequence", "1",\n                "+set", "r_fullscreen", "0",\n                "+set", "r_mode", "-1",\n                "+set", "r_windowedWidth", "1280",\n                "+set", "r_windowedHeight", "720",\n                "+set", "vt_cuda", "0",\n                "+set", "vt_maxPPF", "16"\n            ])\n    else:\n        command = ["legendary", "launch", app, "--wine", wine]\n        if vgl_ready:\n            command.extend(["--wrapper", str(wrap_script)])\n\n    # Prepend vglrun so the process tree inherits its OpenGL interposers.\n    if vgl_ready:\n        command = [vgl, "-d", vgl_display, *command]\n\n    try:\n        selected = select_graphics_backend(\n            facts, os.environ.get("GAMELAB_GRAPHICS_BACKEND", "auto"), is_64bit=is_64bit)\n    except (ValueError, RuntimeError) as exc:\n        print(f"Graphics backend unavailable: {exc}", file=sys.stderr)\n        return 78\n    if selected == "dxvk":\n        try:\n            install_dxvk(env["WINEPREFIX"])\n        except (OSError, RuntimeError) as exc:\n            print(f"DXVK setup failed: {exc}", file=sys.stderr)\n            return 78\n        try:\n            install_vkd3d(env["WINEPREFIX"])\n        except Exception as exc:\n            print(f"VKD3D setup notice: {exc}", file=sys.stderr)\n        env["WINEDLLOVERRIDES"] = "d3d12,d3d12core,d3d11,dxgi,d3d9,d3d10core=n,b;mscoree,mshtml=d"\n        env["DXVK_HUD"] = "devinfo,fps"\n        backend = "NVIDIA Vulkan through DXVK & VKD3D + VirtualGL (GPU 0)"\n    elif selected == "direct_gl":\n        backend = "NVIDIA OpenGL"\n    elif selected == "virtualgl":\n        display = facts.get("virtualgl_display") or "egl0"\n        backend = f"NVIDIA OpenGL through VirtualGL ({display})"\n    elif os.environ.get("GAMELAB_ALLOW_SOFTWARE") == "1":\n        backend = "Mesa software (explicit opt-in)"\n        env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n        env["LIBGL_ALWAYS_SOFTWARE"] = "1"\n        env["GALLIUM_DRIVER"] = "llvmpipe"\n        env["LP_NUM_THREADS"] = str(software_threads())\n        if shutil.which("nice"):\n            command = ["nice", "-n", "5", *command]\n    else:\n        backend = "Mesa software (fallback)"\n        env["WINEDLLOVERRIDES"] = "d3d11,dxgi,d3d9,d3d10core=b;mscoree,mshtml=d"\n        env["LIBGL_ALWAYS_SOFTWARE"] = "1"\n        env["GALLIUM_DRIVER"] = "llvmpipe"\n        env["LP_NUM_THREADS"] = str(software_threads())\n        if shutil.which("nice"):\n            command = ["nice", "-n", "5", *command]\n    print(f"Rendering backend: {backend}", flush=True)\n    if backend.startswith("Mesa software"):\n        print(f"llvmpipe worker threads: {env[\'LP_NUM_THREADS\']}", flush=True)\n        previous_resolution = lower_software_resolution(env)\n        try:\n            return subprocess.call(command, env=env, cwd=launch_cwd)\n        finally:\n            if previous_resolution:\n                run(["xrandr", "-s", previous_resolution], env, timeout=5)\n    return subprocess.call(command, env=env, cwd=launch_cwd)\n\n\nif __name__ == "__main__":\n    if len(sys.argv) == 2 and sys.argv[1] == "diagnose":\n        print(json.dumps(diagnose(), indent=2))\n    elif len(sys.argv) == 4 and sys.argv[1] == "launch":\n        sys.exit(launch(sys.argv[2], sys.argv[3]))\n    else:\n        print("Usage: gpu_runtime.py diagnose | launch APP WINE_BIN", file=sys.stderr)\n        sys.exit(2)\n', encoding='utf-8')
(BENCH / 'launch_idtech5.py').write_text('"""Launch a 64-bit id Tech OpenGL game on the Selkies X desktop.\n\nThe GLX extension must exist on the captured X server.  VirtualGL sends\nOpenGL work to the selected NVIDIA GLX server or EGL device and copies\ncompleted frames into the Wine window on DISPLAY. Selkies captures :0.\n"""\n\nimport argparse\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\nfrom gpu_runtime import configure_mouse, get_input_mode\n\n\nVGL = Path("/opt/VirtualGL/bin/vglrun")\n\n\ndef probe(args, env, timeout=15):\n    result = subprocess.run(args, env=env, text=True, errors="replace",\n                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                            timeout=timeout, check=False)\n    return result.returncode, result.stdout\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("game", type=Path,\n                        help="Path to a 64-bit Windows OpenGL game executable")\n    parser.add_argument("--wine", type=Path, required=True,\n                        help="Path to Wine-GE\'s bin/wine or bin/wine64")\n    parser.add_argument("--vgl-display", default="auto",\n                        help="VirtualGL target: auto, :1, or egl0 (default: auto)")\n    parser.add_argument("--direct-xorg", action="store_true",\n                        help="Run directly on the NVIDIA Xorg display :1 (use its Selkies stream)")\n    parser.add_argument("--egl-device", default=None,\n                        help="Deprecated alias for --vgl-display")\n    parser.add_argument("--prefix", type=Path,\n                        default=Path("/home/gamer/.wine"))\n    parser.add_argument("game_args", nargs=argparse.REMAINDER)\n    args = parser.parse_args()\n\n    game = args.game.resolve()\n    wine = args.wine.resolve()\n    if not game.is_file() or not wine.is_file() or (not args.direct_xorg and not VGL.is_file()):\n        parser.error("game, Wine binary, and selected graphics launcher must exist")\n\n    env = os.environ.copy()\n    env["DISPLAY"] = ":1" if args.direct_xorg else (env.get("DISPLAY") or ":0")\n    env["WINEPREFIX"] = str(args.prefix)\n    env["VGL_COMPRESS"] = "proxy"\n    env["SDL_AUDIODRIVER"] = "pulse"\n    overrides = env.get("WINEDLLOVERRIDES", "")\n    env["WINEDLLOVERRIDES"] = (overrides + ";" if overrides else "") + "opengl32=b"\n    # These variables cannot add GLX to Xvfb or select the VirtualGL GPU.\n    for key in ("LIBGL_ALWAYS_SOFTWARE", "GALLIUM_DRIVER",\n                "MESA_LOADER_DRIVER_OVERRIDE", "MESA_GL_VERSION_OVERRIDE",\n                "MESA_GLSL_VERSION_OVERRIDE", "__NV_PRIME_RENDER_OFFLOAD",\n                "__GLX_VENDOR_LIBRARY_NAME"):\n        env.pop(key, None)\n\n    if args.direct_xorg:\n        # A direct GLX context and the captured window must live on the same\n        # NVIDIA X server. VirtualGL\'s cross-server drawable mapping is not\n        # reliable for this game\'s multi-context WGL path.\n        env.pop("VGL_DISPLAY", None)\n        for key in ("VGL_SYNC", "VGL_READBACK", "VGL_LOGO", "VGL_PROFILE"):\n            env.pop(key, None)\n        input_mode = get_input_mode()\n        env["SDL_JOYSTICK_DISABLE_UDEV"] = "1"\n        if input_mode == "mouse_keyboard":\n            # Wine must not enumerate Selkies\' virtual Xbox devices in FPS\n            # mouse mode. X11/XTEST still carries pointer and keyboard input.\n            env.pop("LD_PRELOAD", None)\n            env.pop("SELKIES_JS_SOCKET_PATH", None)\n            env["SDL_JOYSTICK_DEVICE"] = ""\n            env["WINEDLLOVERRIDES"] += ";winebus.sys=d"\n        else:\n            socket_dir = Path("/tmp/kaggle-game-dev-bench/direct-stream-sockets")\n            candidates = [\n                "/usr/$LIB/selkies_input_interposer.so:/usr/$LIB/selkies_fake_udev.so",\n                "/tmp/kaggle-game-dev-bench/lib64/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/lib64/selkies_fake_udev.so",\n                "/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so"\n            ]\n            interposer = next((c for c in candidates if Path(c.split(":")[0].replace("$LIB", "lib64")).is_file() or Path(c.split(":")[0].replace("$LIB", "lib/x86_64-linux-gnu")).is_file()), candidates[0])\n            env["SELKIES_JS_SOCKET_PATH"] = str(socket_dir) if socket_dir.is_dir() else "/tmp"\n            env["SDL_JOYSTICK_DEVICE"] = "/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003"\n            env["LD_PRELOAD"] = (interposer + ":" + env.get("LD_PRELOAD", "")).rstrip(":")\n        try:\n            rc, info = probe(["glxinfo", "-B"], env)\n            if rc or not re.search(r"OpenGL vendor string:\\s*NVIDIA", info):\n                raise RuntimeError("NVIDIA GLX is unavailable on :1: " + info[-500:])\n        except (OSError, subprocess.TimeoutExpired, RuntimeError) as exc:\n            print(f"Graphics prerequisite failed: {exc}", file=sys.stderr)\n            return 78\n        wine_bin = wine.with_name("wine64") if wine.with_name("wine64").is_file() else wine\n        configure_mouse(str(wine_bin), env)\n        command = [str(wine_bin), str(game)]\n        command.extend(args.game_args or\n                       ["+set", "com_skipIntroVideo", "1",\n                        "+set", "com_skipBootSequence", "1",\n                        "+set", "r_fullscreen", "0", "+set", "r_mode", "-1",\n                        "+set", "r_windowedWidth", "1280",\n                        "+set", "r_windowedHeight", "720",\n                        "+set", "in_mouse", "1"])\n        print("OpenGL target: direct NVIDIA Xorg :1", flush=True)\n        print("Input mode:", input_mode, flush=True)\n        print("Stream this display using start_direct_stream.sh", flush=True)\n        return subprocess.call(command, cwd=game.parent, env=env)\n\n    try:\n        rc, extensions = probe(["xdpyinfo", "-queryExtensions"], env)\n        if rc or not re.search(r"(?m)^\\s*GLX\\s+\\(opcode:", extensions):\n            raise RuntimeError(\n                f"{env[\'DISPLAY\']} does not advertise GLX. Wine\'s winex11.drv "\n                "will disable WGL. Restart the captured Xvfb with GLX enabled "\n                "or use a GLX-capable captured X server."\n            )\n\n        selected = args.egl_device or args.vgl_display\n        if selected == "auto":\n            candidates = ([":1"] if Path("/tmp/kaggle-game-dev-bench/gpu-xorg-ready").exists()\n                          else []) + ["egl0"]\n        else:\n            candidates = [selected]\n        failures = []\n        for candidate in candidates:\n            rc, info = probe([str(VGL), "-d", candidate, "glxinfo", "-B"], env)\n            if (not rc and re.search(r"OpenGL vendor string:\\s*NVIDIA", info)\n                    and not re.search(r"OpenGL renderer string:.*(?:llvmpipe|softpipe)", info, re.I)):\n                selected = candidate\n                break\n            failures.append(f"{candidate}: {info[-500:]}")\n        else:\n            raise RuntimeError("VirtualGL NVIDIA probe failed:\\n" + "\\n".join(failures))\n    except (OSError, subprocess.TimeoutExpired, RuntimeError) as exc:\n        print(f"Graphics prerequisite failed: {exc}", file=sys.stderr)\n        return 78\n\n    # vglrun already preloads libdlfaker and libvglfaker by default.\n    wine_bin = wine.with_name("wine64") if wine.with_name("wine64").is_file() else wine\n    env["VGL_DISPLAY"] = selected\n    command = [str(VGL), "-d", selected, str(wine_bin), str(game)]\n    if not args.game_args:\n        command.extend(["-window", "+com_skipIntroVideo", "1", "+r_fullscreen", "0"])\n    else:\n        command.extend(args.game_args)\n    print("OpenGL target:", selected, flush=True)\n    print("Wine window:", env["DISPLAY"], flush=True)\n    return subprocess.call(command, cwd=game.parent, env=env)\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', encoding='utf-8')
(BENCH / 'start_direct_stream.sh').write_text('#!/bin/bash\n# Stream the NVIDIA Xorg display directly. Run as root after the notebook\'s\n# ordinary desktop session has created PulseAudio and gpu_xorg.sh has run.\nset -euo pipefail\nBENCH=/tmp/kaggle-game-dev-bench\nPORT="${DIRECT_STREAM_PORT:-8080}"\nWIDTH="${DIRECT_STREAM_WIDTH:-1280}"\nHEIGHT="${DIRECT_STREAM_HEIGHT:-720}"\nORIGIN="${SELKIES_ALLOWED_ORIGINS:-https://kkb-production.jupyter-proxy.kaggle.net}"\nif [ "$(id -u)" -ne 0 ]; then\n    echo \'This Xorg installation requires root for Selkies MIT-SHM capture.\' >&2\n    exit 1\nfi\nbash "$BENCH/gpu_xorg.sh"\ntest -e "$BENCH/gpu-xorg-ready"\nDISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" \\\n    glxinfo -B | grep -q \'OpenGL vendor string: NVIDIA\'\nif command -v xrandr >/dev/null; then\n    DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" \\\n        xrandr -s "${WIDTH}x${HEIGHT}" 2>/dev/null || true\nfi\n\n# Use one stable, gamer-owned PulseAudio runtime for Wine and Selkies. A\n# socket pathname alone is not proof of a running or authenticated server.\nAUDIO_RUNTIME="$BENCH/pulse-runtime"\nAUDIO_COOKIE=/home/gamer/.config/pulse/cookie\nPULSE_SERVER="unix:$AUDIO_RUNTIME/pulse/native"\ninstall -d -o gamer -g gamer -m 700 /home/gamer/.config/pulse "$AUDIO_RUNTIME" "$AUDIO_RUNTIME/pulse"\nif [ -f "$AUDIO_COOKIE" ]; then\n    chown gamer:gamer "$AUDIO_COOKIE"\n    chmod 600 "$AUDIO_COOKIE"\nfi\npulse_user() {\n    runuser -u gamer -- env HOME=/home/gamer XDG_RUNTIME_DIR="$AUDIO_RUNTIME" \\\n        PULSE_SERVER="$PULSE_SERVER" PULSE_COOKIE="$AUDIO_COOKIE" "$@"\n}\nif ! pulse_user pactl info >/dev/null 2>&1; then\n    runuser -u gamer -- env HOME=/home/gamer XDG_RUNTIME_DIR="$AUDIO_RUNTIME" \\\n        PULSE_COOKIE="$AUDIO_COOKIE" \\\n        pulseaudio --daemonize=yes --exit-idle-time=-1 --disallow-exit=yes \\\n        >> "$BENCH/logs/pulseaudio.log" 2>&1 || true\nfi\naudio_ready=false\nfor attempt in $(seq 1 20); do\n    if [ -S "$AUDIO_RUNTIME/pulse/native" ] && pulse_user pactl info >/dev/null 2>&1; then\n        audio_ready=true\n        break\n    fi\n    sleep 0.5\ndone\nif [ "$audio_ready" = true ]; then\n    if ! pulse_user pactl list short sinks | awk \'$2 == "output" { found=1 } END { exit !found }\'; then\n        pulse_user pactl load-module module-null-sink sink_name=output \\\n            sink_properties=device.description=GameAudio >/dev/null || true\n    fi\n    pulse_user pactl set-default-sink output || true\n    pulse_user pactl set-sink-mute output 0 || true\n    pulse_user pactl set-sink-volume output 100% || true\n    if ! PULSE_SERVER="$PULSE_SERVER" PULSE_COOKIE="$AUDIO_COOKIE" pactl info >/dev/null 2>&1; then\n        echo \'Selkies cannot authenticate to the gamer PulseAudio server.\' >&2\n    fi\nelse\n    echo "PulseAudio did not start; see $BENCH/logs/pulseaudio.log" >&2\nfi\nprintf \'%s\' "$PULSE_SERVER" > "$BENCH/pulse-server-path"\nchmod 644 "$BENCH/pulse-server-path"\nexport DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" PULSE_SERVER PULSE_COOKIE="$AUDIO_COOKIE"\nexport PULSE_SINK=output\nENCODER_DEVICE=0\nif [ -e /dev/nvidia1 ]; then ENCODER_DEVICE=1; fi\nexport CUDA_VISIBLE_DEVICES="$ENCODER_DEVICE"\nexport SELKIES_ALLOWED_ORIGINS="$ORIGIN"\nexport SELKIES_JS_SOCKET_PATH="$BENCH/direct-stream-sockets"\nmkdir -p "$SELKIES_JS_SOCKET_PATH" "$BENCH/logs"\ntest -f "$BENCH/web-root/index.html"\n# Keep the system Openbox settings, with a rule for the game\'s Wine window.\n# EWMH fullscreen removes both the title bar and tint2\'s reserved screen area.\npython3 - "$BENCH/direct-openbox-rc.xml" <<\'PY\'\nfrom pathlib import Path\nimport sys\n\nsource = Path(\'/etc/xdg/openbox/rc.xml\').read_text(encoding=\'utf-8\')\nrule = \'\'\'<applications>\n  <application class="wolfneworder_x64.exe">\n    <decor>no</decor>\n    <fullscreen>yes</fullscreen>\n    <focus>yes</focus>\n  </application>\n  <application class="*dev_launcher*">\n    <decor>no</decor>\n    <position force="yes"><x>0</x><y>0</y></position>\n    <focus>yes</focus>\n  </application>\n  <application title="*7BZ Cloud Gaming*">\n    <decor>no</decor>\n    <position force="yes"><x>0</x><y>0</y></position>\n    <focus>yes</focus>\n  </application>\'\'\'\nif source.count(\'<applications>\') != 1:\n    raise SystemExit(\'Cannot locate Openbox application rules\')\nPath(sys.argv[1]).write_text(source.replace(\'<applications>\', rule, 1), encoding=\'utf-8\')\nPY\nstart_desktop_program() {\n    local label="$1"\n    shift\n    local pidfile="$BENCH/direct-${label}.pid"\n    if [ -f "$pidfile" ] && kill -0 "$(cat "$pidfile")" 2>/dev/null; then\n        return\n    fi\n    runuser -u gamer -- env DISPLAY=:1 HOME=/home/gamer \\\n        XDG_RUNTIME_DIR="$AUDIO_RUNTIME" PULSE_SERVER="$PULSE_SERVER" \\\n        PULSE_COOKIE="$AUDIO_COOKIE" PULSE_SINK=output SDL_AUDIODRIVER=pulse \\\n        CUDA_VISIBLE_DEVICES=0 GAMELAB_DIRECT_XORG=1 \\\n        SELKIES_JS_SOCKET_PATH="$SELKIES_JS_SOCKET_PATH" \\\n        PYTHONPATH=/root/.local/lib/python3.13/site-packages:/usr/local/lib/python3.13/dist-packages \\\n        "$@" > "$BENCH/logs/direct-${label}.log" 2>&1 < /dev/null &\n    echo $! > "$pidfile"\n}\nstart_desktop() {\n    chmod 666 "$SELKIES_JS_SOCKET_PATH"/*.sock 2>/dev/null || true\n    # The HUD\'s input-mode switch and diagnostics live on this control port.\n    # The fallback desktop starts it separately, but direct Xorg must too.\n    start_desktop_program control python3 "$BENCH/session_control.py"\n    start_desktop_program openbox openbox --config-file "$BENCH/direct-openbox-rc.xml"\n    if command -v tint2 >/dev/null; then\n        start_desktop_program tint2 tint2 -c "$BENCH/tint2rc"\n    fi\n    start_desktop_program dashboard python3 "$BENCH/dev_launcher.py"\n    start_desktop_program legendary python3 "$BENCH/legendary_gui.py"\n}\nif curl -fsS "http://127.0.0.1:${PORT}/api/health" >/dev/null 2>&1; then\n    start_desktop\n    echo "Direct desktop already streaming on port $PORT"\n    exit 0\nfi\n# This process runs as root because Xorg rejected the gamer\'s MIT-SHM attach.\n# Disable command execution and transfers on this privileged stream.\nnohup env SELKIES_ENCODER_DEVICE="$ENCODER_DEVICE" SELKIES_ENCODER=h264enc \\\n    SELKIES_AUDIO_ENABLED=true SELKIES_AUDIO_ON_START=true \\\n    SELKIES_AUDIO_DEVICE_NAME=output.monitor \\\n    python3 -m selkies --port="$PORT" --enable-basic-auth=false \\\n    --framerate=60-60 --video-bitrate=3000,800-6000 \\\n    --rate-control-mode=cbr --congestion-control=true \\\n    --backpressure-queue-size=4 \\\n    --command-enabled=false --file-transfers=none --printing-enabled=false \\\n    --web-root="$BENCH/web-root" --ui-title=\'7BZ Dev Cloud Game\' \\\n    --ui-show-sidebar=true --ui-show-logo=false \\\n    --ui-sidebar-show-video-settings=false \\\n    --ui-sidebar-show-screen-settings=false \\\n    --manual-resolution=true --manual-width="$WIDTH" --manual-height="$HEIGHT" \\\n    > "$BENCH/logs/direct-stream.log" 2>&1 < /dev/null &\necho $! > "$BENCH/direct-stream.pid"\nfor attempt in $(seq 1 20); do\n    if curl -fsS "http://127.0.0.1:${PORT}/api/health" >/dev/null 2>&1; then\n        start_desktop\n        echo "Direct NVIDIA stream ready on port $PORT"\n        exit 0\n    fi\n    sleep 0.5\ndone\necho "Direct stream did not start; see $BENCH/logs/direct-stream.log" >&2\nexit 1\n', encoding='utf-8')
(BENCH / 'bench_runtime.py').write_text('"""Sample a live game session without confusing stream FPS with game FPS.\n\nRun after entering repeatable gameplay. Save one JSON report per configuration.\nThe HUD\'s stream FPS and NVIDIA encoder usage are deliberately not game FPS.\n"""\nimport argparse\nfrom datetime import datetime, timezone\nimport json\nimport os\nfrom pathlib import Path\nimport statistics\nimport subprocess\nimport time\n\n\ndef cpu_stat():\n    path = Path("/sys/fs/cgroup/cpu.stat")\n    try:\n        return {key: int(value) for key, value in\n                (line.split() for line in path.read_text().splitlines())}\n    except (OSError, ValueError):\n        return {}\n\n\ndef gpus():\n    try:\n        output = subprocess.check_output([\n            "nvidia-smi", "--query-gpu=index,utilization.gpu,utilization.encoder,memory.used",\n            "--format=csv,noheader,nounits"], text=True, timeout=5,\n            stderr=subprocess.DEVNULL)\n    except (OSError, subprocess.SubprocessError):\n        return []\n    result = []\n    for line in output.splitlines():\n        fields = [part.strip() for part in line.split(",")]\n        if len(fields) != 4:\n            continue\n        try:\n            result.append(dict(index=int(fields[0]), gpu_percent=int(fields[1]),\n                               encoder_percent=int(fields[2]), memory_mib=int(fields[3])))\n        except ValueError:\n            continue\n    return result\n\n\ndef summarize(samples):\n    if not samples:\n        return None\n    return dict(mean=round(statistics.mean(samples), 2),\n                maximum=max(samples))\n\n\ndef benchmark(seconds, interval):\n    start = time.monotonic()\n    previous_time = start\n    previous_cpu = cpu_stat()\n    cpu_cores = []\n    throttled_ms_per_second = []\n    gpu_samples = {}\n    while True:\n        remaining = seconds - (time.monotonic() - start)\n        if remaining <= 0:\n            break\n        time.sleep(min(interval, remaining))\n        now = time.monotonic()\n        current_cpu = cpu_stat()\n        elapsed = now - previous_time\n        if "usage_usec" in current_cpu and "usage_usec" in previous_cpu:\n            cpu_cores.append((current_cpu["usage_usec"] - previous_cpu["usage_usec"])\n                             / (elapsed * 1_000_000))\n        if "throttled_usec" in current_cpu and "throttled_usec" in previous_cpu:\n            throttled_ms_per_second.append(\n                (current_cpu["throttled_usec"] - previous_cpu["throttled_usec"])\n                / (elapsed * 1000))\n        previous_cpu, previous_time = current_cpu, now\n        for gpu in gpus():\n            bucket = gpu_samples.setdefault(gpu["index"], dict(gpu_percent=[],\n                                    encoder_percent=[], memory_mib=[]))\n            for metric in ("gpu_percent", "encoder_percent", "memory_mib"):\n                bucket[metric].append(gpu[metric])\n    return dict(cpu_cores=summarize(cpu_cores),\n                cpu_throttled_ms_per_second=summarize(throttled_ms_per_second),\n                gpus={index: {name: summarize(values) for name, values in metrics.items()}\n                      for index, metrics in gpu_samples.items()})\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--label", required=True, help="game/backend/settings for this run")\n    parser.add_argument("--seconds", type=int, default=30)\n    parser.add_argument("--interval", type=float, default=1)\n    parser.add_argument("--game-fps", type=float,\n                        help="measured in-game FPS, never Selkies stream FPS")\n    parser.add_argument("--input-latency-ms", type=float,\n                        help="client HUD latency measured during this run")\n    parser.add_argument("--output", type=Path)\n    args = parser.parse_args()\n    if args.seconds < 1 or args.interval <= 0:\n        parser.error("seconds and interval must be positive")\n    report = dict(timestamp=datetime.now(timezone.utc).isoformat(), label=args.label,\n                  duration_seconds=args.seconds,\n                  graphics_backend=os.environ.get("GAMELAB_GRAPHICS_BACKEND", "auto"),\n                  vgl_fps=os.environ.get("GAMELAB_VGL_FPS", "60"),\n                  websocket_queue=os.environ.get("GAMELAB_WS_QUEUE", "12"),\n                  game_fps=args.game_fps, input_latency_ms=args.input_latency_ms,\n                  measurements=benchmark(args.seconds, args.interval))\n    encoded = json.dumps(report, indent=2)\n    if args.output:\n        args.output.parent.mkdir(parents=True, exist_ok=True)\n        args.output.write_text(encoded + "\\n", encoding="utf-8")\n    print(encoded)\n', encoding='utf-8')
(BENCH / 'gpu_xorg.sh').write_text('#!/bin/bash\n# Optional NVIDIA 3D server for VirtualGL. Selkies keeps its Xvfb desktop.\nset -u\nBENCH=/tmp/kaggle-game-dev-bench\nREADY="$BENCH/gpu-xorg-ready"\nLOG="$BENCH/logs/gpu-xorg.log"\nrm -f "$READY"\nmkdir -p "$BENCH/logs"\n\nif [ ! -c /dev/nvidia-modeset ]; then\n    mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || nvidia-modprobe -m 2>/dev/null || true\nfi\nif [ ! -c /dev/nvidiactl ] || [ ! -c /dev/nvidia0 ] || [ ! -c /dev/nvidia-modeset ]; then\n    echo \'NVIDIA display devices were not mounted by the container host.\' | tee -a "$LOG"\n    exit 0\nfi\nmkdir -p /usr/lib/xorg/modules/drivers /usr/lib/xorg/modules/extensions 2>/dev/null || true\nif [ -d /usr/local/nvidia/lib64/xorg/modules ]; then\n    cp -fL /usr/local/nvidia/lib64/xorg/modules/drivers/nvidia_drv.so /usr/lib/xorg/modules/drivers/ 2>/dev/null || true\n    cp -fL /usr/local/nvidia/lib64/xorg/modules/extensions/libglxserver_nvidia.so* /usr/lib/xorg/modules/extensions/ 2>/dev/null || true\nfi\nfor drv in $(find /usr/local/nvidia /usr/lib -name nvidia_drv.so 2>/dev/null); do\n    if [ -f "$drv" ] && [ ! -f /usr/lib/xorg/modules/drivers/nvidia_drv.so ]; then\n        cp -a "$drv" /usr/lib/xorg/modules/drivers/ 2>/dev/null || true\n    fi\ndone\nfor glx in $(find /usr/local/nvidia /usr/lib -name \'libglxserver_nvidia.so*\' 2>/dev/null); do\n    if [ -f "$glx" ] && [ ! -f /usr/lib/xorg/modules/extensions/libglxserver_nvidia.so ]; then\n        cp -a "$glx" /usr/lib/xorg/modules/extensions/libglxserver_nvidia.so 2>/dev/null || true\n    fi\ndone\n\nif ! command -v Xorg >/dev/null || ! find /usr/lib /usr/local/nvidia -name nvidia_drv.so -print -quit 2>/dev/null | grep -q .; then\n    echo \'NVIDIA Xorg display driver or Xorg is missing. Host graphics/display capability is required.\' | tee -a "$LOG"\n    exit 1\nfi\n# Identical T4 renderer strings cannot identify the card. Pin this X server\n# to nvidia-smi GPU 0\'s PCI address instead of relying on Xorg probe order.\npci_address=$(nvidia-smi -i 0 --query-gpu=pci.bus_id --format=csv,noheader 2>/dev/null | head -n 1 | tr -d \'[:space:]\')\nif [[ ! "$pci_address" =~ ^([[:xdigit:]]+):([[:xdigit:]]+):([[:xdigit:]]+)\\.([0-7])$ ]]; then\n    echo "Cannot determine GPU 0 PCI bus ID: $pci_address" | tee -a "$LOG"\n    exit 0\nfi\ndomain=$((16#${BASH_REMATCH[1]}))\nbus=$((16#${BASH_REMATCH[2]}))\ndevice=$((16#${BASH_REMATCH[3]}))\nfunc_num=${BASH_REMATCH[4]}\nbus_id="PCI:${bus}:${device}:${func_num}"\nif [ "$domain" -ne 0 ]; then bus_id="PCI:${bus}@${domain}:${device}:${func_num}"; fi\n# Xorg can leave its socket and PID lock behind after a crash. A dead lock\n# owner proves that removing those exact files cannot disconnect a live server.\nif [ -S /tmp/.X11-unix/X1 ] && [ -f /tmp/.X1-lock ]; then\n    lock_pid=$(tr -cd \'0-9\' < /tmp/.X1-lock)\n    if [ -n "$lock_pid" ] && ! kill -0 "$lock_pid" 2>/dev/null; then\n        rm -f -- /tmp/.X11-unix/X1 /tmp/.X1-lock\n        echo \'Removed stale Xorg :1 socket and lock.\' | tee -a "$LOG"\n    fi\nfi\nif [ -S /tmp/.X11-unix/X1 ]; then\n    if grep -Fq "BusID \\"$bus_id\\"" "$BENCH/nvidia-headless.conf" 2>/dev/null && \\\n       DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" glxinfo -B 2>&1 | grep -Eiq \'OpenGL vendor string:.*NVIDIA\'; then\n        DISPLAY=:1 XAUTHORITY="$BENCH/nvidia-Xauthority" xhost +si:localuser:gamer >> "$LOG" 2>&1\n        touch "$READY"\n        echo \'Reusing verified NVIDIA Xorg display :1.\' | tee -a "$LOG"\n    else\n        echo \'Display :1 is occupied by an unverified X server.\' | tee -a "$LOG"\n    fi\n    exit 0\nfi\n\ncat > "$BENCH/nvidia-headless.conf" <<EOF\nSection "ServerLayout"\n    Identifier "HeadlessNVIDIA"\n    Screen 0 "Screen0"\nEndSection\nSection "Device"\n    Identifier "GPU0"\n    Driver "nvidia"\n    BusID "$bus_id"\n    Option "AllowEmptyInitialConfiguration" "true"\n    Option "ConnectedMonitor" "DFP-0"\nEndSection\nSection "Screen"\n    Identifier "Screen0"\n    Device "GPU0"\n    DefaultDepth 24\n    SubSection "Display"\n        Depth 24\n        Modes "1280x720"\n    EndSubSection\nEndSection\nEOF\n\nAUTH="$BENCH/nvidia-Xauthority"\ntouch "$AUTH"\nchmod 600 "$AUTH"\nxauth -f "$AUTH" add :1 . "$(od -An -N16 -tx1 /dev/urandom | tr -d \' \\n\')"\nnohup /usr/lib/xorg/Xorg :1 -config "$BENCH/nvidia-headless.conf" -auth "$AUTH" -nolisten tcp -noreset >> "$LOG" 2>&1 < /dev/null &\nPID=$!\ndisown "$PID" 2>/dev/null || true\nfor attempt in $(seq 1 20); do\n    if [ -S /tmp/.X11-unix/X1 ]; then break; fi\n    if ! kill -0 "$PID" 2>/dev/null; then break; fi\n    sleep 0.5\ndone\nif [ ! -S /tmp/.X11-unix/X1 ]; then\n    echo \'NVIDIA Xorg failed; see gpu-xorg.log.\' | tee -a "$LOG"\n    exit 0\nfi\nDISPLAY=:1 XAUTHORITY="$AUTH" xhost +si:localuser:gamer >> "$LOG" 2>&1\nif DISPLAY=:1 XAUTHORITY="$AUTH" glxinfo -B 2>&1 | grep -Eiq \'OpenGL (vendor|renderer) string:.*(NVIDIA|Tesla)\'; then\n    touch "$READY"\n    echo \'Verified NVIDIA OpenGL on display :1.\' | tee -a "$LOG"\nelse\n    echo \'Display :1 is not NVIDIA accelerated.\' | tee -a "$LOG"\nfi\n', encoding='utf-8')
(BENCH / 'probe_nvidia_userspace.sh').write_text('#!/usr/bin/env bash\n# High-Performance NVIDIA Userspace 3D Driver & Vulkan ICD Provisioner for Kaggle\n# Offloads 3D rendering to NVIDIA GPU hardware (OpenGL & Vulkan) without modifying the kernel.\nset -uo pipefail\n\nbench=/tmp/kaggle-game-dev-bench\nwork="$bench/nvidia-userspace-probe"\nmkdir -p "$work" "$work/lib64" "$work/lib32"\n\necho "=========================================================="\necho "    NVIDIA 3D HARDWARE ACCELERATION PROVISIONER           "\necho "=========================================================="\n\nif ! command -v nvidia-smi >/dev/null 2>&1; then\n    echo "Notice: nvidia-smi not found. Running in CPU-only environment."\n    exit 0\nfi\n\nversion=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | head -n 1 | tr -d \'[:space:]\')\nif [[ ! "$version" =~ ^[0-9]+(\\.[0-9]+)+$ ]]; then\n    echo "Notice: Cannot determine loaded NVIDIA driver version ($version)."\n    exit 0\nfi\n\nmajor=$(echo "$version" | cut -d. -f1)\necho "▶ Host NVIDIA Driver Version detected: $version (Major Branch: $major)"\n\ninstalled_ok=false\n\n# Tier 1: Fast APT package installation (~10 seconds)\necho "▶ Tier 1: Checking official NVIDIA 3D graphics packages for driver $major..."\nif [ "$(id -u)" -eq 0 ]; then\n    apt-get update -o DPkg::Lock::Timeout=60 -qq 2>/dev/null || true\n    for pkg in "libnvidia-gl-$major" "libnvidia-gl-${major}-server"; do\n        if apt-cache show "$pkg" >/dev/null 2>&1; then\n            echo "▶ Found matching repository package: $pkg. Installing userspace libraries..."\n            apt-get install -y --no-install-recommends \\\n                -o DPkg::Lock::Timeout=120 -o APT::Get::Assume-Yes=true \\\n                "$pkg" "${pkg}:i386" vulkan-tools libgl1-mesa-dri mesa-utils 2>/dev/null || \\\n            apt-get install -y --no-install-recommends \\\n                -o DPkg::Lock::Timeout=120 -o APT::Get::Assume-Yes=true \\\n                "$pkg" vulkan-tools libgl1-mesa-dri mesa-utils 2>/dev/null || true\n            break\n        fi\n    done\nfi\n\n# Verify if 64-bit GLX/EGL library is now present in standard paths\nif ldconfig -p | grep -q "libGLX_nvidia.so.0" || [ -f "/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0" ]; then\n    echo "✓ NVIDIA 3D graphics libraries verified via system package manager."\n    installed_ok=true\nfi\n\n# Tier 2: Direct matching NVIDIA archive extraction if APT package was unavailable\nif [ "$installed_ok" = false ]; then\n    echo "▶ Tier 2: Fetching matching NVIDIA userspace binary archive for $version..."\n    runfile="$work/NVIDIA-Linux-x86_64-$version.run"\n    if [ ! -f "$runfile" ] || [ $(stat -c%s "$runfile" 2>/dev/null || echo 0) -lt 10000000 ]; then\n        urls=(\n            "https://us.download.nvidia.com/tesla/$version/NVIDIA-Linux-x86_64-$version.run"\n            "https://us.download.nvidia.com/XFree86/Linux-x86_64/$version/NVIDIA-Linux-x86_64-$version.run"\n            "https://download.nvidia.com/XFree86/Linux-x86_64/$version/NVIDIA-Linux-x86_64-$version.run"\n        )\n        for u in "${urls[@]}"; do\n            echo "Attempting download from: $u"\n            if curl -fL --connect-timeout 10 --max-time 120 "$u" -o "$runfile" 2>/dev/null; then\n                if [ -f "$runfile" ] && [ $(stat -c%s "$runfile") -gt 10000000 ]; then\n                    echo "✓ Downloaded matching NVIDIA installer: $(stat -c%s "$runfile") bytes"\n                    break\n                fi\n            fi\n            rm -f "$runfile"\n        done\n    fi\n\n    driver="$work/NVIDIA-Linux-x86_64-$version"\n    if [ -f "$runfile" ] && [ ! -d "$driver" ]; then\n        echo "▶ Extracting userspace graphics components from archive..."\n        (cd "$work" && sh "$runfile" --extract-only) 2>/dev/null || true\n    fi\n\n    if [ -d "$driver" ]; then\n        link_libraries() {\n            local src=$1 dst=$2\n            mkdir -p "$dst"\n            find "$src" -maxdepth 1 -type f \\( -name \'libnvidia-*.so*\' -o -name \'libGLX_nvidia.so*\' -o -name \'libEGL_nvidia.so*\' \\) -exec cp -sf {} "$dst/" \\; 2>/dev/null || true\n        }\n        link_libraries "$driver" "$work/lib64"\n        [ -d "$driver/32" ] && link_libraries "$driver/32" "$work/lib32"\n\n        for libdir in "$work/lib64" "$work/lib32"; do\n            for name in libGLX_nvidia libEGL_nvidia; do\n                vlib=$(find "$libdir" -name "${name}.so.*" 2>/dev/null | head -n1)\n                [ -n "$vlib" ] && ln -sfn "$(basename "$vlib")" "$libdir/${name}.so.0" 2>/dev/null || true\n            done\n        done\n        installed_ok=true\n    fi\nfi\n\n# Step 3: Register GLVND and Vulkan ICD manifests\nmkdir -p /usr/share/glvnd/egl_vendor.d /etc/vulkan/icd.d /usr/share/vulkan/icd.d "$work"\n\ncat > "$work/10_nvidia.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}\nEOF\n\ncat > "$work/nvidia_icd.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libGLX_nvidia.so.0","api_version":"1.3.0"}}\nEOF\n\ncat > "$work/nvidia_egl_icd.json" <<\'EOF\'\n{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0","api_version":"1.3.0"}}\nEOF\n\ncp -f "$work/10_nvidia.json" /usr/share/glvnd/egl_vendor.d/10_nvidia.json 2>/dev/null || true\ncp -f "$work/nvidia_icd.json" /etc/vulkan/icd.d/nvidia_icd.json 2>/dev/null || true\ncp -f "$work/nvidia_icd.json" /usr/share/vulkan/icd.d/nvidia_icd.json 2>/dev/null || true\n\n# Step 4: Register libraries with dynamic linker\nif [ -d "$work/lib64" ] && [ "$(ls -A "$work/lib64" 2>/dev/null)" ]; then\n    echo "$work/lib64" > /etc/ld.so.conf.d/nvidia-gpu.conf\n    [ -d "$work/lib32" ] && [ "$(ls -A "$work/lib32" 2>/dev/null)" ] && echo "$work/lib32" >> /etc/ld.so.conf.d/nvidia-gpu.conf\nfi\nldconfig 2>/dev/null || true\n\n# Step 5: Write environment exports\n{\n    printf \'export __EGL_VENDOR_LIBRARY_FILENAMES="/usr/share/glvnd/egl_vendor.d/10_nvidia.json"\\n\'\n    printf \'export VK_DRIVER_FILES="/etc/vulkan/icd.d/nvidia_icd.json"\\n\'\n    printf \'export VK_ICD_FILENAMES="/etc/vulkan/icd.d/nvidia_icd.json"\\n\'\n    if [ -d "$work/lib64" ] && [ "$(ls -A "$work/lib64" 2>/dev/null)" ]; then\n        printf \'export LD_LIBRARY_PATH="%s:%s${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}"\\n\' "$work/lib64" "$work/lib32"\n        printf \'export GAMELAB_NVIDIA_LIB32="%s"\\n\' "$work/lib32"\n    fi\n} > "$work/env.sh"\n\necho "✓ NVIDIA userspace graphics bridge successfully configured."\necho "  Vulkan ICD: /etc/vulkan/icd.d/nvidia_icd.json"\necho "  EGL Vendor: /usr/share/glvnd/egl_vendor.d/10_nvidia.json"\n', encoding='utf-8')

legendary_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export HOME="/home/gamer"
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/legendary_gui.log"

for w in /home/gamer/.config/heroic/tools/wine/*/bin/wine; do
    if [ -f "$w" ]; then
        if [ ! -f /usr/bin/wine ]; then
            ln -sf "$w" /usr/bin/wine 2>/dev/null || true
            ln -sf "$(dirname "$w")/wineserver" /usr/bin/wineserver 2>/dev/null || true
            ln -sf "$(dirname "$w")/wineboot" /usr/bin/wineboot 2>/dev/null || true
        fi
        if [ ! -f /usr/local/bin/wine ]; then
            ln -sf "$w" /usr/local/bin/wine 2>/dev/null || true
        fi
        break
    fi
done

if python3 -c "import tkinter" >/dev/null 2>&1; then
    exec python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> "$LOG" 2>&1
else
    if [ "$(id -u)" -eq 0 ]; then
        apt-get update -qq && apt-get install -y -qq python3-tk >> "$LOG" 2>&1 || true
        if python3 -c "import tkinter" >/dev/null 2>&1; then
            exec python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> "$LOG" 2>&1
        fi
    fi
    exec xterm -geometry 105x32 -title "Legendary Epic Games Manager" -e /bin/bash /tmp/kaggle-game-dev-bench/legendary_menu.sh >> "$LOG" 2>&1
fi
"""
(BENCH / 'launch_legendary.sh').write_text(legendary_sh, encoding='utf-8')

# Auto-Optimizer script and launch wrapper
(BENCH / 'auto_optimize.py').write_text('"""Legacy desktop quality tuner for 7BZ Dev Cloud Game."""\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport threading\nimport time\nimport tkinter as tk\nfrom tkinter import ttk, messagebox\nimport urllib.request\n\nHOME = Path(os.environ.get(\'KAGGLE_GAMER_HOME\', \'/home/gamer\'))\nBENCH = Path(\'/tmp/kaggle-game-dev-bench\')\nWINE_ENV_FILE = HOME / \'.config/gamelab_wine_env.sh\'\nSETTINGS_FILE = HOME / \'.config/gamelab_optimize.json\'\n\n# ---------------------------------------------------------------------------\n# Quality presets\n# ---------------------------------------------------------------------------\nPRESETS = {\n    \'low\': {\n        \'label\': \'📶  Low\',\n        \'resolution\': \'854x480\',\n        \'wine_csmt\': False,\n        \'wine_extra\': \'renderer=gl\',\n        \'desc\': \'Best for slow or mobile connections (< 5 Mbps).\\nLower resolution but very smooth streaming.\',\n    },\n    \'medium\': {\n        \'label\': \'📶📶  Medium\',\n        \'resolution\': \'1024x576\',\n        \'wine_csmt\': False,\n        \'wine_extra\': \'renderer=gl\',\n        \'desc\': \'Balanced quality for average connections (5–15 Mbps).\\nGood balance between clarity and smoothness.\',\n    },\n    \'high\': {\n        \'label\': \'📶📶📶  High\',\n        \'resolution\': \'1280x720\',\n        \'wine_csmt\': False,\n        \'wine_extra\': \'\',\n        \'desc\': \'Full 720p HD at 60 FPS (15–30 Mbps).\\nRecommended for broadband connections.\',\n    },\n    \'ultra\': {\n        \'label\': \'📶📶📶📶  Ultra\',\n        \'resolution\': \'1280x720\',\n        \'wine_csmt\': True,\n        \'wine_extra\': \'\',\n        \'desc\': \'Maximum quality with all renderer features (30+ Mbps).\\nRequires fast, stable connection.\',\n    },\n}\nPRESET_ORDER = [\'low\', \'medium\', \'high\', \'ultra\']\n\n# ---------------------------------------------------------------------------\n# Keyboard Lock JavaScript (injected into Selkies HTML)\n# ---------------------------------------------------------------------------\nKEYBOARD_LOCK_JS = """\n<!-- 7BZ Dev Cloud Game: Keyboard Lock – prevents Esc from exiting browser fullscreen -->\n<script>\n(function(){\n  var _kl=false;\n  function _lock(){\n    if(document.fullscreenElement && navigator.keyboard && navigator.keyboard.lock){\n      navigator.keyboard.lock([\'Escape\']).then(function(){_kl=true;\n        console.log(\'[GameLab] Keyboard locked – Esc captured for gaming\');\n      }).catch(function(e){console.log(\'[GameLab] Keyboard lock:\',e.message);});\n    }\n  }\n  document.addEventListener(\'fullscreenchange\',function(){\n    if(document.fullscreenElement){_lock();}\n    else if(_kl && navigator.keyboard && navigator.keyboard.unlock){navigator.keyboard.unlock();_kl=false;}\n  });\n  _lock();\n})();\n</script>\n"""\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef get_current_resolution():\n    try:\n        r = subprocess.run([\'xrandr\', \'--query\'], capture_output=True, text=True, timeout=5)\n        for line in r.stdout.splitlines():\n            if \'*\' in line:\n                return line.strip().split()[0]\n    except Exception:\n        pass\n    return \'1280x720\'\n\n\ndef set_resolution(res):\n    try:\n        w, h = res.split(\'x\')\n        # Try to set directly first\n        r = subprocess.run([\'xrandr\', \'-s\', res], capture_output=True, text=True, timeout=5)\n        if r.returncode != 0:\n            # Create the mode and try again\n            cvt = subprocess.run([\'cvt\', w, h], capture_output=True, text=True, timeout=5)\n            for line in cvt.stdout.splitlines():\n                if \'Modeline\' in line:\n                    parts = line.split(\'"\')\n                    if len(parts) >= 3:\n                        mode_name = parts[1]\n                        mode_args = parts[2].strip()\n                        subprocess.run([\'xrandr\', \'--newmode\', mode_name] + mode_args.split(), check=False, timeout=5)\n                        for output in (\'screen\', \'SCREEN\', \'default\', \'VGA-0\', \'Virtual-1\'):\n                            subprocess.run([\'xrandr\', \'--addmode\', output, mode_name], check=False, timeout=5)\n                        subprocess.run([\'xrandr\', \'-s\', mode_name], check=False, timeout=5)\n        return True\n    except Exception as e:\n        print(f\'Resolution change error: {e}\')\n        return False\n\n\ndef write_wine_env(preset_key):\n    p = PRESETS[preset_key]\n    csmt = \'1\' if p[\'wine_csmt\'] else \'0\'\n    d3d = f"csmt={csmt}"\n    if p.get(\'wine_extra\'):\n        d3d += \';\' + p[\'wine_extra\']\n\n    content = f"""#!/bin/bash\n# Auto-generated by 7BZ Auto-Optimizer – preset: {preset_key}\nexport WINE_D3D_CONFIG="{d3d}"\nexport MESA_GL_VERSION_OVERRIDE="4.3"\nexport __GL_SYNC_TO_VBLANK=0\nexport vblank_mode=0\nexport WINEDLLOVERRIDES="d3d11,dxgi,d3d9,d3d10core=b"\n"""\n    WINE_ENV_FILE.parent.mkdir(parents=True, exist_ok=True)\n    WINE_ENV_FILE.write_text(content, encoding=\'utf-8\')\n\n\ndef save_settings(preset_key):\n    SETTINGS_FILE.parent.mkdir(parents=True, exist_ok=True)\n    SETTINGS_FILE.write_text(json.dumps({\'preset\': preset_key, \'ts\': time.time()}, indent=2), encoding=\'utf-8\')\n\n\ndef load_settings():\n    try:\n        return json.loads(SETTINGS_FILE.read_text(encoding=\'utf-8\')).get(\'preset\', \'high\')\n    except Exception:\n        return \'high\'\n\n\ndef inject_keyboard_lock():\n    """Inject keyboard lock JS into every Selkies HTML file found on disk."""\n    marker = \'7BZ Dev Cloud Game: Keyboard Lock\'\n    patched = []\n    search_roots = [Path(\'/usr/local/lib\'), Path(\'/usr/lib\')]\n    for root in search_roots:\n        if not root.is_dir():\n            continue\n        for html in root.rglob(\'index.html\'):\n            if \'selkies\' not in str(html).lower():\n                continue\n            try:\n                text = html.read_text(encoding=\'utf-8\')\n                if marker in text:\n                    patched.append(f\'{html} (already patched)\')\n                    continue\n                if \'</head>\' in text:\n                    html.write_text(text.replace(\'</head>\', KEYBOARD_LOCK_JS + \'\\n</head>\'), encoding=\'utf-8\')\n                    patched.append(str(html))\n                elif \'</body>\' in text:\n                    html.write_text(text.replace(\'</body>\', KEYBOARD_LOCK_JS + \'\\n</body>\'), encoding=\'utf-8\')\n                    patched.append(str(html))\n            except Exception as e:\n                patched.append(f\'{html} (error: {e})\')\n    return patched\n\n\ndef test_bandwidth():\n    """Quick bandwidth measurement (server-side egress estimate)."""\n    urls = [\n        \'https://speed.cloudflare.com/__down?bytes=2000000\',\n        \'https://www.google.com/images/branding/googlelogo/2x/googlelogo_color_272x92dp.png\',\n    ]\n    for url in urls:\n        try:\n            req = urllib.request.Request(url, headers={\'User-Agent\': \'GameLab/1.0\'})\n            t0 = time.time()\n            with urllib.request.urlopen(req, timeout=8) as resp:\n                data = resp.read()\n            elapsed = time.time() - t0\n            if elapsed > 0.01:\n                return (len(data) * 8) / (elapsed * 1_000_000)\n        except Exception:\n            continue\n    return 0.0\n\n\ndef recommend_preset(speed_mbps):\n    if speed_mbps >= 30:\n        return \'ultra\'\n    elif speed_mbps >= 15:\n        return \'high\'\n    elif speed_mbps >= 5:\n        return \'medium\'\n    return \'low\'\n\n\n# ---------------------------------------------------------------------------\n# GUI\n# ---------------------------------------------------------------------------\n\nclass AutoOptimizer(tk.Tk):\n    def __init__(self):\n        super().__init__()\n        self.title("7BZ Dev Cloud Game · Auto-Optimizer")\n        self.geometry("720x560")\n        self.minsize(640, 480)\n\n        # Dark theme\n        self.c_bg = "#141923"\n        self.c_panel = "#1b2230"\n        self.c_card = "#222c3e"\n        self.c_fg = "#f0f4fc"\n        self.c_muted = "#8fa0bd"\n        self.c_accent = "#3498db"\n        self.c_green = "#2ecc71"\n        self.c_red = "#e74c3c"\n        self.c_yellow = "#f39c12"\n        self.c_orange = "#e67e22"\n\n        self.configure(bg=self.c_bg)\n        self.current_preset = load_settings()\n\n        self.build_header()\n        self.build_status_panel()\n        self.build_presets_panel()\n        self.build_actions_panel()\n        self.build_footer()\n\n        self.refresh_status()\n\n    # ---- Header ----\n    def build_header(self):\n        hdr = tk.Frame(self, bg=self.c_panel)\n        hdr.pack(fill=\'x\', padx=10, pady=(8, 4))\n\n        tk.Label(hdr, text=" ⚡ AUTO-OPTIMIZE ", bg=self.c_orange, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 10, \'bold\'), padx=6, pady=2).pack(side=\'left\', padx=(10, 12), pady=8)\n        tk.Label(hdr, text="Adaptive Streaming & Rendering Quality", bg=self.c_panel, fg=self.c_fg,\n                 font=(\'DejaVu Sans\', 13, \'bold\')).pack(side=\'left\', pady=8)\n\n    # ---- Status panel ----\n    def build_status_panel(self):\n        frm = tk.Frame(self, bg=self.c_card, padx=14, pady=10)\n        frm.pack(fill=\'x\', padx=10, pady=6)\n\n        tk.Label(frm, text="Current Status", bg=self.c_card, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 11, \'bold\')).grid(row=0, column=0, columnspan=4, sticky=\'w\', pady=(0, 6))\n\n        self.lbl_resolution = self._stat(frm, 1, 0, "Resolution:")\n        self.lbl_preset = self._stat(frm, 1, 2, "Active Preset:")\n        self.lbl_wine = self._stat(frm, 2, 0, "Wine Renderer:")\n        self.lbl_keyboard = self._stat(frm, 2, 2, "Esc Key Lock:")\n\n    def _stat(self, parent, row, col, label):\n        tk.Label(parent, text=label, bg=self.c_card, fg=self.c_muted,\n                 font=(\'DejaVu Sans\', 9, \'bold\')).grid(row=row, column=col, sticky=\'w\', padx=(0, 6), pady=2)\n        val = tk.Label(parent, text="...", bg=self.c_card, fg=self.c_fg, font=(\'DejaVu Sans\', 9))\n        val.grid(row=row, column=col + 1, sticky=\'w\', padx=(0, 30), pady=2)\n        return val\n\n    # ---- Presets panel ----\n    def build_presets_panel(self):\n        outer = tk.Frame(self, bg=self.c_bg)\n        outer.pack(fill=\'both\', expand=True, padx=10, pady=6)\n\n        tk.Label(outer, text="Select Quality Preset:", bg=self.c_bg, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 11, \'bold\')).pack(anchor=\'w\', pady=(0, 6))\n\n        self.preset_btns = {}\n        for key in PRESET_ORDER:\n            p = PRESETS[key]\n            card = tk.Frame(outer, bg=self.c_card, padx=12, pady=8, cursor=\'hand2\',\n                            highlightthickness=2, highlightbackground=self.c_card)\n            card.pack(fill=\'x\', pady=3)\n\n            lbl = tk.Label(card, text=p[\'label\'], bg=self.c_card, fg=self.c_fg,\n                           font=(\'DejaVu Sans\', 11, \'bold\'), anchor=\'w\')\n            lbl.pack(side=\'left\', padx=(0, 12))\n\n            desc_short = p[\'desc\'].split(\'\\n\')[0]\n            tk.Label(card, text=desc_short, bg=self.c_card, fg=self.c_muted,\n                     font=(\'DejaVu Sans\', 9), anchor=\'w\').pack(side=\'left\', fill=\'x\', expand=True)\n\n            res_lbl = tk.Label(card, text=p[\'resolution\'], bg=self.c_card, fg=self.c_accent,\n                               font=(\'DejaVu Sans Mono\', 10, \'bold\'))\n            res_lbl.pack(side=\'right\', padx=8)\n\n            self.preset_btns[key] = card\n            for widget in (card, lbl, res_lbl):\n                widget.bind(\'<Button-1>\', lambda e, k=key: self.select_preset(k))\n\n    def select_preset(self, key):\n        self.current_preset = key\n        for k, card in self.preset_btns.items():\n            if k == key:\n                card.configure(highlightbackground=self.c_accent, bg=\'#1a3a52\')\n                for w in card.winfo_children():\n                    w.configure(bg=\'#1a3a52\')\n            else:\n                card.configure(highlightbackground=self.c_card, bg=self.c_card)\n                for w in card.winfo_children():\n                    w.configure(bg=self.c_card)\n\n    # ---- Actions panel ----\n    def build_actions_panel(self):\n        frm = tk.Frame(self, bg=self.c_bg)\n        frm.pack(fill=\'x\', padx=10, pady=8)\n\n        btn_auto = tk.Button(frm, text=" 🚀  AUTO-DETECT & APPLY ",\n                             bg=self.c_green, fg=\'#fff\', font=(\'DejaVu Sans\', 11, \'bold\'),\n                             relief=\'flat\', padx=16, pady=8, cursor=\'hand2\',\n                             activebackground=\'#27ae60\', command=self.auto_detect_and_apply)\n        btn_auto.pack(side=\'left\', padx=(0, 10))\n\n        btn_apply = tk.Button(frm, text=" ✅  Apply Selected Preset ",\n                              bg=self.c_accent, fg=\'#fff\', font=(\'DejaVu Sans\', 10, \'bold\'),\n                              relief=\'flat\', padx=14, pady=8, cursor=\'hand2\',\n                              activebackground=\'#2980b9\', command=self.apply_preset)\n        btn_apply.pack(side=\'left\', padx=5)\n\n        btn_esc = tk.Button(frm, text=" 🔒  Fix Esc Key ",\n                            bg=self.c_card, fg=self.c_fg, font=(\'DejaVu Sans\', 10),\n                            relief=\'flat\', padx=12, pady=8, cursor=\'hand2\',\n                            command=self.fix_esc_key)\n        btn_esc.pack(side=\'left\', padx=5)\n\n        self.lbl_action_status = tk.Label(frm, text="", bg=self.c_bg, fg=self.c_yellow,\n                                          font=(\'DejaVu Sans\', 9))\n        self.lbl_action_status.pack(side=\'right\', padx=10)\n\n    # ---- Footer ----\n    def build_footer(self):\n        bar = tk.Frame(self, bg=self.c_panel, height=28)\n        bar.pack(fill=\'x\', side=\'bottom\', padx=10, pady=(0, 6))\n        tk.Label(bar, text="💡 Tip: Press AUTO-DETECT for the best settings, or pick a preset and click Apply.",\n                 bg=self.c_panel, fg=self.c_muted, font=(\'DejaVu Sans\', 9)).pack(padx=10, pady=5)\n\n    # ---- Logic ----\n\n    def refresh_status(self):\n        cur_res = get_current_resolution()\n        self.lbl_resolution.config(text=cur_res)\n        self.lbl_preset.config(text=PRESETS.get(self.current_preset, {}).get(\'label\', self.current_preset))\n\n        # Wine renderer info\n        env_csmt = \'Enabled\' if PRESETS.get(self.current_preset, {}).get(\'wine_csmt\') else \'Disabled\'\n        self.lbl_wine.config(text=f"WineD3D (OpenGL) · CSMT {env_csmt}")\n\n        # Keyboard lock status\n        kl_status = "Not checked"\n        try:\n            for root in [Path(\'/usr/local/lib\'), Path(\'/usr/lib\')]:\n                for html in root.rglob(\'index.html\'):\n                    if \'selkies\' in str(html).lower():\n                        if \'7BZ Dev Cloud Game: Keyboard Lock\' in html.read_text(encoding=\'utf-8\'):\n                            kl_status = "✅ Active (Esc captured)"\n                        else:\n                            kl_status = "❌ Not injected"\n                        break\n        except Exception:\n            kl_status = "Unknown"\n        self.lbl_keyboard.config(text=kl_status)\n\n        self.select_preset(self.current_preset)\n\n    def apply_preset(self):\n        self.lbl_action_status.config(text="Applying...", fg=self.c_yellow)\n        self.update()\n        threading.Thread(target=self._apply_worker, daemon=True).start()\n\n    def _apply_worker(self):\n        key = self.current_preset\n        p = PRESETS[key]\n        msgs = []\n\n        # 1. Resolution\n        if set_resolution(p[\'resolution\']):\n            msgs.append(f"Resolution → {p[\'resolution\']}")\n        else:\n            msgs.append(f"Resolution change to {p[\'resolution\']} failed (may already be set)")\n\n        # 2. Wine env\n        write_wine_env(key)\n        msgs.append(f"Wine config → csmt={\'on\' if p[\'wine_csmt\'] else \'off\'}")\n\n        # 3. Save\n        save_settings(key)\n        msgs.append(f"Preset saved: {key}")\n\n        summary = " · ".join(msgs)\n        self.after(0, lambda: self.lbl_action_status.config(text=f"✅ {summary}", fg=self.c_green))\n        self.after(0, self.refresh_status)\n\n    def auto_detect_and_apply(self):\n        self.lbl_action_status.config(text="🔍 Testing bandwidth...", fg=self.c_yellow)\n        self.update()\n        threading.Thread(target=self._auto_worker, daemon=True).start()\n\n    def _auto_worker(self):\n        speed = test_bandwidth()\n        recommended = recommend_preset(speed)\n\n        self.current_preset = recommended\n        self.after(0, lambda: self.select_preset(recommended))\n        self.after(0, lambda: self.lbl_action_status.config(\n            text=f"📶 Speed: {speed:.1f} Mbps → Recommended: {PRESETS[recommended][\'label\']}",\n            fg=self.c_accent))\n        self.after(100, lambda: None)  # small delay\n\n        # Apply\n        p = PRESETS[recommended]\n        set_resolution(p[\'resolution\'])\n        write_wine_env(recommended)\n        save_settings(recommended)\n\n        # Also inject keyboard lock\n        patched = inject_keyboard_lock()\n\n        self.after(200, lambda: self.lbl_action_status.config(\n            text=f"✅ Applied: {PRESETS[recommended][\'label\']} ({p[\'resolution\']}) · {speed:.1f} Mbps" +\n                 (" · Esc Lock ✅" if patched else ""),\n            fg=self.c_green))\n        self.after(200, self.refresh_status)\n\n    def fix_esc_key(self):\n        self.lbl_action_status.config(text="🔒 Injecting keyboard lock...", fg=self.c_yellow)\n        self.update()\n\n        def _worker():\n            patched = inject_keyboard_lock()\n            if patched:\n                msg = f"✅ Keyboard lock injected into {len(patched)} file(s). Reload your browser tab (F5) to activate."\n            else:\n                msg = "⚠️ Could not find Selkies HTML files. Use the browser bookmarklet instead."\n            self.after(0, lambda: self.lbl_action_status.config(text=msg, fg=self.c_green if patched else self.c_yellow))\n            self.after(0, self.refresh_status)\n\n            if not patched:\n                self.after(500, self.show_bookmarklet_fallback)\n\n        threading.Thread(target=_worker, daemon=True).start()\n\n    def show_bookmarklet_fallback(self):\n        dlg = tk.Toplevel(self)\n        dlg.title("🔒 Esc Key Fix – Browser Bookmarklet")\n        dlg.geometry("540x260")\n        dlg.configure(bg=self.c_bg)\n        dlg.transient(self)\n\n        tk.Label(dlg, text="Alternative: Browser Bookmarklet", bg=self.c_bg, fg=\'#fff\',\n                 font=(\'DejaVu Sans\', 12, \'bold\')).pack(pady=(12, 8))\n\n        tk.Label(dlg, text=(\n            "If the automatic injection didn\'t work, you can fix the Esc key\\n"\n            "by creating a browser bookmark with this code.\\n"\n            "Click the bookmark after entering fullscreen (F11)."\n        ), bg=self.c_bg, fg=self.c_muted, font=(\'DejaVu Sans\', 9), justify=\'left\').pack(padx=14)\n\n        code = "javascript:void(navigator.keyboard&&navigator.keyboard.lock([\'Escape\']).then(()=>document.title+=\' 🔒\').catch(e=>alert(\'Keyboard lock: \'+e)))"\n        ent = tk.Entry(dlg, bg=\'#1a202c\', fg=\'#0f0\', insertbackground=\'#fff\',\n                       font=(\'DejaVu Sans Mono\', 8), relief=\'flat\')\n        ent.pack(fill=\'x\', padx=14, pady=8, ipady=4)\n        ent.insert(0, code)\n        ent.select_range(0, tk.END)\n\n        def _copy():\n            self.clipboard_clear()\n            self.clipboard_append(code)\n            btn_copy.config(text="✓ Copied!")\n\n        btn_copy = tk.Button(dlg, text=" 📋 Copy to Clipboard ", bg=self.c_card, fg=self.c_fg,\n                             relief=\'flat\', cursor=\'hand2\', command=_copy)\n        btn_copy.pack(pady=6)\n\n\nif __name__ == "__main__":\n    app = AutoOptimizer()\n    app.mainloop()\n', encoding='utf-8')

optimizer_sh = """#!/bin/bash
export DISPLAY="${DISPLAY:-:0}"
export PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"
export HOME="/home/gamer"
mkdir -p /home/gamer/logs
LOG="/home/gamer/logs/auto_optimize.log"

if ! python3 -c "import tkinter" >/dev/null 2>&1; then
    apt-get update -qq && apt-get install -y -qq python3-tk >> "$LOG" 2>&1 || true
fi

exec python3 /tmp/kaggle-game-dev-bench/auto_optimize.py >> "$LOG" 2>&1
"""
(BENCH / 'launch_optimizer.sh').write_text(optimizer_sh, encoding='utf-8')

# Make all shell scripts executable
for p in BENCH.glob('*.sh'):
    p.chmod(0o755)

print('Launcher scripts, storefront wrappers, and desktop configuration saved successfully.', flush=True)


In [ ]:
from pathlib import Path
import json, os, shutil, signal, subprocess, sys, time, urllib.request

BENCH = Path('/tmp/kaggle-game-dev-bench')
BUILDS = Path('/tmp/game-builds')
BUILDS.mkdir(parents=True, exist_ok=True)
SCREENSHOTS = Path('/kaggle/working/screenshots')
SCREENSHOTS.mkdir(parents=True, exist_ok=True)

# Master desktop session wrapper
wrapper = BENCH / 'start-desktop.sh'
wrapper.write_text('#!/bin/bash\n# Asymmetric Dual-GPU Allocation:\n# Force all desktop session applications, game runners, and Wine to use GPU 0.\nexport CUDA_VISIBLE_DEVICES=0\nexport DRI_PRIME=0\nexport VGL_DISPLAY=egl0\nexport MESA_GL_VERSION_OVERRIDE=4.5COMPAT\nexport MESA_GLSL_VERSION_OVERRIDE=450\n\nexport LD_PRELOAD=\'/tmp/kaggle-game-dev-bench/$LIB/selkies_input_interposer.so:/tmp/kaggle-game-dev-bench/$LIB/selkies_fake_udev.so\'\nexport SDL_AUDIODRIVER=pulse\nexport SELKIES_JS_SOCKET_PATH="/tmp"\nexport SDL_JOYSTICK_DISABLE_UDEV=1\nexport SDL_JOYSTICK_DEVICE=/dev/input/event1000:/dev/input/event1001:/dev/input/event1002:/dev/input/event1003\nexport DISPLAY="${DISPLAY:-:0}"\nexport PATH="/usr/games:/usr/local/bin:/usr/bin:$PATH"\n\n# Ensure /dev/input nodes and permissions exist\nmkdir -p /dev/input 2>/dev/null || true\nfor i in 0 1 2 3; do touch /dev/input/js$i 2>/dev/null || true; done\nfor i in 1000 1001 1002 1003; do touch /dev/input/event$i 2>/dev/null || true; done\nchmod -R 666 /dev/input 2>/dev/null || true\n\n# Continuous socket permission and mirroring monitor so gamer user can connect to gamepad sockets\n(while true; do\n    chmod 755 /run/user /run/user/0 2>/dev/null || true\n    chmod 666 /tmp/selkies_*.sock /run/user/*/selkies_*.sock 2>/dev/null || true\n    ln -sf /run/user/0/selkies_*.sock /tmp/ 2>/dev/null || true\n    ln -sf /tmp/selkies_*.sock /run/user/1000/ 2>/dev/null || true\n    sleep 2\ndone) &\n\n# Load NVIDIA userspace driver environment if available\nprobe_env=/tmp/kaggle-game-dev-bench/nvidia-userspace-probe/env.sh\nif [ -f "$probe_env" ]; then\n    # shellcheck disable=SC1090\n    . "$probe_env"\nfi\nprintf \'%s\' "${XAUTHORITY:-}" > /tmp/kaggle-game-dev-bench/xauthority-path\nprintf \'%s\' "${PULSE_SERVER:-}" > /tmp/kaggle-game-dev-bench/pulse-server-path\n\nif [ -z "${PULSE_SERVER:-}" ]; then\n    for pulse_socket in /run/user/1000/pulse/native /tmp/pulse-*/native; do\n        if [ -S "$pulse_socket" ]; then\n            export PULSE_SERVER="unix:$pulse_socket"\n            break\n        fi\n    done\nfi\n\n# Xvfb remains the Selkies capture desktop. A separate NVIDIA Xorg display is\n# used only when the host mounted the display driver and a renderer test passes.\nif [ "$(id -u)" -eq 0 ]; then\n    /bin/bash /tmp/kaggle-game-dev-bench/gpu_xorg.sh || true\nfi\n\n# Grant unprivileged gamer user access to X11 display and root packages\nxhost + 2>/dev/null || true\nchmod 755 /root 2>/dev/null || true\nchmod -R 755 /root/.local 2>/dev/null || true\nchmod -R 777 /home/gamer 2>/dev/null || true\nxset m 0 0 2>/dev/null || true\n\n# Grant unprivileged gamer user access to PulseAudio socket and runtime directory\nchmod 777 "$XDG_RUNTIME_DIR" 2>/dev/null || true\nchmod -R 777 "$XDG_RUNTIME_DIR/pulse" 2>/dev/null || true\n\ninstall -d -o gamer -g gamer -m 700 /home/gamer/.config/pulse\nif [ -f "$HOME/.config/pulse/cookie" ]; then\n    install -o gamer -g gamer -m 600 "$HOME/.config/pulse/cookie" /home/gamer/.config/pulse/cookie\nfi\n\n# Ensure /usr/bin/wine exists by symlinking Wine-GE if system wine is absent\nfor w_candidate in /home/gamer/.config/heroic/tools/wine/*/bin/wine; do\n    if [ -f "$w_candidate" ]; then\n        if [ ! -f /usr/bin/wine ]; then\n            ln -sf "$w_candidate" /usr/bin/wine 2>/dev/null || true\n            ln -sf "$(dirname "$w_candidate")/wineserver" /usr/bin/wineserver 2>/dev/null || true\n            ln -sf "$(dirname "$w_candidate")/wineboot" /usr/bin/wineboot 2>/dev/null || true\n        fi\n        if [ ! -f /usr/local/bin/wine ]; then\n            ln -sf "$w_candidate" /usr/local/bin/wine 2>/dev/null || true\n        fi\n        break\n    fi\ndone\n\n# Start Openbox window manager in background for window frames, moving & resizing\nopenbox &\n\n# Start tint2 taskbar panel in background for window switching\nif which tint2 >/dev/null 2>&1; then\n    tint2 -c /tmp/kaggle-game-dev-bench/tint2rc &\nfi\n\n# Keep the emergency HUD control plane separate from Selkies and the desktop.\ncontrol_pidfile=/tmp/kaggle-game-dev-bench/session-control.pid\nif [ -f "$control_pidfile" ]; then\n    control_pid=$(cat "$control_pidfile" 2>/dev/null || true)\n    if [ -n "$control_pid" ] && [ -r "/proc/$control_pid/cmdline" ] && \\\n       tr \'\\0\' \' \' < "/proc/$control_pid/cmdline" | grep -q \'/tmp/kaggle-game-dev-bench/session_control.py\'; then\n        kill "$control_pid" 2>/dev/null || true\n    fi\nfi\nrunuser -u gamer -- env DISPLAY="$DISPLAY" HOME=/home/gamer PATH="$PATH" \\\n    PULSE_SERVER="${PULSE_SERVER:-}" SDL_AUDIODRIVER=pulse \\\n    python3 /tmp/kaggle-game-dev-bench/session_control.py \\\n    >> /home/gamer/logs/session-control.log 2>&1 &\necho $! > "$control_pidfile"\n\n# Run the controller-friendly launcher UI as the main session process\nif [ "$(id -u)" -eq 0 ]; then\n    exec runuser -u gamer -- env -u DBUS_SESSION_BUS_ADDRESS DISPLAY="$DISPLAY" HOME=/home/gamer PATH="$PATH" \\\n        XDG_RUNTIME_DIR="${XDG_RUNTIME_DIR:-}" SDL_AUDIODRIVER=pulse \\\n        PULSE_SERVER="${PULSE_SERVER:-}" SELKIES_JS_SOCKET_PATH="${SELKIES_JS_SOCKET_PATH:-}" \\\n        GAMELAB_ALLOW_SOFTWARE="${GAMELAB_ALLOW_SOFTWARE:-0}" \\\n        GAMELAB_SOFTWARE_RESOLUTION="${GAMELAB_SOFTWARE_RESOLUTION:-854x480}" \\\n        LP_NUM_THREADS="${LP_NUM_THREADS:-}" \\\n        LD_LIBRARY_PATH="${LD_LIBRARY_PATH:-}" \\\n        __EGL_VENDOR_LIBRARY_FILENAMES="${__EGL_VENDOR_LIBRARY_FILENAMES:-}" \\\n        VK_DRIVER_FILES="${VK_DRIVER_FILES:-}" VK_ICD_FILENAMES="${VK_ICD_FILENAMES:-}" \\\n        GAMELAB_NVIDIA_LIB32="${GAMELAB_NVIDIA_LIB32:-}" \\\n        PYTHONPATH="/root/.local/lib/python3.13/site-packages:/usr/local/lib/python3.13/dist-packages:${PYTHONPATH:-}" \\\n        LD_PRELOAD="$LD_PRELOAD" SDL_JOYSTICK_DISABLE_UDEV=1 \\\n        SDL_JOYSTICK_DEVICE="$SDL_JOYSTICK_DEVICE" \\\n        bash -c "while true; do python3 /tmp/kaggle-game-dev-bench/dev_launcher.py >> /home/gamer/logs/launcher.log 2>&1 || python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> /home/gamer/logs/gui.log 2>&1 || sleep 2; done"\nfi\nexec bash -c "while true; do python3 /tmp/kaggle-game-dev-bench/dev_launcher.py >> /home/gamer/logs/launcher.log 2>&1 || python3 /tmp/kaggle-game-dev-bench/legendary_gui.py >> /home/gamer/logs/gui.log 2>&1 || sleep 2; done"\n', encoding='utf-8')
wrapper.chmod(0o755)

# Build a custom Selkies web root without changing the installed streaming core.
subprocess.run([sys.executable, str(BENCH / 'install_web_client.py')], check=True)

# Stop any previous session
pidfile = BENCH / 'selkies.pid'
if pidfile.exists():
    try:
        previous = int(pidfile.read_text())
        command = Path(f'/proc/{previous}/cmdline').read_bytes().replace(b'\0', b' ')
        if b'selkies-session' in command and b'kaggle-game-dev-bench' in command:
            os.killpg(previous, signal.SIGTERM)
            time.sleep(2)
    except (ValueError, OSError, ProcessLookupError):
        pass

env = os.environ.copy()
env['PATH'] = f"/usr/local/bin:/usr/games:/usr/bin:{env.get('PATH', '')}"

# Asymmetric Dual-GPU Pipeline:
# If two GPUs are detected (/dev/nvidia0 and /dev/nvidia1):
# GPU 1 is dedicated exclusively to Selkies desktop capture & NVENC H.264 hardware encoding.
# GPU 0 is dedicated exclusively to 3D game rendering (Vulkan/DXVK/VKD3D/VirtualGL).
if Path('/dev/nvidia1').exists() and Path('/dev/nvidia0').exists():
    print('▶ Dual-GPU environment detected: Dedicating GPU 1 to NVENC encoding and GPU 0 to 3D Game Rendering.', flush=True)
    env['CUDA_VISIBLE_DEVICES'] = '1'
    env['SELKIES_ENCODER_DEVICE'] = '1'
else:
    print('▶ Single-GPU environment detected: GPU 0 active for rendering and encoding.', flush=True)
    env['CUDA_VISIBLE_DEVICES'] = '0'
    env['SELKIES_ENCODER_DEVICE'] = '0'

queue_size = os.environ.get('GAMELAB_WS_QUEUE', '4')
if not queue_size.isdigit() or not 1 <= int(queue_size) <= 120:
    raise ValueError('GAMELAB_WS_QUEUE must be an integer from 1 to 120')
env.update(SELKIES_ENABLE_HTTPS='false', SELKIES_MODE='websockets',
           SELKIES_ALLOWED_ORIGINS='https://kkb-production.jupyter-proxy.kaggle.net',
           SELKIES_ENCODER='h264enc', SELKIES_FRAMERATE='60-60',
           SELKIES_VIDEO_FULLCOLOR='false|locked',
           SELKIES_VIDEO_10BIT='false|locked',
           SELKIES_BACKPRESSURE_QUEUE_SIZE=queue_size,
           SELKIES_VIDEO_BITRATE='8000,1000-12000',
           SELKIES_RATE_CONTROL_MODE='cbr', SELKIES_CONGESTION_CONTROL='true',
           SELKIES_ENABLE_RESIZE='true',
           SELKIES_AUDIO_ENABLED='true', SELKIES_AUDIO_CHANNELS='2',
           SELKIES_AUDIO_BITRATE='128000', SELKIES_GAMEPAD_ENABLED='true',
           SELKIES_UINPUT_GAMEPAD='false',
           SELKIES_JS_SOCKET_PATH='/tmp',
           SELKIES_UI_TITLE='7BZ Dev Cloud Game',
           SELKIES_UI_SHOW_SIDEBAR='true', SELKIES_UI_SHOW_LOGO='false',
           SELKIES_UI_SIDEBAR_SHOW_VIDEO_SETTINGS='false',
           SELKIES_UI_SIDEBAR_SHOW_SCREEN_SETTINGS='false',
           SELKIES_MANUAL_WIDTH='1280', SELKIES_MANUAL_HEIGHT='720',
           SELKIES_WEB_ROOT=str(BENCH / 'web-root'),
           FILE_MANAGER_PATH=str(BUILDS),
           SDL_AUDIODRIVER='pulse')

# Enforce GLX extension on Xvfb in selkies package with Mesa isolation to prevent NVIDIA EGL-GBM crash
try:
    import selkies
    s_session = Path(selkies.__file__).parent / 'session.py'
    if s_session.is_file():
        s_code = s_session.read_text(encoding='utf-8')
        s_target = 'for extra in gpu + [glx, ["-extension", "GLX"]]:\n            read, write = os.pipe()\n            proc = self.spawn(["Xvfb", "-displayfd", str(write), "-auth", auth, *XVFB_ARGS, *extra],\n                              pass_fds=(write,))'
        s_target_alt = 'for extra in [glx]:\n            read, write = os.pipe()\n            proc = self.spawn(["Xvfb", "-displayfd", str(write), "-auth", auth, *XVFB_ARGS, *extra],\n                              pass_fds=(write,))'
        s_repl = ('xvfb_env = os.environ.copy()\n'
                  '        xvfb_env.update(LIBGL_ALWAYS_SOFTWARE="1", __GLX_VENDOR_LIBRARY_NAME="mesa", __EGL_VENDOR_LIBRARY_FILENAMES="", GBM_BACKENDS_PATH="/dev/null")\n'
                  '        for extra in [glx]:\n'
                  '            read, write = os.pipe()\n'
                  '            proc = self.spawn(["Xvfb", "-displayfd", str(write), "-auth", auth, *XVFB_ARGS, *extra],\n'
                  '                              pass_fds=(write,), env=xvfb_env)')
        if s_target in s_code:
            s_code = s_code.replace(s_target, s_repl)
        elif s_target_alt in s_code:
            s_code = s_code.replace(s_target_alt, s_repl)
        s_session.write_text(s_code, encoding='utf-8')
except Exception:
    pass

# Launch streaming gaming desktop on port 8080
log = (BENCH / 'selkies.log').open('a', buffering=1)
direct_script = BENCH / 'start_direct_stream.sh'
started = False

if direct_script.is_file():
    try:
        print('▶ Starting unified direct NVIDIA Xorg stream on port 8080...', flush=True)
        direct_res = subprocess.run(['bash', str(direct_script)],
                                    env=dict(os.environ, DIRECT_STREAM_PORT='8080'),
                                    capture_output=True, text=True, timeout=35)
        if direct_res.returncode == 0:
            started = True
            server_pid = (BENCH / 'direct-stream.pid').read_text().strip() if (BENCH / 'direct-stream.pid').exists() else None
            if server_pid:
                pidfile.write_text(server_pid)
        else:
            print('Direct stream notice:', (direct_res.stdout + direct_res.stderr)[-500:], flush=True)
    except Exception as e:
        print('Direct stream attempt skipped:', e, flush=True)

if not started:
    print('▶ Starting verified Selkies fallback desktop on port 8080...', flush=True)
    server = subprocess.Popen(['selkies-session', '--session=' + str(wrapper),
                               '--port=8080', '--enable-basic-auth=false'],
                              stdout=log, stderr=subprocess.STDOUT,
                              stdin=subprocess.DEVNULL, env=env,
                              start_new_session=True)
    pidfile.write_text(str(server.pid))

health = None
for _ in range(75):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/api/health', timeout=2) as response:
            if response.status == 200:
                health = response.read().decode(errors='replace')
                break
    except Exception:
        time.sleep(1)

if health != 'OK':
    selkies_tail = (BENCH / 'selkies.log').read_text(errors='replace')[-3000:] if (BENCH / 'selkies.log').exists() else ''
    stream_tail = (BENCH / 'logs' / 'direct-stream.log').read_text(errors='replace')[-3000:] if (BENCH / 'logs' / 'direct-stream.log').exists() else ''
    xorg_tail = (BENCH / 'logs' / 'gpu-xorg.log').read_text(errors='replace')[-2000:] if (BENCH / 'logs' / 'gpu-xorg.log').exists() else ''
    raise RuntimeError(f'Desktop did not start. Logs:\n{stream_tail}\n{selkies_tail}\n{xorg_tail}')

with urllib.request.urlopen('http://127.0.0.1:8080/api/status', timeout=5) as response:
    status = json.load(response)

time.sleep(2)
for sock in Path('/tmp').glob('selkies_*.sock'):
    try: sock.chmod(0o666)
    except Exception: pass
serverlog = (BENCH / 'selkies.log').read_text(errors='replace')

# =========================================================================
# EMPIRICAL STOREFRONT GUI LAUNCH & WINDOW VERIFICATION PROBE
# =========================================================================
print('▶ Starting empirical GUI launch tests for storefronts...', flush=True)
auth_path_file = BENCH / 'xauthority-path'
active_display = ':1' if (BENCH / 'gpu-xorg-ready').exists() or os.path.exists('/tmp/.X11-unix/X1') else ':0'
probe_auth = (BENCH / 'nvidia-Xauthority') if active_display == ':1' else auth_path_file
probe_env = dict(os.environ, DISPLAY=active_display)
if probe_auth.exists():
    probe_env['XAUTHORITY'] = str(probe_auth) if active_display == ':1' else probe_auth.read_text().strip()

def get_wm_windows():
    try:
        res = subprocess.run(['wmctrl', '-l'], capture_output=True, text=True, timeout=5,
                             env=probe_env)
        return res.stdout.splitlines() if res.returncode == 0 else []
    except Exception:
        return []

def capture_screenshot(out_path):
    try:
        result = subprocess.run(['ffmpeg', '-f', 'x11grab', '-video_size', '1280x720', '-i', active_display,
                        '-vframes', '1', '-y', str(out_path)],
                       capture_output=True, timeout=10, env=probe_env)
        if result.returncode != 0:
            print('Screenshot failed:', result.stderr.decode(errors='replace')[-350:], flush=True)
        return out_path.exists() and out_path.stat().st_size > 1000
    except Exception:
        return False

smoke_tests = [
    {
        'id': 'minigalaxy',
        'title': 'Minigalaxy GOG Client',
        'script': str(BENCH / 'launch_minigalaxy.sh'),
        'match': 'minigalaxy',
        'log': '/home/gamer/logs/minigalaxy.log'
    },
    {
        'id': 'heroic',
        'title': 'Heroic Games Launcher',
        'script': str(BENCH / 'launch_heroic.sh'),
        'match': 'heroic',
        'log': '/home/gamer/logs/heroic.log'
    },
    {
        'id': 'lutris',
        'title': 'Lutris Gaming Hub',
        'script': str(BENCH / 'launch_lutris.sh'),
        'match': 'lutris',
        'log': '/home/gamer/logs/lutris.log'
    },
    {
        'id': 'steam',
        'title': 'Steam Native Client',
        'script': str(BENCH / 'launch_steam.sh'),
        'match': 'steam',
        'log': '/home/gamer/logs/steam.log'
    },
    {
        'id': 'ea_app',
        'title': 'EA App Wine Runner',
        'script': str(BENCH / 'launch_ea.sh'),
        'match': 'ea',
        'log': '/home/gamer/logs/ea.log'
    }
]

def get_active_pulse() -> str:
    path = BENCH / 'pulse-server-path'
    if path.is_file():
        try:
            val = path.read_text().strip()
            if val:
                return val
        except Exception:
            pass
    for s in [Path('/run/user/1000/pulse/native'), *Path('/tmp').glob('pulse-*/native'), Path('/run/user/0/pulse/native')]:
        if s.exists():
            val = f"unix:{s}"
            try:
                path.write_text(val, encoding='utf-8')
                os.chmod(path, 0o666)
            except Exception:
                pass
            return val
    fallback = "unix:/run/user/1000/pulse/native"
    try:
        path.write_text(fallback, encoding='utf-8')
        os.chmod(path, 0o666)
    except Exception:
        pass
    return fallback

active_pulse = get_active_pulse()
smoke_results = {}
for test in smoke_tests:
    tid = test['id']
    script = test['script']
    match_str = test['match'].lower()
    log_file = Path(test['log'])
    shot_file = SCREENSHOTS / f'{tid}_smoke.png'
    
    print(f'  Testing storefront GUI: {test["title"]}...', flush=True)
    if not os.path.exists(script):
        smoke_results[tid] = {'tested': False, 'error': 'Launch script missing'}
        continue
        
    proc = subprocess.Popen(['runuser', '-u', 'gamer', '--', 'env', '-u', 'DBUS_SESSION_BUS_ADDRESS',
                             'PULSE_SERVER=' + active_pulse,
                             'dbus-run-session', '--', '/bin/bash', script], start_new_session=True)
    window_found = False
    matching_titles = []
    
    # Steam cold setup often takes longer than storefront startup.
    max_polls = 60 if tid == 'steam' else 20 if tid == 'heroic' else 12
    for _ in range(max_polls):
        time.sleep(1.5)
        wins = get_wm_windows()
        for w in wins:
            if match_str in w.lower():
                window_found = True
                matching_titles.append(w.strip())
        if window_found:
            break
            
    # Capture screen regardless of window detection to document actual virtual display state
    all_windows = get_wm_windows()
    error_shot = None
    if tid == 'steam':
        error_windows = [line for line in all_windows if 'error' in line.lower()]
        if error_windows:
            subprocess.run(['wmctrl', '-ia', error_windows[-1].split()[0]],
                           capture_output=True, timeout=5, env=probe_env)
            time.sleep(1)
            error_path = SCREENSHOTS / 'steam_error.png'
            if capture_screenshot(error_path):
                error_shot = str(error_path)
    shot_captured = capture_screenshot(shot_file)
    
    # Read process log tail
    log_tail = []
    if log_file.exists():
        lines = [l.strip() for l in log_file.read_text(encoding='utf-8', errors='replace').splitlines() if l.strip()]
        log_tail = lines[-6:]
        
    smoke_results[tid] = {
        'tested': True,
        'pid': proc.pid,
        'window_mapped': window_found,
        'detected_windows': matching_titles,
        'all_windows': all_windows,
        'error_screenshot': error_shot,
        'screenshot_saved': shot_captured,
        'screenshot_path': str(shot_file) if shot_captured else None,
        'process_poll': proc.poll(),
        'log_tail': log_tail
    }
    
    # Terminate smoke test process so desktop stays clean
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        time.sleep(1)
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGKILL)
    except Exception:
        pass

subprocess.run(['wmctrl', '-a', '7BZ Dev Cloud Game'], capture_output=True, timeout=5,
               env=probe_env)
time.sleep(1)

# Compile comprehensive health and verification audit
def check_bin(name):
    path = shutil.which(name) or (f'/usr/games/{name}' if os.path.exists(f'/usr/games/{name}') else None)
    return {'installed': path is not None, 'path': path}

def epic_installed_apps():
    installed = Path('/home/gamer/.config/heroic/legendaryConfig/legendary/installed.json')
    try:
        data = json.loads(installed.read_text())
        return sorted(data) if isinstance(data, dict) else []
    except (OSError, ValueError):
        return []

try:
    gpu_facts = json.loads(subprocess.run(
        [sys.executable, str(BENCH / 'gpu_runtime.py'), 'diagnose'],
        capture_output=True, text=True, timeout=50, env=probe_env,
        check=True).stdout)
except (OSError, ValueError, subprocess.SubprocessError):
    gpu_facts = {}

server_pid_val = None
if 'server' in locals() and hasattr(server, 'pid'):
    server_pid_val = server.pid
elif (BENCH / 'direct-stream.pid').exists():
    try:
        server_pid_val = int((BENCH / 'direct-stream.pid').read_text().strip())
    except Exception:
        server_pid_val = None

audio_cookie = '/home/gamer/.config/pulse/cookie'
audio_env = dict(os.environ, PULSE_SERVER=active_pulse, PULSE_COOKIE=audio_cookie)
gamer_audio = subprocess.run(
    ['runuser', '-u', 'gamer', '--', 'env', 'HOME=/home/gamer',
     'XDG_RUNTIME_DIR=' + str(BENCH / 'pulse-runtime'),
     'PULSE_SERVER=' + active_pulse, 'PULSE_COOKIE=' + audio_cookie,
     'pactl', 'info'], capture_output=True, text=True, timeout=10)
streamer_audio = subprocess.run(['pactl', 'info'], capture_output=True,
                                text=True, timeout=10, env=audio_env)
audio_sources = subprocess.run(['pactl', 'list', 'short', 'sources'],
                               capture_output=True, text=True, timeout=10,
                               env=audio_env)

report = {
    'health': health,
    'status': status,
    'nvenc_detected': 'encodes H264, H265 on nvenc' in serverlog,
    'gamepad_sockets_ready': '4 virtual gamepads ready' in serverlog,
    'server_pid': server_pid_val or 8080,
    'launcher_processes': subprocess.run(['pgrep', '-u', 'gamer', '-f', '/tmp/kaggle-game-dev-bench/dev_launcher.py'],
                                         capture_output=True, text=True).stdout.splitlines(),
    'launcher_screenshot': str(SCREENSHOTS / 'game_lab_home.png') if capture_screenshot(SCREENSHOTS / 'game_lab_home.png') else None,
    'desktop_processes': [line for line in subprocess.run(['ps', '-eo', 'user,pid,ppid,args'],
                                      capture_output=True, text=True).stdout.splitlines()
                          if any(name in line for name in ('Xvfb', 'openbox', 'tint2', 'dev_launcher', 'selkies-session'))][:20],
    'x11_probe': subprocess.run(['xdpyinfo', '-display', ':0'], capture_output=True,
                                text=True, timeout=10, env=probe_env).stderr[-800:],
    'xauthority_path': probe_env.get('XAUTHORITY'),
    'gamer_audio_probe': gamer_audio.returncode,
    'gamer_audio_error': gamer_audio.stderr[-400:],
    'streamer_audio_probe': streamer_audio.returncode,
    'streamer_audio_error': streamer_audio.stderr[-400:],
    'audio_monitor_probe': 'output.monitor' in audio_sources.stdout,
    'selkies_log_tail': (BENCH / 'selkies.log').read_text(errors='replace')[-3000:],
    'desktop_components': {
        'openbox': check_bin('openbox'),
        'tint2': check_bin('tint2'),
        'xterm': check_bin('xterm'),
        'pulseaudio': check_bin('pulseaudio')
    },
    'installed_executables': {
        'steam': check_bin('steam'),
        'heroic': check_bin('heroic'),
        'legendary': check_bin('legendary'),
        'minigalaxy': check_bin('minigalaxy'),
        'lutris': check_bin('lutris'),
        'wine': check_bin('wine'),
        'wine_ge': {'installed': any(os.path.exists(p) for p in ['/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine', *Path('/home/gamer/.config/heroic/tools/wine').glob('*/bin/wine')]), 'path': next((str(p) for p in ['/home/gamer/.config/heroic/tools/wine/Wine-GE-Proton8-26/bin/wine', *Path('/home/gamer/.config/heroic/tools/wine').glob('*/bin/wine')] if os.path.exists(str(p))), None)}
    },
    'empirical_gui_smoke_tests': smoke_results,
    'epic_library_probe': {
        'heroic_legendary_config': '/home/gamer/.config/heroic/legendaryConfig/legendary',
        'heroic_installed_apps': epic_installed_apps(),
        'standalone_legendary_config': '/home/gamer/.config/legendary',
        'interactive_account_test_required': True
    },
    'compatibility_matrix': {
        'native_linux_2d_and_cpu_games': 'Fully supported (Mesa llvmpipe software rasterizer)',
        'native_linux_opengl_games': 'NVIDIA OpenGL through VirtualGL' if gpu_facts.get('virtualgl') else 'OpenGL renderer requires live verification',
        'native_vulkan_and_dxvk_games': 'Vulkan X11 presentation verified' if gpu_facts.get('vulkan_window') else 'Vulkan X11 presentation unavailable in this session',
        'windows_games_under_wine_proton': 'DXVK and WineD3D available' if gpu_facts.get('vulkan_window') else 'WineD3D on NVIDIA OpenGL' if gpu_facts.get('virtualgl') else 'Graphics backend requires live verification',
        'epic_games_support': 'Heroic and Legendary installed; individual game launch needs authenticated interactive testing',
        'controller_support': '4 Xbox 360 userspace virtual gamepads via interposer',
        'audio_support': 'PulseAudio stereo monitor -> Selkies Opus 128 kbps',
        'video_support': 'Tesla T4 NVENC hardware H.264 4:2:0 encode at 60 FPS'
    },
    'gpu_runtime': gpu_facts,
    'stream_config': {'codec': 'H.264 4:2:0 8-bit', 'websocket_queue': os.environ.get('GAMELAB_WS_QUEUE', '4')}
}

try:
    listing = subprocess.run([sys.executable, '-m', 'jupyter', 'server', 'list', '--json'],
                             capture_output=True, text=True, timeout=15).stdout
    for line in listing.splitlines():
        if not line.lstrip().startswith('{'):
            continue
        item = json.loads(line)
        base = item.get('url', '').rstrip('/') + '/'
        token = item.get('token', '')
        if not base.startswith(('http://127.0.0.1:', 'http://localhost:')):
            continue
        candidate = base + 'proxy/8080/api/health'
        if token:
            candidate += '?token=' + token
        with urllib.request.urlopen(candidate, timeout=5) as response:
            report['jupyter_proxy_local'] = response.status == 200 and response.read() == b'OK'
        break
except Exception as exc:
    report['jupyter_proxy_local'] = False
    report['jupyter_proxy_error'] = type(exc).__name__

# Persist reports into output directory
Path('/kaggle/working/game_dev_bench_health.json').write_text(json.dumps(report, indent=2))
print('================== HEALTH & DIAGNOSTIC REPORT ==================')
print(json.dumps(report, indent=2))
print('================================================================')
preferred_port = '8080'
(BENCH / 'preferred-stream-port').write_text(preferred_port, encoding='utf-8')
print('Unified direct NVIDIA desktop ready on port 8080.', flush=True)
print('7BZ Dev Cloud Game is running in the active session.')


In [ ]:
from IPython.display import display, HTML
from pathlib import Path
port_file = Path('/tmp/kaggle-game-dev-bench/preferred-stream-port')
stream_port = port_file.read_text().strip() if port_file.is_file() else '8080'
html = r"""
<div style="font:16px system-ui, -apple-system, sans-serif;padding:24px;border-radius:16px;background:#0d1729;color:#f4f8ff;max-width:760px;box-shadow:0 8px 24px rgba(0,0,0,0.5)">
  <h2 style="margin-top:0;color:#4dcdf2;display:flex;align-items:center;gap:10px">
    <span>🎮</span> 7BZ Dev Cloud Game
  </h2>
  <p style="color:#b0c4de;line-height:1.5">
    Controller-first cloud game library with <b>Steam</b>, <b>Heroic</b>, <b>Legendary</b>, <b>Minigalaxy</b>, and <b>Lutris</b>.
  </p>

  <div style="background:#132238;padding:16px;border-radius:12px;margin:16px 0;border-left:4px solid #4dcdf2">
    <b>Connection Steps:</b>
    <ol style="margin:8px 0 0 18px;padding:0;color:#c8daf0;line-height:1.6">
      <li>In Kaggle, open <b>Run → Kaggle Jupyter Server</b> and copy the <b>VSCode Compatible URL</b>.</li>
      <li>Paste it into the field below and click <b>Connect to Stream</b>.</li>
      <li>Open the 7BZ HUD and select <b>Audio Unmute</b> for Opus stereo audio.</li>
      <li>For FPS mouse look, use a normal Chrome or Edge tab and enable <b>Gaming Mode</b>; embedded viewers may block pointer lock.</li>
      <li>Set your local keyboard layout to English (EN), then use mouse, keyboard, or gamepad navigation.</li>
    </ol>
  </div>

  <div style="margin:16px 0">
    <input id="kaggle-base" type="text" style="width:96%;padding:12px;border-radius:8px;border:1px solid #233b5d;background:#080e1a;color:#fff;font-size:14px" placeholder="Paste VSCode Compatible URL (https://...)">
    <button id="kaggle-open" style="margin-top:12px;padding:12px 24px;border-radius:8px;border:none;background:#4dcdf2;color:#081325;font-weight:bold;font-size:15px;cursor:pointer">Connect to Stream</button>
  </div>
  <p id="kaggle-result" style="font-size:13px;color:#9bdcf5;margin-top:6px"></p>

  <div style="border-top:1px solid #1a2a44;padding-top:14px;margin-top:18px;font-size:13px;color:#8ba2c4">
    <b>Empirical Hardware & Security Reality:</b>
    <ul style="margin:6px 0 0 16px;padding:0;line-height:1.5">
      <li><b>User state:</b> Storefronts run as <code>gamer</code>; their temporary credentials and caches are under <code>/home/gamer</code>.</li>
      <li><b>Storefront EULA:</b> Valve's Subscriber Agreement is presented to the user directly in the Steam GUI on first launch.</li>
      <li><b>GPU path:</b> GPU 0 renders the direct NVIDIA Xorg desktop and games; GPU 1 encodes the unified 8080 stream with NVENC.</li>
      <li><b>Custom Builds:</b> Upload test Linux builds into <code style="color:#4dcdf2">/tmp/game-builds</code> via Selkies Files panel.</li>
    </ul>
  </div>
</div>
<script>
document.getElementById('kaggle-open').onclick = function () {
  try {
    const raw = document.getElementById('kaggle-base').value.trim();
    if (!raw) throw new Error('Please paste your URL.');
    const u = new URL(raw);
    let p = u.pathname.replace(/\/(tree|lab|notebooks)(\/.*)?$/, '/');
    if (!p.endsWith('/')) p += '/';
    u.pathname = p + 'proxy/__STREAM_PORT__/';
    const target = u.toString();
    document.getElementById('kaggle-result').textContent = 'Opening authenticated Jupyter proxy route...';
    window.open(target, '_blank', 'noopener,noreferrer');
  } catch (e) {
    document.getElementById('kaggle-result').textContent = 'Please enter a valid https:// Kaggle Jupyter Server URL.';
  }
};
</script>
"""
display(HTML(html.replace('__STREAM_PORT__', stream_port)))
